# Notebook 05 — Aggregation & Export Layer (Refactored)

## Objective
This notebook turns the cleaned analytical base into curated, business-ready output tables for:

- Excel reporting
- SQL loading
- Tableau dashboards
- Power BI dashboards

## What was improved in this refactored version
- sections are grouped by purpose instead of long linear sprawl
- repeated interpretation markdown blocks were removed to keep the notebook easier to scan
- map outputs are now placed before the final registry so they can be governed like the rest of the deliverables
- final manifests are saved in a cleaner way
- the closure now appears at the real end of the notebook

## Section map
1. Source loading and aggregation-readiness audit
2. Core collision-level aggregations
3. MO-based and dashboard-focused aggregations
4. Map-ready outputs
5. Output governance, manifests, and cleanup
6. Final closure

## Guiding principle
Python remains the analytical source of truth. All downstream tool files should inherit the same logic, definitions, and counting rules from this notebook.


## Section 1 — Source Loading and Aggregation Readiness

In [1]:
# ============================================================
# Notebook 05 — Aggregation & Export Layer
# Step 1: Environment setup and source loading
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# Display options
# -----------------------------
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# -----------------------------
# Project root discovery
# -----------------------------
def find_project_root(start_path: Path, project_name: str = "la_traffic_collision_project") -> Path:
    """
    Traverse upward from start_path until the project root folder is found.
    This keeps the notebook portable across machines and folder locations.
    """
    start_path = start_path.resolve()
    
    for path in [start_path] + list(start_path.parents):
        if path.name == project_name:
            return path
    
    raise FileNotFoundError(
        f"Could not locate project root folder named '{project_name}' from: {start_path}"
    )

# Use current working directory as notebook starting point
PROJECT_ROOT = find_project_root(Path.cwd())

# -----------------------------
# Define core paths
# -----------------------------
DATA_CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables" / "aggregations"
OUTPUT_REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports" / "methodology"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

FACT_FILE = DATA_CLEANED_DIR / "fact_collision_mo_enriched.csv"
COLLISIONS_CLEAN_FILE = DATA_CLEANED_DIR / "collisions_clean.csv"
DIM_MO_FILE = DATA_CLEANED_DIR / "dim_mo_codes.csv"

# -----------------------------
# File existence check
# -----------------------------
readiness_df = pd.DataFrame({
    "file_name": [
        "fact_collision_mo_enriched.csv",
        "collisions_clean.csv",
        "dim_mo_codes.csv"
    ],
    "path": [
        str(FACT_FILE),
        str(COLLISIONS_CLEAN_FILE),
        str(DIM_MO_FILE)
    ],
    "exists": [
        FACT_FILE.exists(),
        COLLISIONS_CLEAN_FILE.exists(),
        DIM_MO_FILE.exists()
    ]
})

print("=" * 70)
print("Notebook 05 | Source Readiness Check")
print("=" * 70)
display(readiness_df)

if not FACT_FILE.exists():
    raise FileNotFoundError(
        f"Required analytical base file is missing: {FACT_FILE}"
    )

# -----------------------------
# Load source data
# -----------------------------
fact_collision_mo = pd.read_csv(FACT_FILE)

print("\n" + "=" * 70)
print("Loaded fact_collision_mo_enriched.csv successfully")
print("=" * 70)
print(f"Shape: {fact_collision_mo.shape}")

print("\nColumns preview:")
display(pd.DataFrame({"column_name": fact_collision_mo.columns}))

print("\nSample rows:")
display(fact_collision_mo.head(5))

Notebook 05 | Source Readiness Check


,file_name,path,exists
0,fact_collision_mo_enriched.csv,C:\Users\MKamal\Desktop\Project\without output...,True
1,collisions_clean.csv,C:\Users\MKamal\Desktop\Project\without output...,True
2,dim_mo_codes.csv,C:\Users\MKamal\Desktop\Project\without output...,True



Loaded fact_collision_mo_enriched.csv successfully
Shape: (3521494, 14)

Columns preview:


,column_name
0,dr_number
1,mo_code
2,mo_description
3,analytical_category
4,analytical_subcategory
5,analytical_domain
6,code_family
7,primary_entity
8,dashboard_group
9,traffic_relevance_score



Sample rows:


,dr_number,mo_code,mo_description,analytical_category,analytical_subcategory,analytical_domain,code_family,primary_entity,dashboard_group,traffic_relevance_score,traffic_relevance_band,keep_in_main_collision_dashboard,modeling_role,recommended_use
0,212013850,3004,T/C – Vehicle vs vehicle,Traffic Collision,Collision counterpart / mode pairing,Traffic Collision Core,3000s,Collision,Core collision analytics,5.0000,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
1,212013850,3027,T/C – (K) Fatal injury,Traffic Collision,Injury severity,Traffic Collision Core,3000s,Collision,Collision severity,5.0000,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
2,212013850,3034,T/C – City property involved: Yes,Traffic Collision,Scene context / property / intersection,Traffic Collision Core,3000s,Collision,Scene context,5.0000,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
3,212013850,4027,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,Traffic Collision Jurisdiction,4000s,Jurisdiction,Geography / reporting area,5.0000,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."
4,212013850,3036,T/C – At intersection: Yes,Traffic Collision,Scene context / property / intersection,Traffic Collision Core,3000s,Collision,Scene context,5.0000,5 - Core,Yes,Core collision dimension,"Use directly in dashboards, drill-downs, and m..."


## Step 2 — Aggregation Readiness Audit

The initial source inspection showed that `fact_collision_mo_enriched.csv` is not a full collision master table.
It is an MO-enriched analytical structure at the collision-code relationship level.

Therefore, before building summary tables, we must confirm the analytical role of each source table.

### Source roles to validate
We will inspect:

1. `collisions_clean.csv`
   - expected to be the main collision-level table
   - expected to contain time, geography, victim, and structural fields

2. `fact_collision_mo_enriched.csv`
   - expected to support MO-specific aggregation
   - expected to contain one or more rows per collision depending on MO code count

### Why this step is methodologically necessary
A valid aggregation layer depends on grain awareness.

If we aggregate from the wrong grain:
- counts may be overstated
- collisions may be duplicated
- dashboard KPIs may become inconsistent
- exported files may contradict each other across tools

### Scientific / analytical principle
Before aggregation, we must identify:
- row grain
- grouping keys
- safe counting logic
- domain-specific source suitability

This is a data modeling and aggregation governance step.

In [2]:
# ============================================================
# Step 2: Load collision-level base and audit aggregation readiness
# ============================================================

if not COLLISIONS_CLEAN_FILE.exists():
    raise FileNotFoundError(f"Required collision-level file is missing: {COLLISIONS_CLEAN_FILE}")

collisions_clean = pd.read_csv(COLLISIONS_CLEAN_FILE)

print("=" * 70)
print("Loaded collisions_clean.csv successfully")
print("=" * 70)
print(f"Shape: {collisions_clean.shape}")

print("\nFirst 40 columns from collisions_clean:")
display(pd.DataFrame({"column_name": collisions_clean.columns[:40]}))

print("\nSample rows from collisions_clean:")
display(collisions_clean.head(5))

Loaded collisions_clean.csv successfully
Shape: (621677, 34)

First 40 columns from collisions_clean:


,column_name
0,dr_number
1,date_reported
2,date_occurred
3,time_occurred
4,time_occurred_str
5,occ_hour
6,occ_minute
7,occ_time_of_day
8,occ_year
9,occ_month



Sample rows from collisions_clean:


,dr_number,date_reported,date_occurred,time_occurred,time_occurred_str,occ_hour,occ_minute,occ_time_of_day,occ_year,occ_month,occ_month_name,occ_weekday,occ_weekday_name,area_id,area_name,reporting_district,crime_code,crime_code_description,mo_codes,victim_age,is_valid_victim_age,victim_sex,is_valid_victim_sex_analytical,victim_descent,is_valid_victim_descent_analytical,premise_code,premise_description,has_complete_premise,address,cross_street,location,latitude,longitude,has_valid_coordinates
0,212013850,2021-09-03,2021-09-02,2335,2335,23,35,Night,2021,9,September,3,Thursday,20,Olympic,2021,997,TRAFFIC COLLISION,3004 3027 3034 4027 3036 3101 3401 3701,25.0000,True,F,True,W,True,101.0000,STREET,True,WILTON PL,6TH ST,"(34.063, -118.3141)",34.0630,-118.3141,True
1,221417787,2022-10-17,2022-10-17,1620,1620,16,20,Afternoon,2022,10,October,0,Monday,14,Pacific,1406,997,TRAFFIC COLLISION,4027 3011 3028 3034 3037 3101 3401 3701,21.0000,True,NaN,False,NaN,False,101.0000,STREET,True,NATIONAL BL,MOTOR AV,"(34.029, -118.4113)",34.0290,-118.4113,True
2,221418141,2022-10-26,2022-10-26,1135,1135,11,35,Morning,2022,10,October,2,Wednesday,14,Pacific,1434,997,TRAFFIC COLLISION,4027 3011 3025 3034 3037 3101 3401 3701,21.0000,True,NaN,False,NaN,False,101.0000,STREET,True,PALMS BL,ROSEWOOD AV,"(34.0052, -118.4478)",34.0052,-118.4478,True
3,222017859,2022-12-01,2022-12-01,230,230,2,30,Late Night,2022,12,December,3,Thursday,20,Olympic,2044,997,TRAFFIC COLLISION,3003 0913 3026 3035 3037 3101 3401 3701 4020,33.0000,True,M,True,H,True,101.0000,STREET,True,IROLO ST,SAN MARINO ST,"(34.0545, -118.3009)",34.0545,-118.3009,True
4,190319651,2019-08-24,2019-08-24,450,450,4,50,Late Night,2019,8,August,5,Saturday,3,Southwest,356,997,TRAFFIC COLLISION,3036 3004 3026 3101 4003,22.0000,True,M,True,H,True,101.0000,STREET,True,JEFFERSON BL,NORMANDIE AV,"(34.0255, -118.3002)",34.0255,-118.3002,True


In [3]:
# ============================================================
# Step 2A: Structural comparison between source tables
# ============================================================

source_profile = pd.DataFrame({
    "dataset_name": ["collisions_clean", "fact_collision_mo_enriched"],
    "row_count": [len(collisions_clean), len(fact_collision_mo)],
    "column_count": [collisions_clean.shape[1], fact_collision_mo.shape[1]],
    "unique_dr_number": [
        collisions_clean["dr_number"].nunique() if "dr_number" in collisions_clean.columns else np.nan,
        fact_collision_mo["dr_number"].nunique() if "dr_number" in fact_collision_mo.columns else np.nan,
    ]
})

if "dr_number" in fact_collision_mo.columns:
    source_profile["rows_per_unique_collision"] = [
        round(len(collisions_clean) / collisions_clean["dr_number"].nunique(), 4) if "dr_number" in collisions_clean.columns else np.nan,
        round(len(fact_collision_mo) / fact_collision_mo["dr_number"].nunique(), 4)
    ]
else:
    source_profile["rows_per_unique_collision"] = np.nan

print("=" * 70)
print("Source Table Structural Comparison")
print("=" * 70)
display(source_profile)

Source Table Structural Comparison


,dataset_name,row_count,column_count,unique_dr_number,rows_per_unique_collision
0,collisions_clean,621677,34,621677,1.0000
1,fact_collision_mo_enriched,3521494,14,534352,6.5902


In [4]:
# ============================================================
# Step 2B: Build semantic column map for aggregation planning
# ============================================================

def classify_column_role(col_name: str) -> str:
    col = col_name.lower()
    
    if any(x in col for x in ["date", "year", "month", "weekday", "hour", "time"]):
        return "time_dimension"
    elif any(x in col for x in ["area", "district", "premise", "lat", "lon", "street", "location"]):
        return "geography_location"
    elif any(x in col for x in ["victim", "sex", "descent", "age"]):
        return "victim_dimension"
    elif any(x in col for x in ["mo_", "analytical_", "code_family", "dashboard_group", "traffic_relevance", "primary_entity"]):
        return "mo_dimension"
    elif "dr_number" in col:
        return "collision_key"
    else:
        return "other_or_supporting"

collision_column_map = pd.DataFrame({
    "column_name": collisions_clean.columns
})

collision_column_map["semantic_role"] = collision_column_map["column_name"].apply(classify_column_role)

print("=" * 70)
print("Collision-Level Semantic Column Map")
print("=" * 70)
display(collision_column_map.head(100))

semantic_role_summary = (
    collision_column_map["semantic_role"]
    .value_counts(dropna=False)
    .rename_axis("semantic_role")
    .reset_index(name="column_count")
)

print("\nSemantic role distribution:")
display(semantic_role_summary)

Collision-Level Semantic Column Map


,column_name,semantic_role
0,dr_number,collision_key
1,date_reported,time_dimension
2,date_occurred,time_dimension
3,time_occurred,time_dimension
4,time_occurred_str,time_dimension
5,occ_hour,time_dimension
6,occ_minute,other_or_supporting
7,occ_time_of_day,time_dimension
8,occ_year,time_dimension
9,occ_month,time_dimension



Semantic role distribution:


,semantic_role,column_count
0,time_dimension,11
1,geography_location,10
2,victim_dimension,6
3,other_or_supporting,5
4,collision_key,1
5,mo_dimension,1


## Section 2 — Core Collision-Level Aggregations

## Step 3 — Build Yearly Collision Summary

### Objective
The purpose of this step is to create the first business-ready aggregated table:
`yearly_summary`

This table summarizes total collisions per year using a validated collision-level dataset.

### Why this table is important
Yearly aggregation is the highest-level temporal view and serves as:

- a baseline validation for time data
- a top-level KPI for reporting
- a foundation for trend analysis
- a reference for all downstream aggregations (monthly, weekday, etc.)

### Data source
We will use:
`collisions_clean.csv`

Because:
- each row represents one collision
- it is already cleaned and standardized
- it contains derived time fields (year, month, weekday, etc.)

### Aggregation grain
Grouping will be performed on:

- `occ_year`

### Metric definition
Primary metric:
- `collision_count` = COUNT DISTINCT `dr_number`

This ensures correctness even if duplication appears in future transformations.

### Additional metrics
We will compute:

- percentage of total collisions
- year-over-year absolute change
- year-over-year percentage change

These metrics are essential for business dashboards.

### Export strategy (updated)
Outputs will be saved under:

`outputs/`

Separated by tool:

- outputs/excel_ready/
- outputs/sql_ready/
- outputs/tableau_ready/
- outputs/powerbi_ready/

Even if duplicated, each tool gets its own clean dataset.

### Scientific / methodological references
- Aggregation follows the split-apply-combine paradigm (pandas groupby)
- Distinct counting ensures correctness across different data grains
- Export layer design follows BI best practices for performance and consistency

In [5]:
# ============================================================
# Step 3: Prepare tool-specific export folders under outputs/
# ============================================================

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

EXCEL_READY_DIR = OUTPUTS_DIR / "excel_ready"
SQL_READY_DIR = OUTPUTS_DIR / "sql_ready"
TABLEAU_READY_DIR = OUTPUTS_DIR / "tableau_ready"
POWERBI_READY_DIR = OUTPUTS_DIR / "powerbi_ready"

for folder in [EXCEL_READY_DIR, SQL_READY_DIR, TABLEAU_READY_DIR, POWERBI_READY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

tool_export_dirs = {
    "excel": EXCEL_READY_DIR,
    "sql": SQL_READY_DIR,
    "tableau": TABLEAU_READY_DIR,
    "powerbi": POWERBI_READY_DIR
}

print("=" * 70)
print("Tool-specific export folders under outputs/ are ready")
print("=" * 70)

for tool_name, folder_path in tool_export_dirs.items():
    print(f"{tool_name:<10} -> {folder_path}")

Tool-specific export folders under outputs/ are ready
excel      -> C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready
sql        -> C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready
tableau    -> C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready
powerbi    -> C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready


In [6]:
# ============================================================
# Step 3A: Validate required columns
# ============================================================

required_columns = ["dr_number", "occ_year"]

missing_columns = [col for col in required_columns if col not in collisions_clean.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available ✔")

All required columns are available ✔


In [7]:
# ============================================================
# Step 3B: Build yearly summary
# ============================================================

yearly_summary = (
    collisions_clean
    .groupby("occ_year")
    .agg(
        collision_count=("dr_number", "nunique")
    )
    .reset_index()
    .sort_values("occ_year")
)

print("=" * 70)
print("Yearly Summary Created")
print("=" * 70)

display(yearly_summary.head())

Yearly Summary Created


,occ_year,collision_count
0,2010,45098
1,2011,45280
2,2012,45409
3,2013,45040
4,2014,46957


In [8]:
# ============================================================
# Step 3C: Add analytical metrics
# ============================================================

# Total collisions
total_collisions = yearly_summary["collision_count"].sum()

# Share of total
yearly_summary["collision_pct"] = yearly_summary["collision_count"] / total_collisions

# Year-over-year change
yearly_summary["yoy_change"] = yearly_summary["collision_count"].diff()

# Year-over-year %
yearly_summary["yoy_pct_change"] = yearly_summary["collision_count"].pct_change()

print("=" * 70)
print("Enhanced Yearly Summary")
print("=" * 70)

display(yearly_summary)

Enhanced Yearly Summary


,occ_year,collision_count,collision_pct,yoy_change,yoy_pct_change
0,2010,45098,0.0725,NaN,NaN
1,2011,45280,0.0728,182.0000,0.0040
2,2012,45409,0.0730,129.0000,0.0028
3,2013,45040,0.0724,-369.0000,-0.0081
4,2014,46957,0.0755,"1,917.0000",0.0426
5,2015,52488,0.0844,"5,531.0000",0.1178
6,2016,56532,0.0909,"4,044.0000",0.0770
7,2017,57727,0.0929,"1,195.0000",0.0211
8,2018,57159,0.0919,-568.0000,-0.0098
9,2019,56628,0.0911,-531.0000,-0.0093


In [9]:
# ============================================================
# Step 3D: Export to all tool folders
# ============================================================

file_name = "yearly_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    yearly_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\yearly_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\yearly_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\yearly_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\yearly_summary.csv


In [10]:
# ============================================================
# Step 3E: Year completeness audit + dashboard-safe adjustments
# ============================================================

# Measure month coverage per year from the collision-level source
year_month_coverage = (
    collisions_clean
    .groupby("occ_year")
    .agg(
        months_covered=("occ_month", "nunique"),
        min_occ_date=("date_occurred", "min"),
        max_occ_date=("date_occurred", "max")
    )
    .reset_index()
)

# Merge coverage metadata into yearly summary
yearly_summary = yearly_summary.merge(
    year_month_coverage,
    on="occ_year",
    how="left"
)

# Flag complete years
yearly_summary["is_complete_year"] = yearly_summary["months_covered"] == 12

# Keep dashboard / analysis-safe YoY fields
yearly_summary["yoy_change_analysis"] = yearly_summary["yoy_change"]
yearly_summary["yoy_pct_change_analysis"] = yearly_summary["yoy_pct_change"]

# Incomplete years should not be used for comparative YoY interpretation
yearly_summary.loc[
    ~yearly_summary["is_complete_year"],
    ["yoy_change_analysis", "yoy_pct_change_analysis"]
] = np.nan

# Round percentage fields for cleaner exports
yearly_summary["collision_pct"] = yearly_summary["collision_pct"].round(4)
yearly_summary["yoy_pct_change"] = yearly_summary["yoy_pct_change"].round(4)
yearly_summary["yoy_pct_change_analysis"] = yearly_summary["yoy_pct_change_analysis"].round(4)

print("=" * 70)
print("Year completeness audit and analysis-safe yearly summary")
print("=" * 70)

display(yearly_summary)

# Re-export updated version to all tool-specific folders
file_name = "yearly_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    yearly_summary.to_csv(export_path, index=False)
    print(f"Updated export saved to {tool_name}: {export_path}")

Year completeness audit and analysis-safe yearly summary


,occ_year,collision_count,collision_pct,yoy_change,yoy_pct_change,months_covered,min_occ_date,max_occ_date,is_complete_year,yoy_change_analysis,yoy_pct_change_analysis
0,2010,45098,0.0725,NaN,NaN,12,2010-01-01,2010-12-31,True,NaN,NaN
1,2011,45280,0.0728,182.0000,0.0040,12,2011-01-01,2011-12-31,True,182.0000,0.0040
2,2012,45409,0.0730,129.0000,0.0028,12,2012-01-01,2012-12-31,True,129.0000,0.0028
3,2013,45040,0.0724,-369.0000,-0.0081,12,2013-01-01,2013-12-31,True,-369.0000,-0.0081
4,2014,46957,0.0755,"1,917.0000",0.0426,12,2014-01-01,2014-12-31,True,"1,917.0000",0.0426
5,2015,52488,0.0844,"5,531.0000",0.1178,12,2015-01-01,2015-12-31,True,"5,531.0000",0.1178
6,2016,56532,0.0909,"4,044.0000",0.0770,12,2016-01-01,2016-12-31,True,"4,044.0000",0.0770
7,2017,57727,0.0929,"1,195.0000",0.0211,12,2017-01-01,2017-12-31,True,"1,195.0000",0.0211
8,2018,57159,0.0919,-568.0000,-0.0098,12,2018-01-01,2018-12-31,True,-568.0000,-0.0098
9,2019,56628,0.0911,-531.0000,-0.0093,12,2019-01-01,2019-12-31,True,-531.0000,-0.0093


Updated export saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\yearly_summary.csv
Updated export saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\yearly_summary.csv
Updated export saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\yearly_summary.csv
Updated export saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\yearly_summary.csv


## Step 4 — Monthly Collision Summary

### Objective
Create a monthly-level aggregation to analyze:

- seasonality patterns
- monthly trends within each year
- time-series behavior for dashboards

### Grouping logic
- occ_year
- occ_month
- occ_month_name

### Metric
- distinct collision count

### Analytical value
This table will support:
- line charts
- seasonality heatmaps
- monthly comparisons across years

In [11]:
# ============================================================
# Step 4A: Build monthly summary
# ============================================================

monthly_summary = (
    collisions_clean
    .groupby(["occ_year", "occ_month", "occ_month_name"])
    .agg(
        collision_count=("dr_number", "nunique")
    )
    .reset_index()
)

In [12]:
# ============================================================
# Step 4B: Sort properly (critical for time series)
# ============================================================

monthly_summary = monthly_summary.sort_values(
    ["occ_year", "occ_month"]
)

display(monthly_summary.head(15))

,occ_year,occ_month,occ_month_name,collision_count
0,2010,1,January,3723
1,2010,2,February,3492
2,2010,3,March,3900
3,2010,4,April,3670
4,2010,5,May,3809
5,2010,6,June,3634
6,2010,7,July,3758
7,2010,8,August,3680
8,2010,9,September,3583
9,2010,10,October,4050


In [13]:
# ============================================================
# Step 4C: Add percentage
# ============================================================

total_collisions = monthly_summary["collision_count"].sum()

monthly_summary["collision_pct"] = (
    monthly_summary["collision_count"] / total_collisions
)

monthly_summary["collision_pct"] = monthly_summary["collision_pct"].round(4)

In [14]:
# ============================================================
# Step 4D: Export
# ============================================================

file_name = "monthly_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    monthly_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\monthly_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\monthly_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\monthly_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\monthly_summary.csv


In [15]:
# ============================================================
# Step 4E: Monthly completeness and within-year analytical metrics
# ============================================================

# Year-level month coverage from the collision-level source
monthly_year_coverage = (
    collisions_clean
    .groupby("occ_year")
    .agg(
        months_covered=("occ_month", "nunique"),
        min_occ_date=("date_occurred", "min"),
        max_occ_date=("date_occurred", "max")
    )
    .reset_index()
)

monthly_summary = monthly_summary.merge(
    monthly_year_coverage,
    on="occ_year",
    how="left"
)

# Flag complete years at monthly table level
monthly_summary["is_complete_year"] = monthly_summary["months_covered"] == 12

# Share of each month within its own year
monthly_summary["year_total_collisions"] = (
    monthly_summary.groupby("occ_year")["collision_count"].transform("sum")
)

monthly_summary["monthly_share_within_year"] = (
    monthly_summary["collision_count"] / monthly_summary["year_total_collisions"]
)

# Safer formatting
monthly_summary["collision_pct"] = monthly_summary["collision_pct"].round(4)
monthly_summary["monthly_share_within_year"] = monthly_summary["monthly_share_within_year"].round(4)

print("=" * 70)
print("Enhanced Monthly Summary")
print("=" * 70)

display(monthly_summary.head(24))

Enhanced Monthly Summary


,occ_year,occ_month,occ_month_name,collision_count,collision_pct,months_covered,min_occ_date,max_occ_date,is_complete_year,year_total_collisions,monthly_share_within_year
0,2010,1,January,3723,0.0060,12,2010-01-01,2010-12-31,True,45098,0.0826
1,2010,2,February,3492,0.0056,12,2010-01-01,2010-12-31,True,45098,0.0774
2,2010,3,March,3900,0.0063,12,2010-01-01,2010-12-31,True,45098,0.0865
3,2010,4,April,3670,0.0059,12,2010-01-01,2010-12-31,True,45098,0.0814
4,2010,5,May,3809,0.0061,12,2010-01-01,2010-12-31,True,45098,0.0845
5,2010,6,June,3634,0.0058,12,2010-01-01,2010-12-31,True,45098,0.0806
6,2010,7,July,3758,0.0060,12,2010-01-01,2010-12-31,True,45098,0.0833
7,2010,8,August,3680,0.0059,12,2010-01-01,2010-12-31,True,45098,0.0816
8,2010,9,September,3583,0.0058,12,2010-01-01,2010-12-31,True,45098,0.0794
9,2010,10,October,4050,0.0065,12,2010-01-01,2010-12-31,True,45098,0.0898


In [16]:
# ============================================================
# Step 4F: Re-export enhanced monthly summary
# ============================================================

file_name = "monthly_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    monthly_summary.to_csv(export_path, index=False)
    print(f"Updated export saved to {tool_name}: {export_path}")

Updated export saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\monthly_summary.csv
Updated export saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\monthly_summary.csv
Updated export saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\monthly_summary.csv
Updated export saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\monthly_summary.csv


## Step 5 — Build Weekday-Hour Collision Summary

### Objective
Create a weekday-hour aggregation table to identify when collisions are most concentrated across the week.

### Why this table matters
This table is analytically valuable because it reveals:

- peak risk hours
- weekday traffic concentration patterns
- operational timing hotspots
- high-density windows for dashboard heatmaps

### Data source
We will use:
`collisions_clean.csv`

because it contains one row per collision and includes derived time variables.

### Grouping logic
The aggregation will group by:

- `occ_weekday`
- `occ_weekday_name`
- `occ_hour`

### Primary metric
- distinct collision count using `dr_number`

### Expected use cases
This table will support:

- heatmaps
- weekday-hour matrices
- peak period ranking
- operational traffic risk storytelling

In [17]:
# ============================================================
# Step 5A: Validate required columns for weekday-hour summary
# ============================================================

required_weekday_hour_columns = ["dr_number", "occ_weekday", "occ_weekday_name", "occ_hour"]

missing_weekday_hour_columns = [
    col for col in required_weekday_hour_columns
    if col not in collisions_clean.columns
]

if missing_weekday_hour_columns:
    raise ValueError(f"Missing required columns: {missing_weekday_hour_columns}")

print("All required weekday-hour columns are available ✔")

All required weekday-hour columns are available ✔


In [18]:
# ============================================================
# Step 5B: Build weekday-hour summary
# ============================================================

weekday_hour_summary = (
    collisions_clean
    .groupby(["occ_weekday", "occ_weekday_name", "occ_hour"])
    .agg(
        collision_count=("dr_number", "nunique")
    )
    .reset_index()
    .sort_values(["occ_weekday", "occ_hour"])
)

print("=" * 70)
print("Weekday-Hour Summary Created")
print("=" * 70)

display(weekday_hour_summary.head(20))

Weekday-Hour Summary Created


,occ_weekday,occ_weekday_name,occ_hour,collision_count
0,0,Monday,0,2164
1,0,Monday,1,1806
2,0,Monday,2,1649
3,0,Monday,3,1120
4,0,Monday,4,905
5,0,Monday,5,1178
6,0,Monday,6,2068
7,0,Monday,7,4213
8,0,Monday,8,4707
9,0,Monday,9,3881


In [19]:
# ============================================================
# Step 5C: Add analytical proportions
# ============================================================

total_collisions_weekday_hour = weekday_hour_summary["collision_count"].sum()

weekday_hour_summary["collision_pct"] = (
    weekday_hour_summary["collision_count"] / total_collisions_weekday_hour
)

weekday_hour_summary["weekday_total_collisions"] = (
    weekday_hour_summary.groupby("occ_weekday")["collision_count"].transform("sum")
)

weekday_hour_summary["hour_share_within_weekday"] = (
    weekday_hour_summary["collision_count"] / weekday_hour_summary["weekday_total_collisions"]
)

weekday_hour_summary["collision_pct"] = weekday_hour_summary["collision_pct"].round(4)
weekday_hour_summary["hour_share_within_weekday"] = weekday_hour_summary["hour_share_within_weekday"].round(4)

display(weekday_hour_summary.head(20))

,occ_weekday,occ_weekday_name,occ_hour,collision_count,collision_pct,weekday_total_collisions,hour_share_within_weekday
0,0,Monday,0,2164,0.0035,85683,0.0253
1,0,Monday,1,1806,0.0029,85683,0.0211
2,0,Monday,2,1649,0.0027,85683,0.0192
3,0,Monday,3,1120,0.0018,85683,0.0131
4,0,Monday,4,905,0.0015,85683,0.0106
5,0,Monday,5,1178,0.0019,85683,0.0137
6,0,Monday,6,2068,0.0033,85683,0.0241
7,0,Monday,7,4213,0.0068,85683,0.0492
8,0,Monday,8,4707,0.0076,85683,0.0549
9,0,Monday,9,3881,0.0062,85683,0.0453


In [20]:
# ============================================================
# Step 5D: Export weekday-hour summary
# ============================================================

file_name = "weekday_hour_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    weekday_hour_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\weekday_hour_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\weekday_hour_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\weekday_hour_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\weekday_hour_summary.csv


## Step 6 — Build Area Summary

### Objective
Create an area-level collision summary to identify which LAPD areas have the highest collision concentration.

### Why this table matters
This table is important because it supports:

- geographic ranking
- area comparison dashboards
- hotspot concentration analysis
- operational prioritization by area

### Data source
We will use:
`collisions_clean.csv`

because it is the validated collision-level source table and contains:

- `area_id`
- `area_name`
- `dr_number`
- coordinate validity
- premise completeness

### Grouping logic
The aggregation will group by:

- `area_id`
- `area_name`

### Primary metric
- distinct collision count using `dr_number`

### Additional metrics
We will also compute:

- share of total collisions
- area rank by collision count
- valid-coordinate collision count
- valid-coordinate rate
- complete-premise collision count
- complete-premise rate

These additions improve the usefulness of the table for dashboarding and map readiness assessment.

### Export strategy
The final table will be exported to:

- `outputs/excel_ready/`
- `outputs/sql_ready/`
- `outputs/tableau_ready/`
- `outputs/powerbi_ready/`

### Scientific / methodological note
This aggregation must be built from the collision-level table rather than the MO-expanded table,
because geographic totals should reflect unique collisions, not collision-code rows.

In [21]:
# ============================================================
# Step 6A: Validate required columns for area summary
# ============================================================

required_area_columns = [
    "dr_number",
    "area_id",
    "area_name",
    "has_valid_coordinates",
    "has_complete_premise"
]

missing_area_columns = [
    col for col in required_area_columns
    if col not in collisions_clean.columns
]

if missing_area_columns:
    raise ValueError(f"Missing required columns: {missing_area_columns}")

print("All required area summary columns are available ✔")

All required area summary columns are available ✔


In [22]:
# ============================================================
# Step 6B: Prepare area-level helper fields
# ============================================================

area_base = collisions_clean.copy()

area_base["valid_coordinate_collision_key"] = np.where(
    area_base["has_valid_coordinates"] == True,
    area_base["dr_number"],
    pd.NA
)

area_base["complete_premise_collision_key"] = np.where(
    area_base["has_complete_premise"] == True,
    area_base["dr_number"],
    pd.NA
)

print("=" * 70)
print("Area helper fields prepared")
print("=" * 70)

display(
    area_base[
        [
            "dr_number",
            "area_id",
            "area_name",
            "has_valid_coordinates",
            "has_complete_premise",
            "valid_coordinate_collision_key",
            "complete_premise_collision_key"
        ]
    ].head(5)
)

Area helper fields prepared


,dr_number,area_id,area_name,has_valid_coordinates,has_complete_premise,valid_coordinate_collision_key,complete_premise_collision_key
0,212013850,20,Olympic,True,True,212013850,212013850
1,221417787,14,Pacific,True,True,221417787,221417787
2,221418141,14,Pacific,True,True,221418141,221418141
3,222017859,20,Olympic,True,True,222017859,222017859
4,190319651,3,Southwest,True,True,190319651,190319651


In [23]:
# ============================================================
# Step 6C: Build area summary
# ============================================================

area_summary = (
    area_base
    .groupby(["area_id", "area_name"])
    .agg(
        collision_count=("dr_number", "nunique"),
        valid_coordinate_collision_count=("valid_coordinate_collision_key", "nunique"),
        complete_premise_collision_count=("complete_premise_collision_key", "nunique")
    )
    .reset_index()
)

# Derived metrics
total_area_collisions = area_summary["collision_count"].sum()

area_summary["collision_pct"] = area_summary["collision_count"] / total_area_collisions
area_summary["valid_coordinate_rate"] = (
    area_summary["valid_coordinate_collision_count"] / area_summary["collision_count"]
)
area_summary["complete_premise_rate"] = (
    area_summary["complete_premise_collision_count"] / area_summary["collision_count"]
)

# Area rank
area_summary["area_rank_by_collision_count"] = (
    area_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Formatting
area_summary["collision_pct"] = area_summary["collision_pct"].round(4)
area_summary["valid_coordinate_rate"] = area_summary["valid_coordinate_rate"].round(4)
area_summary["complete_premise_rate"] = area_summary["complete_premise_rate"].round(4)

# Sort
area_summary = area_summary.sort_values(
    ["collision_count", "area_id"],
    ascending=[False, True]
).reset_index(drop=True)

print("=" * 70)
print("Area Summary Created")
print("=" * 70)

display(area_summary)

Area Summary Created


,area_id,area_name,collision_count,valid_coordinate_collision_count,complete_premise_collision_count,collision_pct,valid_coordinate_rate,complete_premise_rate,area_rank_by_collision_count
0,12,77th Street,41780,41749,41716,0.0672,0.9993,0.9985,1
1,3,Southwest,36394,36357,36344,0.0585,0.9990,0.9986,2
2,7,Wilshire,34640,34606,34486,0.0557,0.9990,0.9956,3
3,20,Olympic,32445,32410,32349,0.0522,0.9989,0.9970,4
4,13,Newton,32399,32335,32380,0.0521,0.9980,0.9994,5
5,15,N Hollywood,32348,32251,32310,0.0520,0.9970,0.9988,6
6,8,West LA,32208,32150,32085,0.0518,0.9982,0.9962,7
7,14,Pacific,31867,31812,31790,0.0513,0.9983,0.9976,8
8,9,Van Nuys,30621,30561,30600,0.0493,0.9980,0.9993,9
9,17,Devonshire,30289,30205,30218,0.0487,0.9972,0.9977,10


In [24]:
# ============================================================
# Step 6C: Build area summary
# ============================================================

area_summary = (
    area_base
    .groupby(["area_id", "area_name"])
    .agg(
        collision_count=("dr_number", "nunique"),
        valid_coordinate_collision_count=("valid_coordinate_collision_key", "nunique"),
        complete_premise_collision_count=("complete_premise_collision_key", "nunique")
    )
    .reset_index()
)

# Derived metrics
total_area_collisions = area_summary["collision_count"].sum()

area_summary["collision_pct"] = area_summary["collision_count"] / total_area_collisions
area_summary["valid_coordinate_rate"] = (
    area_summary["valid_coordinate_collision_count"] / area_summary["collision_count"]
)
area_summary["complete_premise_rate"] = (
    area_summary["complete_premise_collision_count"] / area_summary["collision_count"]
)

# Area rank
area_summary["area_rank_by_collision_count"] = (
    area_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Formatting
area_summary["collision_pct"] = area_summary["collision_pct"].round(4)
area_summary["valid_coordinate_rate"] = area_summary["valid_coordinate_rate"].round(4)
area_summary["complete_premise_rate"] = area_summary["complete_premise_rate"].round(4)

# Sort
area_summary = area_summary.sort_values(
    ["collision_count", "area_id"],
    ascending=[False, True]
).reset_index(drop=True)

print("=" * 70)
print("Area Summary Created")
print("=" * 70)

display(area_summary)

Area Summary Created


,area_id,area_name,collision_count,valid_coordinate_collision_count,complete_premise_collision_count,collision_pct,valid_coordinate_rate,complete_premise_rate,area_rank_by_collision_count
0,12,77th Street,41780,41749,41716,0.0672,0.9993,0.9985,1
1,3,Southwest,36394,36357,36344,0.0585,0.9990,0.9986,2
2,7,Wilshire,34640,34606,34486,0.0557,0.9990,0.9956,3
3,20,Olympic,32445,32410,32349,0.0522,0.9989,0.9970,4
4,13,Newton,32399,32335,32380,0.0521,0.9980,0.9994,5
5,15,N Hollywood,32348,32251,32310,0.0520,0.9970,0.9988,6
6,8,West LA,32208,32150,32085,0.0518,0.9982,0.9962,7
7,14,Pacific,31867,31812,31790,0.0513,0.9983,0.9976,8
8,9,Van Nuys,30621,30561,30600,0.0493,0.9980,0.9993,9
9,17,Devonshire,30289,30205,30218,0.0487,0.9972,0.9977,10


In [25]:
# ============================================================
# Step 6D: Export area summary
# ============================================================

file_name = "area_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    area_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\area_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\area_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\area_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\area_summary.csv


## Step 7 — Build Premise Summary

### Objective
Create a premise-level collision summary to identify where collisions are most frequently recorded by premise type.

### Why this table matters
This table supports:

- ranking of collision settings
- comparison between street and non-street environments
- dashboard filters by premise type
- context analysis for operational storytelling

### Data source
We will use:
`collisions_clean.csv`

because it is the validated collision-level source and contains:

- `premise_code`
- `premise_description`
- `dr_number`
- `has_valid_coordinates`
- `has_complete_premise`

### Grouping logic
The aggregation will group by:

- `premise_code`
- `premise_description`

### Primary metric
- distinct collision count using `dr_number`

### Additional metrics
We will also compute:

- share of total collisions
- premise rank by collision count
- valid-coordinate collision count
- valid-coordinate rate

### Analytical note
Premise summaries are especially useful because they describe the collision setting.
However, very broad premise categories such as `STREET` may dominate the ranking,
so this general summary should later be complemented by a non-street version if needed.

### Export strategy
The final table will be exported to:

- `outputs/excel_ready/`
- `outputs/sql_ready/`
- `outputs/tableau_ready/`
- `outputs/powerbi_ready/`

In [26]:
# ============================================================
# Step 7A: Validate required columns for premise summary
# ============================================================

required_premise_columns = [
    "dr_number",
    "premise_code",
    "premise_description",
    "has_valid_coordinates",
    "has_complete_premise"
]

missing_premise_columns = [
    col for col in required_premise_columns
    if col not in collisions_clean.columns
]

if missing_premise_columns:
    raise ValueError(f"Missing required columns: {missing_premise_columns}")

print("All required premise summary columns are available ✔")

All required premise summary columns are available ✔


In [27]:
# ============================================================
# Step 7B: Prepare premise-level helper fields
# ============================================================

premise_base = collisions_clean.copy()

premise_base["valid_coordinate_collision_key"] = np.where(
    premise_base["has_valid_coordinates"] == True,
    premise_base["dr_number"],
    pd.NA
)

premise_base["complete_premise_collision_key"] = np.where(
    premise_base["has_complete_premise"] == True,
    premise_base["dr_number"],
    pd.NA
)

print("=" * 70)
print("Premise helper fields prepared")
print("=" * 70)

display(
    premise_base[
        [
            "dr_number",
            "premise_code",
            "premise_description",
            "has_valid_coordinates",
            "has_complete_premise",
            "valid_coordinate_collision_key",
            "complete_premise_collision_key"
        ]
    ].head(5)
)

Premise helper fields prepared


,dr_number,premise_code,premise_description,has_valid_coordinates,has_complete_premise,valid_coordinate_collision_key,complete_premise_collision_key
0,212013850,101.0000,STREET,True,True,212013850,212013850
1,221417787,101.0000,STREET,True,True,221417787,221417787
2,221418141,101.0000,STREET,True,True,221418141,221418141
3,222017859,101.0000,STREET,True,True,222017859,222017859
4,190319651,101.0000,STREET,True,True,190319651,190319651


In [28]:
# ============================================================
# Step 7C: Build premise summary
# ============================================================

premise_summary = (
    premise_base
    .groupby(["premise_code", "premise_description"])
    .agg(
        collision_count=("dr_number", "nunique"),
        valid_coordinate_collision_count=("valid_coordinate_collision_key", "nunique"),
        complete_premise_collision_count=("complete_premise_collision_key", "nunique")
    )
    .reset_index()
)

total_premise_collisions = premise_summary["collision_count"].sum()

premise_summary["collision_pct"] = (
    premise_summary["collision_count"] / total_premise_collisions
)

premise_summary["valid_coordinate_rate"] = (
    premise_summary["valid_coordinate_collision_count"] / premise_summary["collision_count"]
)

premise_summary["complete_premise_rate"] = (
    premise_summary["complete_premise_collision_count"] / premise_summary["collision_count"]
)

premise_summary["premise_rank_by_collision_count"] = (
    premise_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

premise_summary["collision_pct"] = premise_summary["collision_pct"].round(4)
premise_summary["valid_coordinate_rate"] = premise_summary["valid_coordinate_rate"].round(4)
premise_summary["complete_premise_rate"] = premise_summary["complete_premise_rate"].round(4)

premise_summary = premise_summary.sort_values(
    ["collision_count", "premise_code"],
    ascending=[False, True]
).reset_index(drop=True)

print("=" * 70)
print("Premise Summary Created")
print("=" * 70)

display(premise_summary)

Premise Summary Created


,premise_code,premise_description,collision_count,valid_coordinate_collision_count,complete_premise_collision_count,collision_pct,valid_coordinate_rate,complete_premise_rate,premise_rank_by_collision_count
0,101.0000,STREET,591306,590385,591306,0.9526,0.9984,1.0000,1
1,108.0000,PARKING LOT,19487,19461,19487,0.0314,0.9987,1.0000,2
2,102.0000,SIDEWALK,4183,4168,4183,0.0067,0.9964,1.0000,3
3,103.0000,ALLEY,1208,1206,1208,0.0019,0.9983,1.0000,4
4,104.0000,DRIVEWAY,1127,1126,1127,0.0018,0.9991,1.0000,5
5,110.0000,FREEWAY,620,618,620,0.0010,0.9968,1.0000,6
6,501.0000,SINGLE FAMILY DWELLING,378,377,378,0.0006,0.9974,1.0000,7
7,301.0000,GAS STATION,367,366,367,0.0006,0.9973,1.0000,8
8,212.0000,TRANSPORTATION FACILITY (AIRPORT),221,221,221,0.0004,1.0000,1.0000,9
9,710.0000,OTHER PREMISE,206,206,206,0.0003,1.0000,1.0000,10


In [29]:
# ============================================================
# Step 7D: Export premise summary
# ============================================================

file_name = "premise_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    premise_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\premise_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\premise_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\premise_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\premise_summary.csv


## Step 8 — Build Premise Summary Excluding STREET

### Objective
Create a second premise-level summary that excludes the dominant `STREET` category.

### Why this step matters
The general premise summary is useful, but it is heavily dominated by `STREET`.
That dominance can hide meaningful variation across secondary collision settings.

By excluding `STREET`, this table helps reveal:

- non-street collision environments
- parking-lot concentration
- sidewalk and alley contexts
- lower-frequency but still meaningful premise categories

### Data source
We will use:
`collisions_clean.csv`

### Filtering rule
We will exclude records where:

- `premise_code == 101`
or
- `premise_description == "STREET"`

### Grouping logic
The aggregation will group by:

- `premise_code`
- `premise_description`

### Primary metric
- distinct collision count using `dr_number`

### Additional metrics
We will compute:

- share of non-street collisions
- premise rank within non-street environments
- valid-coordinate collision count
- valid-coordinate rate

### Export strategy
The final table will be exported to:

- `outputs/excel_ready/`
- `outputs/sql_ready/`
- `outputs/tableau_ready/`
- `outputs/powerbi_ready/`

### Methodological note
This table is not a replacement for the full premise summary.
It is a complementary analytical view designed to reduce dominance bias and improve interpretability.

In [30]:
# ============================================================
# Step 8A: Build non-street filtered base
# ============================================================

premise_non_street_base = premise_base[
    ~(
        (premise_base["premise_code"] == 101) |
        (premise_base["premise_description"].astype(str).str.upper() == "STREET")
    )
].copy()

print("=" * 70)
print("Non-street premise base created")
print("=" * 70)
print(f"Rows after excluding STREET: {len(premise_non_street_base):,}")

display(
    premise_non_street_base[
        [
            "dr_number",
            "premise_code",
            "premise_description",
            "has_valid_coordinates",
            "has_complete_premise"
        ]
    ].head(10)
)

Non-street premise base created
Rows after excluding STREET: 30,371


,dr_number,premise_code,premise_description,has_valid_coordinates,has_complete_premise
19,190514692,108.0000,PARKING LOT,True,True
32,190514774,103.0000,ALLEY,True,True
51,230118048,108.0000,PARKING LOT,True,True
176,230314969,102.0000,SIDEWALK,True,True
179,190814859,108.0000,PARKING LOT,True,True
186,190913252,108.0000,PARKING LOT,True,True
210,191013897,108.0000,PARKING LOT,True,True
264,230411598,406.0000,OTHER STORE,True,True
283,191314474,103.0000,ALLEY,True,True
285,230512486,102.0000,SIDEWALK,True,True


In [31]:
# ============================================================
# Step 8B: Build non-street premise summary
# ============================================================

premise_summary_non_street = (
    premise_non_street_base
    .groupby(["premise_code", "premise_description"])
    .agg(
        collision_count=("dr_number", "nunique"),
        valid_coordinate_collision_count=("valid_coordinate_collision_key", "nunique"),
        complete_premise_collision_count=("complete_premise_collision_key", "nunique")
    )
    .reset_index()
)

total_non_street_collisions = premise_summary_non_street["collision_count"].sum()

premise_summary_non_street["collision_pct_non_street"] = (
    premise_summary_non_street["collision_count"] / total_non_street_collisions
)

premise_summary_non_street["valid_coordinate_rate"] = (
    premise_summary_non_street["valid_coordinate_collision_count"] /
    premise_summary_non_street["collision_count"]
)

premise_summary_non_street["complete_premise_rate"] = (
    premise_summary_non_street["complete_premise_collision_count"] /
    premise_summary_non_street["collision_count"]
)

premise_summary_non_street["premise_rank_non_street"] = (
    premise_summary_non_street["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

premise_summary_non_street["collision_pct_non_street"] = (
    premise_summary_non_street["collision_pct_non_street"].round(4)
)
premise_summary_non_street["valid_coordinate_rate"] = (
    premise_summary_non_street["valid_coordinate_rate"].round(4)
)
premise_summary_non_street["complete_premise_rate"] = (
    premise_summary_non_street["complete_premise_rate"].round(4)
)

premise_summary_non_street = premise_summary_non_street.sort_values(
    ["collision_count", "premise_code"],
    ascending=[False, True]
).reset_index(drop=True)

print("=" * 70)
print("Premise Summary Excluding STREET Created")
print("=" * 70)

display(premise_summary_non_street)

Premise Summary Excluding STREET Created


,premise_code,premise_description,collision_count,valid_coordinate_collision_count,complete_premise_collision_count,collision_pct_non_street,valid_coordinate_rate,complete_premise_rate,premise_rank_non_street
0,108.0000,PARKING LOT,19487,19461,19487,0.6626,0.9987,1.0000,1
1,102.0000,SIDEWALK,4183,4168,4183,0.1422,0.9964,1.0000,2
2,103.0000,ALLEY,1208,1206,1208,0.0411,0.9983,1.0000,3
3,104.0000,DRIVEWAY,1127,1126,1127,0.0383,0.9991,1.0000,4
4,110.0000,FREEWAY,620,618,620,0.0211,0.9968,1.0000,5
5,501.0000,SINGLE FAMILY DWELLING,378,377,378,0.0129,0.9974,1.0000,6
6,301.0000,GAS STATION,367,366,367,0.0125,0.9973,1.0000,7
7,212.0000,TRANSPORTATION FACILITY (AIRPORT),221,221,221,0.0075,1.0000,1.0000,8
8,710.0000,OTHER PREMISE,206,206,206,0.0070,1.0000,1.0000,9
9,707.0000,GARAGE/CARPORT,173,173,173,0.0059,1.0000,1.0000,10


In [32]:
# ============================================================
# Step 8C: Export non-street premise summary
# ============================================================

file_name = "premise_summary_non_street.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    premise_summary_non_street.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\premise_summary_non_street.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\premise_summary_non_street.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\premise_summary_non_street.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\premise_summary_non_street.csv


## Step 9 — Build Victim Age Group Summary

### Objective
Create an age-group summary for collision records using victim age information.

### Why this table matters
This table supports:

- demographic profiling of collisions
- age-based dashboard filters
- identification of the most represented age groups
- comparison between valid age groups and unknown/invalid age records

### Data source
We will use:
`collisions_clean.csv`

because it contains:

- `dr_number`
- `victim_age`
- `is_valid_victim_age`

### Analytical design
Victim ages will be grouped into interpretable analytical bands:

- 0–12
- 13–17
- 18–24
- 25–34
- 35–44
- 45–54
- 55–64
- 65+
- Unknown / Invalid

### Why include Unknown / Invalid
Including an explicit Unknown / Invalid group is methodologically important because:

- it preserves full collision coverage
- it prevents hidden data loss
- it makes age-data quality visible in dashboards

### Primary metric
- distinct collision count using `dr_number`

### Additional metrics
We will compute:

- share of total collisions
- share within valid-age collisions only
- age-group rank among valid groups
- explicit age-data quality flag

### Export strategy
The final table will be exported to:

- `outputs/excel_ready/`
- `outputs/sql_ready/`
- `outputs/tableau_ready/`
- `outputs/powerbi_ready/`

### Methodological note
This summary is collision-based, not person-based.
It reflects the victim age field available in the collision record structure.

In [33]:
# ============================================================
# Step 9A: Validate required columns for victim age summary
# ============================================================

required_age_columns = [
    "dr_number",
    "victim_age",
    "is_valid_victim_age"
]

missing_age_columns = [
    col for col in required_age_columns
    if col not in collisions_clean.columns
]

if missing_age_columns:
    raise ValueError(f"Missing required columns: {missing_age_columns}")

print("All required victim age summary columns are available ✔")

All required victim age summary columns are available ✔


In [34]:
# ============================================================
# Step 9B: Build age-group helper fields
# ============================================================

victim_age_base = collisions_clean.copy()

def assign_age_group(age, is_valid):
    if (is_valid is not True) or pd.isna(age):
        return "Unknown / Invalid", 99, "Unknown / Invalid"

    age = float(age)

    if 0 <= age <= 12:
        return "0-12", 1, "Valid Age"
    elif 13 <= age <= 17:
        return "13-17", 2, "Valid Age"
    elif 18 <= age <= 24:
        return "18-24", 3, "Valid Age"
    elif 25 <= age <= 34:
        return "25-34", 4, "Valid Age"
    elif 35 <= age <= 44:
        return "35-44", 5, "Valid Age"
    elif 45 <= age <= 54:
        return "45-54", 6, "Valid Age"
    elif 55 <= age <= 64:
        return "55-64", 7, "Valid Age"
    elif age >= 65:
        return "65+", 8, "Valid Age"
    else:
        return "Unknown / Invalid", 99, "Unknown / Invalid"

age_assignments = victim_age_base.apply(
    lambda row: assign_age_group(row["victim_age"], row["is_valid_victim_age"]),
    axis=1,
    result_type="expand"
)

age_assignments.columns = [
    "victim_age_group",
    "victim_age_group_order",
    "age_data_quality_flag"
]

victim_age_base = pd.concat([victim_age_base, age_assignments], axis=1)

print("=" * 70)
print("Victim age helper fields prepared")
print("=" * 70)

display(
    victim_age_base[
        [
            "dr_number",
            "victim_age",
            "is_valid_victim_age",
            "victim_age_group",
            "victim_age_group_order",
            "age_data_quality_flag"
        ]
    ].head(10)
)

Victim age helper fields prepared


,dr_number,victim_age,is_valid_victim_age,victim_age_group,victim_age_group_order,age_data_quality_flag
0,212013850,25.0000,True,25-34,4,Valid Age
1,221417787,21.0000,True,18-24,3,Valid Age
2,221418141,21.0000,True,18-24,3,Valid Age
3,222017859,33.0000,True,25-34,4,Valid Age
4,190319651,22.0000,True,18-24,3,Valid Age
5,190319680,30.0000,True,25-34,4,Valid Age
6,190413769,NaN,False,Unknown / Invalid,99,Unknown / Invalid
7,190127578,21.0000,True,18-24,3,Valid Age
8,190319695,49.0000,True,45-54,6,Valid Age
9,190411883,60.0000,True,55-64,7,Valid Age


In [35]:
# ============================================================
# Step 9C: Build victim age-group summary
# ============================================================

victim_age_group_summary = (
    victim_age_base
    .groupby(
        ["victim_age_group_order", "victim_age_group", "age_data_quality_flag"],
        dropna=False
    )
    .agg(
        collision_count=("dr_number", "nunique")
    )
    .reset_index()
    .sort_values(["victim_age_group_order", "victim_age_group"])
    .reset_index(drop=True)
)

total_age_group_collisions = victim_age_group_summary["collision_count"].sum()

valid_age_collision_total = victim_age_group_summary.loc[
    victim_age_group_summary["age_data_quality_flag"] == "Valid Age",
    "collision_count"
].sum()

victim_age_group_summary["collision_pct_total"] = (
    victim_age_group_summary["collision_count"] / total_age_group_collisions
)

victim_age_group_summary["collision_pct_valid_age_only"] = np.where(
    victim_age_group_summary["age_data_quality_flag"] == "Valid Age",
    victim_age_group_summary["collision_count"] / valid_age_collision_total,
    np.nan
)

valid_rank_mask = victim_age_group_summary["age_data_quality_flag"] == "Valid Age"

victim_age_group_summary["age_group_rank_valid_only"] = np.nan
victim_age_group_summary.loc[valid_rank_mask, "age_group_rank_valid_only"] = (
    victim_age_group_summary.loc[valid_rank_mask, "collision_count"]
    .rank(method="dense", ascending=False)
)

victim_age_group_summary["collision_pct_total"] = (
    victim_age_group_summary["collision_pct_total"].round(4)
)
victim_age_group_summary["collision_pct_valid_age_only"] = (
    victim_age_group_summary["collision_pct_valid_age_only"].round(4)
)

print("=" * 70)
print("Victim Age Group Summary Created")
print("=" * 70)

display(victim_age_group_summary)

Victim Age Group Summary Created


,victim_age_group_order,victim_age_group,age_data_quality_flag,collision_count,collision_pct_total,collision_pct_valid_age_only,age_group_rank_valid_only
0,1,0-12,Valid Age,373,0.0006,0.0007,8.0000
1,2,13-17,Valid Age,1730,0.0028,0.0032,7.0000
2,3,18-24,Valid Age,76107,0.1224,0.1427,4.0000
3,4,25-34,Valid Age,141627,0.2278,0.2655,1.0000
4,5,35-44,Valid Age,110644,0.1780,0.2074,2.0000
5,6,45-54,Valid Age,94987,0.1528,0.1781,3.0000
6,7,55-64,Valid Age,61596,0.0991,0.1155,5.0000
7,8,65+,Valid Age,46419,0.0747,0.0870,6.0000
8,99,Unknown / Invalid,Unknown / Invalid,88194,0.1419,NaN,NaN


In [36]:
# ============================================================
# Step 9D: Export victim age-group summary
# ============================================================

file_name = "victim_age_group_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    victim_age_group_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\victim_age_group_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\victim_age_group_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\victim_age_group_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\victim_age_group_summary.csv


## Step 10 — Build Victim Sex Summary

### Objective
Create a victim sex summary to describe collision distribution by reported victim sex.

### Why this table matters
This table supports:

- demographic dashboard analysis
- victim profile summaries
- visibility of valid versus invalid / missing sex records
- cross-comparison with age-group results

### Data source
We will use:
`collisions_clean.csv`

because it contains:

- `dr_number`
- `victim_sex`
- `is_valid_victim_sex_analytical`

### Analytical design
Victim sex values will be grouped into clear analytical categories:

- Male
- Female
- Unknown / Invalid

### Why include Unknown / Invalid
Unknown or invalid values should remain visible because:

- they preserve full collision coverage
- they prevent silent exclusion
- they make data quality transparent in reporting outputs

### Primary metric
- distinct collision count using `dr_number`

### Additional metrics
We will compute:

- share of total collisions
- share within valid-sex collisions only
- category rank among valid groups
- explicit sex-data quality flag

### Methodological note
This summary is collision-based, not person-based.
It reflects the victim sex field available in the collision record structure.

In [37]:
# ============================================================
# Step 10A: Validate required columns for victim sex summary
# ============================================================

required_sex_columns = [
    "dr_number",
    "victim_sex",
    "is_valid_victim_sex_analytical"
]

missing_sex_columns = [
    col for col in required_sex_columns
    if col not in collisions_clean.columns
]

if missing_sex_columns:
    raise ValueError(f"Missing required columns: {missing_sex_columns}")

print("All required victim sex summary columns are available ✔")

All required victim sex summary columns are available ✔


In [38]:
# ============================================================
# Step 10B: Build victim sex helper fields
# ============================================================

victim_sex_base = collisions_clean.copy()

def assign_sex_group(sex_value, is_valid):
    if (is_valid is not True) or pd.isna(sex_value):
        return "Unknown / Invalid", 99, "Unknown / Invalid"

    sex_value = str(sex_value).strip().upper()

    if sex_value == "M":
        return "Male", 1, "Valid Sex"
    elif sex_value == "F":
        return "Female", 2, "Valid Sex"
    else:
        return "Unknown / Invalid", 99, "Unknown / Invalid"

sex_assignments = victim_sex_base.apply(
    lambda row: assign_sex_group(row["victim_sex"], row["is_valid_victim_sex_analytical"]),
    axis=1,
    result_type="expand"
)

sex_assignments.columns = [
    "victim_sex_group",
    "victim_sex_group_order",
    "sex_data_quality_flag"
]

victim_sex_base = pd.concat([victim_sex_base, sex_assignments], axis=1)

print("=" * 70)
print("Victim sex helper fields prepared")
print("=" * 70)

display(
    victim_sex_base[
        [
            "dr_number",
            "victim_sex",
            "is_valid_victim_sex_analytical",
            "victim_sex_group",
            "victim_sex_group_order",
            "sex_data_quality_flag"
        ]
    ].head(10)
)

Victim sex helper fields prepared


,dr_number,victim_sex,is_valid_victim_sex_analytical,victim_sex_group,victim_sex_group_order,sex_data_quality_flag
0,212013850,F,True,Female,2,Valid Sex
1,221417787,NaN,False,Unknown / Invalid,99,Unknown / Invalid
2,221418141,NaN,False,Unknown / Invalid,99,Unknown / Invalid
3,222017859,M,True,Male,1,Valid Sex
4,190319651,M,True,Male,1,Valid Sex
5,190319680,F,True,Female,2,Valid Sex
6,190413769,M,True,Male,1,Valid Sex
7,190127578,M,True,Male,1,Valid Sex
8,190319695,M,True,Male,1,Valid Sex
9,190411883,M,True,Male,1,Valid Sex


In [39]:
# ============================================================
# Step 10C: Build victim sex summary
# ============================================================

victim_sex_summary = (
    victim_sex_base
    .groupby(
        ["victim_sex_group_order", "victim_sex_group", "sex_data_quality_flag"],
        dropna=False
    )
    .agg(
        collision_count=("dr_number", "nunique")
    )
    .reset_index()
    .sort_values(["victim_sex_group_order", "victim_sex_group"])
    .reset_index(drop=True)
)

total_sex_collisions = victim_sex_summary["collision_count"].sum()

valid_sex_collision_total = victim_sex_summary.loc[
    victim_sex_summary["sex_data_quality_flag"] == "Valid Sex",
    "collision_count"
].sum()

victim_sex_summary["collision_pct_total"] = (
    victim_sex_summary["collision_count"] / total_sex_collisions
)

victim_sex_summary["collision_pct_valid_sex_only"] = np.where(
    victim_sex_summary["sex_data_quality_flag"] == "Valid Sex",
    victim_sex_summary["collision_count"] / valid_sex_collision_total,
    np.nan
)

valid_rank_mask = victim_sex_summary["sex_data_quality_flag"] == "Valid Sex"

victim_sex_summary["sex_group_rank_valid_only"] = np.nan
victim_sex_summary.loc[valid_rank_mask, "sex_group_rank_valid_only"] = (
    victim_sex_summary.loc[valid_rank_mask, "collision_count"]
    .rank(method="dense", ascending=False)
)

victim_sex_summary["collision_pct_total"] = (
    victim_sex_summary["collision_pct_total"].round(4)
)
victim_sex_summary["collision_pct_valid_sex_only"] = (
    victim_sex_summary["collision_pct_valid_sex_only"].round(4)
)

print("=" * 70)
print("Victim Sex Summary Created")
print("=" * 70)

display(victim_sex_summary)

Victim Sex Summary Created


,victim_sex_group_order,victim_sex_group,sex_data_quality_flag,collision_count,collision_pct_total,collision_pct_valid_sex_only,sex_group_rank_valid_only
0,1,Male,Valid Sex,366612,0.5897,0.6180,1.0000
1,2,Female,Valid Sex,226576,0.3645,0.3820,2.0000
2,99,Unknown / Invalid,Unknown / Invalid,28489,0.0458,NaN,NaN


In [40]:
# ============================================================
# Step 10D: Export victim sex summary
# ============================================================

file_name = "victim_sex_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    victim_sex_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\victim_sex_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\victim_sex_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\victim_sex_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\victim_sex_summary.csv


## Step 11 — Build Victim Descent Summary

### Objective
Create a victim descent summary to describe collision distribution by reported victim descent.

### Why this table matters
This table supports:

- demographic profiling
- victim descent comparisons
- visibility of valid versus invalid / missing descent records
- dashboard segmentation by reported descent group

### Data source
We will use:
`collisions_clean.csv`

because it contains:

- `dr_number`
- `victim_descent`
- `is_valid_victim_descent_analytical`

### Analytical design
Victim descent values will be grouped into interpretable categories using the analytical descent code values.

### Why include Unknown / Invalid
Unknown or invalid values should remain visible because:

- they preserve full collision coverage
- they prevent silent exclusion
- they make data quality transparent in downstream reporting

### Primary metric
- distinct collision count using `dr_number`

### Additional metrics
We will compute:

- share of total collisions
- share within valid-descent collisions only
- descent-group rank among valid groups
- explicit descent-data quality flag

### Methodological note
This summary is collision-based, not person-based.
It reflects the victim descent field available in the collision record structure.

In [41]:
# ============================================================
# Step 11A: Validate required columns for victim descent summary
# ============================================================

required_descent_columns = [
    "dr_number",
    "victim_descent",
    "is_valid_victim_descent_analytical"
]

missing_descent_columns = [
    col for col in required_descent_columns
    if col not in collisions_clean.columns
]

if missing_descent_columns:
    raise ValueError(f"Missing required columns: {missing_descent_columns}")

print("All required victim descent summary columns are available ✔")

All required victim descent summary columns are available ✔


In [42]:
# ============================================================
# Step 11B: Build victim descent helper fields
# ============================================================

victim_descent_base = collisions_clean.copy()

descent_map = {
    "A": "Other Asian",
    "B": "Black",
    "C": "Chinese",
    "D": "Cambodian",
    "F": "Filipino",
    "G": "Guamanian",
    "H": "Hispanic/Latin/Mexican",
    "I": "American Indian/Alaskan Native",
    "J": "Japanese",
    "K": "Korean",
    "L": "Laotian",
    "O": "Other",
    "P": "Pacific Islander",
    "S": "Samoan",
    "U": "Hawaiian",
    "V": "Vietnamese",
    "W": "White",
    "X": "Unknown",
    "Z": "Asian Indian"
}

def assign_descent_group(descent_value, is_valid):
    if (is_valid is not True) or pd.isna(descent_value):
        return "UNK", "Unknown / Invalid", 99, "Unknown / Invalid"

    code = str(descent_value).strip().upper()

    if code in descent_map:
        return code, descent_map[code], 1, "Valid Descent"
    else:
        return code, "Other / Unmapped Valid Code", 98, "Valid Descent"

descent_assignments = victim_descent_base.apply(
    lambda row: assign_descent_group(
        row["victim_descent"],
        row["is_valid_victim_descent_analytical"]
    ),
    axis=1,
    result_type="expand"
)

descent_assignments.columns = [
    "victim_descent_code_clean",
    "victim_descent_group",
    "victim_descent_group_order",
    "descent_data_quality_flag"
]

victim_descent_base = pd.concat([victim_descent_base, descent_assignments], axis=1)

print("=" * 70)
print("Victim descent helper fields prepared")
print("=" * 70)

display(
    victim_descent_base[
        [
            "dr_number",
            "victim_descent",
            "is_valid_victim_descent_analytical",
            "victim_descent_code_clean",
            "victim_descent_group",
            "victim_descent_group_order",
            "descent_data_quality_flag"
        ]
    ].head(12)
)

Victim descent helper fields prepared


,dr_number,victim_descent,is_valid_victim_descent_analytical,victim_descent_code_clean,victim_descent_group,victim_descent_group_order,descent_data_quality_flag
0,212013850,W,True,W,White,1,Valid Descent
1,221417787,NaN,False,UNK,Unknown / Invalid,99,Unknown / Invalid
2,221418141,NaN,False,UNK,Unknown / Invalid,99,Unknown / Invalid
3,222017859,H,True,H,Hispanic/Latin/Mexican,1,Valid Descent
4,190319651,H,True,H,Hispanic/Latin/Mexican,1,Valid Descent
5,190319680,H,True,H,Hispanic/Latin/Mexican,1,Valid Descent
6,190413769,X,True,X,Unknown,1,Valid Descent
7,190127578,H,True,H,Hispanic/Latin/Mexican,1,Valid Descent
8,190319695,B,True,B,Black,1,Valid Descent
9,190411883,H,True,H,Hispanic/Latin/Mexican,1,Valid Descent


In [43]:
# ============================================================
# Step 11C: Build victim descent summary
# ============================================================

victim_descent_summary = (
    victim_descent_base
    .groupby(
        [
            "victim_descent_group_order",
            "victim_descent_code_clean",
            "victim_descent_group",
            "descent_data_quality_flag"
        ],
        dropna=False
    )
    .agg(
        collision_count=("dr_number", "nunique")
    )
    .reset_index()
)

total_descent_collisions = victim_descent_summary["collision_count"].sum()

valid_descent_collision_total = victim_descent_summary.loc[
    victim_descent_summary["descent_data_quality_flag"] == "Valid Descent",
    "collision_count"
].sum()

victim_descent_summary["collision_pct_total"] = (
    victim_descent_summary["collision_count"] / total_descent_collisions
)

victim_descent_summary["collision_pct_valid_descent_only"] = np.where(
    victim_descent_summary["descent_data_quality_flag"] == "Valid Descent",
    victim_descent_summary["collision_count"] / valid_descent_collision_total,
    np.nan
)

victim_descent_summary["descent_group_rank_valid_only"] = np.nan

valid_rank_mask = victim_descent_summary["descent_data_quality_flag"] == "Valid Descent"

victim_descent_summary.loc[valid_rank_mask, "descent_group_rank_valid_only"] = (
    victim_descent_summary.loc[valid_rank_mask, "collision_count"]
    .rank(method="dense", ascending=False)
)

victim_descent_summary["collision_pct_total"] = (
    victim_descent_summary["collision_pct_total"].round(4)
)
victim_descent_summary["collision_pct_valid_descent_only"] = (
    victim_descent_summary["collision_pct_valid_descent_only"].round(4)
)

# Sort: valid groups first by collision volume, then unknown/invalid at the end
victim_descent_summary["quality_sort_order"] = np.where(
    victim_descent_summary["descent_data_quality_flag"] == "Valid Descent",
    1,
    2
)

victim_descent_summary = victim_descent_summary.sort_values(
    ["quality_sort_order", "collision_count", "victim_descent_group"],
    ascending=[True, False, True]
).reset_index(drop=True)

victim_descent_summary = victim_descent_summary.drop(columns=["quality_sort_order"])

print("=" * 70)
print("Victim Descent Summary Created")
print("=" * 70)

display(victim_descent_summary)

Victim Descent Summary Created


,victim_descent_group_order,victim_descent_code_clean,victim_descent_group,descent_data_quality_flag,collision_count,collision_pct_total,collision_pct_valid_descent_only,descent_group_rank_valid_only
0,1,H,Hispanic/Latin/Mexican,Valid Descent,235060,0.3781,0.3853,1.0000
1,1,W,White,Valid Descent,140905,0.2267,0.2310,2.0000
2,1,O,Other,Valid Descent,90589,0.1457,0.1485,3.0000
3,1,B,Black,Valid Descent,82238,0.1323,0.1348,4.0000
4,1,X,Unknown,Valid Descent,29918,0.0481,0.0490,5.0000
5,1,A,Other Asian,Valid Descent,22325,0.0359,0.0366,6.0000
6,1,K,Korean,Valid Descent,4553,0.0073,0.0075,7.0000
7,1,F,Filipino,Valid Descent,1776,0.0029,0.0029,8.0000
8,1,C,Chinese,Valid Descent,918,0.0015,0.0015,9.0000
9,1,U,Hawaiian,Valid Descent,474,0.0008,0.0008,10.0000


In [44]:
# ============================================================
# Step 11D: Export victim descent summary
# ============================================================

file_name = "victim_descent_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    victim_descent_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\victim_descent_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\victim_descent_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\victim_descent_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\victim_descent_summary.csv


## Step 12 — Build Vulnerable Users Summary

### Objective
Create a vulnerable-users summary using MO-enriched collision records.

### Why this table matters
This table supports:

- safety-focused dashboard views
- pedestrian and bicyclist monitoring
- vulnerable-road-user storytelling
- comparison between vulnerable-user collision patterns

### Data source
We will use:
`fact_collision_mo_enriched.csv`

because vulnerable-user indicators are better represented in the MO analytical layer.

### Analytical design
We will identify collision involvement for:

- Pedestrian-related collisions
- Bicycle-related collisions

using MO analytical descriptions and categories.

### Why this table should come from the MO layer
The collision-level table contains the raw `mo_codes` string,
but the MO-enriched analytical table already provides structured classifications such as:

- `analytical_category`
- `analytical_subcategory`
- `analytical_domain`
- `mo_description`

This makes vulnerable-user detection more controlled and explainable.

### Primary metrics
We will compute:

- distinct collision count
- code-row count
- share of total vulnerable-user flagged collisions

### Methodological note
Because `fact_collision_mo_enriched.csv` has multiple rows per collision,
collision totals must use:

- distinct `dr_number`

while code-row intensity can be retained separately as:
- `mo_row_count`

In [45]:
# ============================================================
# Step 12A: Validate required columns for vulnerable users summary
# ============================================================

required_vulnerable_columns = [
    "dr_number",
    "mo_description",
    "analytical_category",
    "analytical_subcategory",
    "analytical_domain"
]

missing_vulnerable_columns = [
    col for col in required_vulnerable_columns
    if col not in fact_collision_mo.columns
]

if missing_vulnerable_columns:
    raise ValueError(f"Missing required columns: {missing_vulnerable_columns}")

print("All required vulnerable-user columns are available ✔")

All required vulnerable-user columns are available ✔


In [46]:
# ============================================================
# Step 12B: Build vulnerable-user flags from MO-enriched table
# ============================================================

vulnerable_base = fact_collision_mo.copy()

# Normalize searchable text
for col in ["mo_description", "analytical_category", "analytical_subcategory", "analytical_domain"]:
    vulnerable_base[col] = vulnerable_base[col].astype(str)

combined_text = (
    vulnerable_base["mo_description"].str.lower().fillna("") + " | " +
    vulnerable_base["analytical_category"].str.lower().fillna("") + " | " +
    vulnerable_base["analytical_subcategory"].str.lower().fillna("") + " | " +
    vulnerable_base["analytical_domain"].str.lower().fillna("")
)

vulnerable_base["is_pedestrian_related"] = combined_text.str.contains("pedestrian", na=False)
vulnerable_base["is_bicycle_related"] = (
    combined_text.str.contains("bicycle", na=False) |
    combined_text.str.contains("bicycl", na=False) |
    combined_text.str.contains("bike", na=False)
)

print("=" * 70)
print("Vulnerable-user flags prepared")
print("=" * 70)

display(
    vulnerable_base[
        [
            "dr_number",
            "mo_description",
            "analytical_category",
            "analytical_subcategory",
            "analytical_domain",
            "is_pedestrian_related",
            "is_bicycle_related"
        ]
    ].head(15)
)

Vulnerable-user flags prepared


,dr_number,mo_description,analytical_category,analytical_subcategory,analytical_domain,is_pedestrian_related,is_bicycle_related
0,212013850,T/C – Vehicle vs vehicle,Traffic Collision,Collision counterpart / mode pairing,Traffic Collision Core,False,False
1,212013850,T/C – (K) Fatal injury,Traffic Collision,Injury severity,Traffic Collision Core,False,False
2,212013850,T/C – City property involved: Yes,Traffic Collision,Scene context / property / intersection,Traffic Collision Core,False,False
3,212013850,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,Traffic Collision Jurisdiction,False,False
4,212013850,T/C – At intersection: Yes,Traffic Collision,Scene context / property / intersection,Traffic Collision Core,False,False
5,212013850,T/C – Primary collision factor (A) in narrative,Traffic Collision Cause,Primary collision factor family,Traffic Collision Causation,False,False
6,212013850,T/C – Type of collision,Collision Pattern,Type of collision,Traffic Collision Type,False,False
7,212013850,T/C – Movement preceding collision,Pre-Collision Movement,Movement preceding collision,Traffic Collision Pre-Collision Movement,False,False
8,221417787,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,Traffic Collision Jurisdiction,False,False
9,221417787,T/C – Vehicle vs fixed object,Traffic Collision,Collision counterpart / mode pairing,Traffic Collision Core,False,False


In [47]:
# ============================================================
# Step 12C: Reshape vulnerable-user categories
# ============================================================

pedestrian_rows = vulnerable_base.loc[
    vulnerable_base["is_pedestrian_related"],
    ["dr_number"]
].copy()
pedestrian_rows["vulnerable_user_group"] = "Pedestrian-related"

bicycle_rows = vulnerable_base.loc[
    vulnerable_base["is_bicycle_related"],
    ["dr_number"]
].copy()
bicycle_rows["vulnerable_user_group"] = "Bicycle-related"

vulnerable_long = pd.concat(
    [pedestrian_rows, bicycle_rows],
    ignore_index=True
)

print("=" * 70)
print("Vulnerable-user long table created")
print("=" * 70)

display(vulnerable_long.head(15))

Vulnerable-user long table created


,dr_number,vulnerable_user_group
0,222017859,Pedestrian-related
1,190411883,Pedestrian-related
2,190514552,Pedestrian-related
3,190616887,Pedestrian-related
4,190617709,Pedestrian-related
5,190617739,Pedestrian-related
6,190617926,Pedestrian-related
7,190618066,Pedestrian-related
8,190618082,Pedestrian-related
9,190713486,Pedestrian-related


In [48]:
# ============================================================
# Step 12D: Build vulnerable users summary
# ============================================================

vulnerable_users_summary = (
    vulnerable_long
    .groupby("vulnerable_user_group")
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_vulnerable_collisions = vulnerable_users_summary["collision_count"].sum()

vulnerable_users_summary["collision_pct_within_vulnerable_groups"] = (
    vulnerable_users_summary["collision_count"] / total_vulnerable_collisions
)

vulnerable_users_summary["collision_pct_within_vulnerable_groups"] = (
    vulnerable_users_summary["collision_pct_within_vulnerable_groups"].round(4)
)

vulnerable_users_summary["group_rank_by_collision_count"] = (
    vulnerable_users_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

vulnerable_users_summary = vulnerable_users_summary.sort_values(
    ["collision_count", "vulnerable_user_group"],
    ascending=[False, True]
).reset_index(drop=True)

print("=" * 70)
print("Vulnerable Users Summary Created")
print("=" * 70)

display(vulnerable_users_summary)

Vulnerable Users Summary Created


,vulnerable_user_group,collision_count,mo_row_count,collision_pct_within_vulnerable_groups,group_rank_by_collision_count
0,Pedestrian-related,37027,37661,0.6311,1
1,Bicycle-related,21639,21915,0.3689,2


In [49]:
# ============================================================
# Step 12E: Export vulnerable users summary
# ============================================================

file_name = "vulnerable_users_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    vulnerable_users_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\vulnerable_users_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\vulnerable_users_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\vulnerable_users_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\vulnerable_users_summary.csv


## Section 3 — MO-Based and Dashboard-Focused Aggregations

## Step 13 — Build MO Analytical Domain Summary

### Objective
Create an MO analytical domain summary using the MO-enriched collision table.

### Why this table matters
This table supports:

- high-level MO dashboard structure
- comparison between major analytical MO domains
- explanation of what types of collision intelligence dominate the dataset
- drill-down planning for later category and subcategory summaries

### Data source
We will use:
`fact_collision_mo_enriched.csv`

because it already contains structured MO classifications such as:

- `analytical_domain`
- `analytical_category`
- `analytical_subcategory`
- `mo_description`

### Why this table must come from the MO layer
The collision-level table stores MO information as a raw combined string,
while the MO-enriched table provides one classified row per collision-code relationship.

This makes the analytical-domain layer:

- structured
- reproducible
- explainable
- ready for dashboard drill-down logic

### Primary metrics
We will compute:

- distinct collision count by analytical domain
- MO row count by analytical domain

### Additional metrics
We will also compute:

- collision coverage rate relative to all unique collisions in the MO table
- MO row share relative to all MO-enriched rows
- domain rank by collision count

### Methodological note
Because one collision may appear in multiple MO analytical domains:

- `collision_count` should be interpreted as domain coverage
- `mo_row_count` should be interpreted as domain intensity

These two metrics should be kept together rather than replacing one another.

In [50]:
# ============================================================
# Step 13A: Validate required columns for MO analytical domain summary
# ============================================================

required_mo_domain_columns = [
    "dr_number",
    "analytical_domain"
]

missing_mo_domain_columns = [
    col for col in required_mo_domain_columns
    if col not in fact_collision_mo.columns
]

if missing_mo_domain_columns:
    raise ValueError(f"Missing required columns: {missing_mo_domain_columns}")

print("All required MO analytical domain columns are available ✔")

All required MO analytical domain columns are available ✔


In [51]:
# ============================================================
# Step 13B: Prepare MO analytical domain base
# ============================================================

mo_domain_base = fact_collision_mo.copy()

mo_domain_base["analytical_domain"] = (
    mo_domain_base["analytical_domain"]
    .astype(str)
    .str.strip()
)

mo_domain_base["analytical_domain"] = mo_domain_base["analytical_domain"].replace({
    "": "Unknown / Unclassified",
    "nan": "Unknown / Unclassified",
    "None": "Unknown / Unclassified"
})

print("=" * 70)
print("MO analytical domain base prepared")
print("=" * 70)

display(
    mo_domain_base[
        [
            "dr_number",
            "mo_description",
            "analytical_category",
            "analytical_subcategory",
            "analytical_domain"
        ]
    ].head(15)
)

MO analytical domain base prepared


,dr_number,mo_description,analytical_category,analytical_subcategory,analytical_domain
0,212013850,T/C – Vehicle vs vehicle,Traffic Collision,Collision counterpart / mode pairing,Traffic Collision Core
1,212013850,T/C – (K) Fatal injury,Traffic Collision,Injury severity,Traffic Collision Core
2,212013850,T/C – City property involved: Yes,Traffic Collision,Scene context / property / intersection,Traffic Collision Core
3,212013850,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,Traffic Collision Jurisdiction
4,212013850,T/C – At intersection: Yes,Traffic Collision,Scene context / property / intersection,Traffic Collision Core
5,212013850,T/C – Primary collision factor (A) in narrative,Traffic Collision Cause,Primary collision factor family,Traffic Collision Causation
6,212013850,T/C – Type of collision,Collision Pattern,Type of collision,Traffic Collision Type
7,212013850,T/C – Movement preceding collision,Pre-Collision Movement,Movement preceding collision,Traffic Collision Pre-Collision Movement
8,221417787,T/C – West Traffic Division (WTD),Reporting Division,LAPD division / traffic division,Traffic Collision Jurisdiction
9,221417787,T/C – Vehicle vs fixed object,Traffic Collision,Collision counterpart / mode pairing,Traffic Collision Core


In [52]:
# ============================================================
# Step 13C: Build MO analytical domain summary
# ============================================================

mo_analytical_domain_summary = (
    mo_domain_base
    .groupby("analytical_domain", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_unique_mo_collisions = mo_domain_base["dr_number"].nunique()
total_mo_rows = len(mo_domain_base)

mo_analytical_domain_summary["collision_coverage_rate"] = (
    mo_analytical_domain_summary["collision_count"] / total_unique_mo_collisions
)

mo_analytical_domain_summary["mo_row_share"] = (
    mo_analytical_domain_summary["mo_row_count"] / total_mo_rows
)

mo_analytical_domain_summary["domain_rank_by_collision_count"] = (
    mo_analytical_domain_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

mo_analytical_domain_summary["collision_coverage_rate"] = (
    mo_analytical_domain_summary["collision_coverage_rate"].round(4)
)
mo_analytical_domain_summary["mo_row_share"] = (
    mo_analytical_domain_summary["mo_row_share"].round(4)
)

mo_analytical_domain_summary = mo_analytical_domain_summary.sort_values(
    ["collision_count", "mo_row_count", "analytical_domain"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("MO Analytical Domain Summary Created")
print("=" * 70)

display(mo_analytical_domain_summary)

MO Analytical Domain Summary Created


,analytical_domain,collision_count,mo_row_count,collision_coverage_rate,mo_row_share,domain_rank_by_collision_count
0,Traffic Collision Core,505112,1801303,0.9453,0.5115,1
1,Traffic Collision Causation,484688,486978,0.9071,0.1383,2
2,Traffic Collision Jurisdiction,371557,372037,0.6953,0.1056,3
3,Traffic Collision Type,361373,361373,0.6763,0.1026,4
4,Traffic Collision Pre-Collision Movement,361237,361237,0.6760,0.1026,5
5,Incident Context,83904,83906,0.1570,0.0238,6
6,Traffic Collision Special Factors,16660,16717,0.0312,0.0047,7
7,Other / Unspecified,12366,12366,0.0231,0.0035,8
8,Vehicle Interaction,9385,9464,0.0176,0.0027,9
9,Evidence / Forensics,4823,4989,0.0090,0.0014,10


In [53]:
# ============================================================
# Step 13D: Export MO analytical domain summary
# ============================================================

file_name = "mo_analytical_domain_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    mo_analytical_domain_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\mo_analytical_domain_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\mo_analytical_domain_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\mo_analytical_domain_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\mo_analytical_domain_summary.csv


## Step 14 — Build MO Analytical Category Summary

### Objective
Create an MO analytical category summary as the second drill-down layer below analytical domain.

### Why this table matters
This table supports:

- MO dashboard drill-down from domain to category
- more detailed interpretation of collision intelligence patterns
- identification of the strongest category buckets inside each analytical structure
- downstream preparation for subcategory-level summaries

### Data source
We will use:
`fact_collision_mo_enriched.csv`

because it contains structured MO classification fields such as:

- `analytical_domain`
- `analytical_category`
- `analytical_subcategory`

### Why this table is important after domain summary
The analytical domain summary is useful for the highest-level story,
but category-level detail is needed to explain what specifically drives each domain.

For example:
- within traffic collision domains,
  categories may distinguish collision pattern, cause, reporting division, or movement logic

### Primary metrics
We will compute:

- distinct collision count by analytical category
- MO row count by analytical category

### Additional metrics
We will also compute:

- parent analytical domain
- collision coverage rate relative to all unique MO collisions
- MO row share relative to all MO-enriched rows
- category rank by collision count

### Methodological note
Because one collision may contribute to multiple analytical categories:

- `collision_count` represents category coverage
- `mo_row_count` represents category intensity

These metrics should be interpreted together.

In [54]:
# ============================================================
# Step 14A: Validate required columns for MO analytical category summary
# ============================================================

required_mo_category_columns = [
    "dr_number",
    "analytical_domain",
    "analytical_category"
]

missing_mo_category_columns = [
    col for col in required_mo_category_columns
    if col not in fact_collision_mo.columns
]

if missing_mo_category_columns:
    raise ValueError(f"Missing required columns: {missing_mo_category_columns}")

print("All required MO analytical category columns are available ✔")

All required MO analytical category columns are available ✔


In [55]:
# ============================================================
# Step 14B: Prepare MO analytical category base
# ============================================================

mo_category_base = fact_collision_mo.copy()

for col in ["analytical_domain", "analytical_category"]:
    mo_category_base[col] = (
        mo_category_base[col]
        .astype(str)
        .str.strip()
        .replace({
            "": "Unknown / Unclassified",
            "nan": "Unknown / Unclassified",
            "None": "Unknown / Unclassified"
        })
    )

print("=" * 70)
print("MO analytical category base prepared")
print("=" * 70)

display(
    mo_category_base[
        [
            "dr_number",
            "mo_description",
            "analytical_domain",
            "analytical_category",
            "analytical_subcategory"
        ]
    ].head(15)
)

MO analytical category base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory
0,212013850,T/C – Vehicle vs vehicle,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing
1,212013850,T/C – (K) Fatal injury,Traffic Collision Core,Traffic Collision,Injury severity
2,212013850,T/C – City property involved: Yes,Traffic Collision Core,Traffic Collision,Scene context / property / intersection
3,212013850,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division
4,212013850,T/C – At intersection: Yes,Traffic Collision Core,Traffic Collision,Scene context / property / intersection
5,212013850,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family
6,212013850,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision
7,212013850,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision
8,221417787,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division
9,221417787,T/C – Vehicle vs fixed object,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing


In [56]:
# ============================================================
# Step 14C: Build MO analytical category summary
# ============================================================

mo_analytical_category_summary = (
    mo_category_base
    .groupby(["analytical_domain", "analytical_category"], dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_unique_mo_collisions = mo_category_base["dr_number"].nunique()
total_mo_rows = len(mo_category_base)

mo_analytical_category_summary["collision_coverage_rate"] = (
    mo_analytical_category_summary["collision_count"] / total_unique_mo_collisions
)

mo_analytical_category_summary["mo_row_share"] = (
    mo_analytical_category_summary["mo_row_count"] / total_mo_rows
)

mo_analytical_category_summary["category_rank_by_collision_count"] = (
    mo_analytical_category_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

mo_analytical_category_summary["collision_coverage_rate"] = (
    mo_analytical_category_summary["collision_coverage_rate"].round(4)
)
mo_analytical_category_summary["mo_row_share"] = (
    mo_analytical_category_summary["mo_row_share"].round(4)
)

mo_analytical_category_summary = mo_analytical_category_summary.sort_values(
    ["collision_count", "mo_row_count", "analytical_domain", "analytical_category"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

print("=" * 70)
print("MO Analytical Category Summary Created")
print("=" * 70)

display(mo_analytical_category_summary)

MO Analytical Category Summary Created


,analytical_domain,analytical_category,collision_count,mo_row_count,collision_coverage_rate,mo_row_share,category_rank_by_collision_count
0,Traffic Collision Core,Traffic Collision,505112,1801303,0.9453,0.5115,1
1,Traffic Collision Causation,Traffic Collision Cause,484688,486978,0.9071,0.1383,2
2,Traffic Collision Jurisdiction,Reporting Division,371557,372037,0.6953,0.1056,3
3,Traffic Collision Type,Collision Pattern,361373,361373,0.6763,0.1026,4
4,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,361237,361237,0.6760,0.1026,5
5,Incident Context,Incident Context,83904,83906,0.1570,0.0238,6
6,Traffic Collision Special Factors,Special Collision Factors,16660,16717,0.0312,0.0047,7
7,Other / Unspecified,Other / Narrative,12366,12366,0.0231,0.0035,8
8,Vehicle Interaction,Vehicle-Centered Interaction,9385,9464,0.0176,0.0027,9
9,Evidence / Forensics,Forensic Evidence,4823,4989,0.0090,0.0014,10


In [57]:
# ============================================================
# Step 14D: Add within-domain share
# ============================================================

domain_collision_totals = (
    mo_analytical_category_summary
    .groupby("analytical_domain")["collision_count"]
    .transform("sum")
)

domain_mo_row_totals = (
    mo_analytical_category_summary
    .groupby("analytical_domain")["mo_row_count"]
    .transform("sum")
)

mo_analytical_category_summary["category_collision_share_within_domain"] = (
    mo_analytical_category_summary["collision_count"] / domain_collision_totals
)

mo_analytical_category_summary["category_mo_row_share_within_domain"] = (
    mo_analytical_category_summary["mo_row_count"] / domain_mo_row_totals
)

mo_analytical_category_summary["category_collision_share_within_domain"] = (
    mo_analytical_category_summary["category_collision_share_within_domain"].round(4)
)

mo_analytical_category_summary["category_mo_row_share_within_domain"] = (
    mo_analytical_category_summary["category_mo_row_share_within_domain"].round(4)
)

display(mo_analytical_category_summary.head(25))

,analytical_domain,analytical_category,collision_count,mo_row_count,collision_coverage_rate,mo_row_share,category_rank_by_collision_count,category_collision_share_within_domain,category_mo_row_share_within_domain
0,Traffic Collision Core,Traffic Collision,505112,1801303,0.9453,0.5115,1,1.0000,1.0000
1,Traffic Collision Causation,Traffic Collision Cause,484688,486978,0.9071,0.1383,2,1.0000,1.0000
2,Traffic Collision Jurisdiction,Reporting Division,371557,372037,0.6953,0.1056,3,1.0000,1.0000
3,Traffic Collision Type,Collision Pattern,361373,361373,0.6763,0.1026,4,1.0000,1.0000
4,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,361237,361237,0.6760,0.1026,5,1.0000,1.0000
5,Incident Context,Incident Context,83904,83906,0.1570,0.0238,6,1.0000,1.0000
6,Traffic Collision Special Factors,Special Collision Factors,16660,16717,0.0312,0.0047,7,1.0000,1.0000
7,Other / Unspecified,Other / Narrative,12366,12366,0.0231,0.0035,8,1.0000,1.0000
8,Vehicle Interaction,Vehicle-Centered Interaction,9385,9464,0.0176,0.0027,9,1.0000,1.0000
9,Evidence / Forensics,Forensic Evidence,4823,4989,0.0090,0.0014,10,1.0000,1.0000


In [58]:
# ============================================================
# Step 14E: Export MO analytical category summary
# ============================================================

file_name = "mo_analytical_category_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    mo_analytical_category_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\mo_analytical_category_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\mo_analytical_category_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\mo_analytical_category_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\mo_analytical_category_summary.csv


## Step 15 — Build MO Analytical Subcategory Summary

### Objective
Create an MO analytical subcategory summary as the most detailed structured drill-down layer in the MO classification hierarchy.

### Why this table matters
This table supports:

- the deepest MO dashboard drill-down
- explanation of what specifically drives each analytical category
- ranking of detailed traffic-collision concepts
- identification of the most repeated collision semantics in the MO layer

### Data source
We will use:
`fact_collision_mo_enriched.csv`

because it contains the full structured MO hierarchy:

- `analytical_domain`
- `analytical_category`
- `analytical_subcategory`

### Why this table is especially important here
The category summary showed that, in the current classification structure,
most analytical domains map almost one-to-one to analytical categories.

Therefore, the subcategory layer becomes the first truly granular drill-down level.

### Primary metrics
We will compute:

- distinct collision count by analytical subcategory
- MO row count by analytical subcategory

### Additional metrics
We will also compute:

- parent analytical domain
- parent analytical category
- collision coverage rate relative to all unique MO collisions
- MO row share relative to all MO-enriched rows
- subcategory rank by collision count
- subcategory share within parent category

### Methodological note
Because one collision may contribute to multiple analytical subcategories:

- `collision_count` represents subcategory coverage
- `mo_row_count` represents subcategory intensity

These two metrics should be interpreted together.

In [59]:
# ============================================================
# Step 15A: Validate required columns for MO analytical subcategory summary
# ============================================================

required_mo_subcategory_columns = [
    "dr_number",
    "analytical_domain",
    "analytical_category",
    "analytical_subcategory"
]

missing_mo_subcategory_columns = [
    col for col in required_mo_subcategory_columns
    if col not in fact_collision_mo.columns
]

if missing_mo_subcategory_columns:
    raise ValueError(f"Missing required columns: {missing_mo_subcategory_columns}")

print("All required MO analytical subcategory columns are available ✔")

All required MO analytical subcategory columns are available ✔


In [60]:
# ============================================================
# Step 15B: Prepare MO analytical subcategory base
# ============================================================

mo_subcategory_base = fact_collision_mo.copy()

for col in ["analytical_domain", "analytical_category", "analytical_subcategory"]:
    mo_subcategory_base[col] = (
        mo_subcategory_base[col]
        .astype(str)
        .str.strip()
        .replace({
            "": "Unknown / Unclassified",
            "nan": "Unknown / Unclassified",
            "None": "Unknown / Unclassified"
        })
    )

print("=" * 70)
print("MO analytical subcategory base prepared")
print("=" * 70)

display(
    mo_subcategory_base[
        [
            "dr_number",
            "mo_description",
            "analytical_domain",
            "analytical_category",
            "analytical_subcategory"
        ]
    ].head(15)
)

MO analytical subcategory base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory
0,212013850,T/C – Vehicle vs vehicle,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing
1,212013850,T/C – (K) Fatal injury,Traffic Collision Core,Traffic Collision,Injury severity
2,212013850,T/C – City property involved: Yes,Traffic Collision Core,Traffic Collision,Scene context / property / intersection
3,212013850,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division
4,212013850,T/C – At intersection: Yes,Traffic Collision Core,Traffic Collision,Scene context / property / intersection
5,212013850,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family
6,212013850,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision
7,212013850,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision
8,221417787,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division
9,221417787,T/C – Vehicle vs fixed object,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing


In [61]:
# ============================================================
# Step 15C: Build MO analytical subcategory summary
# ============================================================

mo_analytical_subcategory_summary = (
    mo_subcategory_base
    .groupby(
        ["analytical_domain", "analytical_category", "analytical_subcategory"],
        dropna=False
    )
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_unique_mo_collisions = mo_subcategory_base["dr_number"].nunique()
total_mo_rows = len(mo_subcategory_base)

mo_analytical_subcategory_summary["collision_coverage_rate"] = (
    mo_analytical_subcategory_summary["collision_count"] / total_unique_mo_collisions
)

mo_analytical_subcategory_summary["mo_row_share"] = (
    mo_analytical_subcategory_summary["mo_row_count"] / total_mo_rows
)

mo_analytical_subcategory_summary["subcategory_rank_by_collision_count"] = (
    mo_analytical_subcategory_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

mo_analytical_subcategory_summary["collision_coverage_rate"] = (
    mo_analytical_subcategory_summary["collision_coverage_rate"].round(4)
)
mo_analytical_subcategory_summary["mo_row_share"] = (
    mo_analytical_subcategory_summary["mo_row_share"].round(4)
)

mo_analytical_subcategory_summary = mo_analytical_subcategory_summary.sort_values(
    [
        "collision_count",
        "mo_row_count",
        "analytical_domain",
        "analytical_category",
        "analytical_subcategory"
    ],
    ascending=[False, False, True, True, True]
).reset_index(drop=True)

print("=" * 70)
print("MO Analytical Subcategory Summary Created")
print("=" * 70)

display(mo_analytical_subcategory_summary)

MO Analytical Subcategory Summary Created


,analytical_domain,analytical_category,analytical_subcategory,collision_count,mo_row_count,collision_coverage_rate,mo_row_share,subcategory_rank_by_collision_count
0,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing,498050,504375,0.9321,0.1432,1
1,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,484688,486978,0.9071,0.1383,2
2,Traffic Collision Core,Traffic Collision,Injury severity,419091,419994,0.7843,0.1193,3
3,Traffic Collision Core,Traffic Collision,Scene context / property / intersection,407841,607186,0.7632,0.1724,4
4,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,371557,372037,0.6953,0.1056,5
5,Traffic Collision Type,Collision Pattern,Type of collision,361373,361373,0.6763,0.1026,6
6,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,361237,361237,0.6760,0.1026,7
7,Traffic Collision Core,Traffic Collision,Hit-and-run status,258871,258982,0.4845,0.0735,8
8,Incident Context,Incident Context,Traffic / roadway incident context,83772,83772,0.1568,0.0238,9
9,Traffic Collision Special Factors,Special Collision Factors,Unlicensed motorist,16033,16033,0.0300,0.0046,10


In [62]:
# ============================================================
# Step 15D: Add within-category share
# ============================================================

category_collision_totals = (
    mo_analytical_subcategory_summary
    .groupby(["analytical_domain", "analytical_category"])["collision_count"]
    .transform("sum")
)

category_mo_row_totals = (
    mo_analytical_subcategory_summary
    .groupby(["analytical_domain", "analytical_category"])["mo_row_count"]
    .transform("sum")
)

mo_analytical_subcategory_summary["subcategory_collision_share_within_category"] = (
    mo_analytical_subcategory_summary["collision_count"] / category_collision_totals
)

mo_analytical_subcategory_summary["subcategory_mo_row_share_within_category"] = (
    mo_analytical_subcategory_summary["mo_row_count"] / category_mo_row_totals
)

mo_analytical_subcategory_summary["subcategory_collision_share_within_category"] = (
    mo_analytical_subcategory_summary["subcategory_collision_share_within_category"].round(4)
)

mo_analytical_subcategory_summary["subcategory_mo_row_share_within_category"] = (
    mo_analytical_subcategory_summary["subcategory_mo_row_share_within_category"].round(4)
)

display(mo_analytical_subcategory_summary.head(30))

,analytical_domain,analytical_category,analytical_subcategory,collision_count,mo_row_count,collision_coverage_rate,mo_row_share,subcategory_rank_by_collision_count,subcategory_collision_share_within_category,subcategory_mo_row_share_within_category
0,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing,498050,504375,0.9321,0.1432,1,0.3123,0.2800
1,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,484688,486978,0.9071,0.1383,2,1.0000,1.0000
2,Traffic Collision Core,Traffic Collision,Injury severity,419091,419994,0.7843,0.1193,3,0.2628,0.2332
3,Traffic Collision Core,Traffic Collision,Scene context / property / intersection,407841,607186,0.7632,0.1724,4,0.2558,0.3371
4,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,371557,372037,0.6953,0.1056,5,1.0000,1.0000
5,Traffic Collision Type,Collision Pattern,Type of collision,361373,361373,0.6763,0.1026,6,1.0000,1.0000
6,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,361237,361237,0.6760,0.1026,7,1.0000,1.0000
7,Traffic Collision Core,Traffic Collision,Hit-and-run status,258871,258982,0.4845,0.0735,8,0.1623,0.1438
8,Incident Context,Incident Context,Traffic / roadway incident context,83772,83772,0.1568,0.0238,9,0.9984,0.9984
9,Traffic Collision Special Factors,Special Collision Factors,Unlicensed motorist,16033,16033,0.0300,0.0046,10,0.9591,0.9591


In [63]:
# ============================================================
# Step 15E: Export MO analytical subcategory summary
# ============================================================

file_name = "mo_analytical_subcategory_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    mo_analytical_subcategory_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\mo_analytical_subcategory_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\mo_analytical_subcategory_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\mo_analytical_subcategory_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\mo_analytical_subcategory_summary.csv


## Step 16 — Build Primary Entity Summary

### Objective
Create a primary-entity summary using the MO-enriched collision table.

### Why this table matters
This table provides a simplified cross-cutting analytical view of the MO layer by identifying
the main entity referenced by each classified MO record.

### Why primary_entity is useful
Compared with domain, category, and subcategory, `primary_entity` is more compact and easier to explain.

It helps answer questions such as:

- Are the classified MO records mainly collision-centered?
- How much of the analytical structure is vehicle-focused?
- How much is victim-focused?
- How much is jurisdiction-focused?

### Data source
We will use:
`fact_collision_mo_enriched.csv`

because it already includes:
- `primary_entity`
- `dr_number`
- the full classified MO structure

### Primary metrics
We will compute:

- distinct collision count by primary entity
- MO row count by primary entity

### Additional metrics
We will also compute:

- collision coverage rate relative to all unique MO collisions
- MO row share relative to all MO-enriched rows
- entity rank by collision count

### Methodological note
Because one collision may contribute to multiple MO rows and potentially multiple entities:

- `collision_count` represents entity coverage
- `mo_row_count` represents entity intensity

These two metrics should be interpreted together.

### Analytical positioning
This table is a complementary view.
It does not replace domain, category, or subcategory summaries.
Instead, it offers a simpler semantic layer for dashboards and storytelling.

In [64]:
# ============================================================
# Step 16A: Validate required columns for primary entity summary
# ============================================================

required_primary_entity_columns = [
    "dr_number",
    "primary_entity"
]

missing_primary_entity_columns = [
    col for col in required_primary_entity_columns
    if col not in fact_collision_mo.columns
]

if missing_primary_entity_columns:
    raise ValueError(f"Missing required columns: {missing_primary_entity_columns}")

print("All required primary_entity columns are available ✔")

All required primary_entity columns are available ✔


In [65]:
# ============================================================
# Step 16B: Prepare primary entity base
# ============================================================

primary_entity_base = fact_collision_mo.copy()

primary_entity_base["primary_entity"] = (
    primary_entity_base["primary_entity"]
    .astype(str)
    .str.strip()
)

primary_entity_base["primary_entity"] = primary_entity_base["primary_entity"].replace({
    "": "Unknown / Unclassified",
    "nan": "Unknown / Unclassified",
    "None": "Unknown / Unclassified"
})

print("=" * 70)
print("Primary entity base prepared")
print("=" * 70)

display(
    primary_entity_base[
        [
            "dr_number",
            "mo_description",
            "analytical_domain",
            "analytical_category",
            "analytical_subcategory",
            "primary_entity"
        ]
    ].head(15)
)

Primary entity base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory,primary_entity
0,212013850,T/C – Vehicle vs vehicle,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing,Collision
1,212013850,T/C – (K) Fatal injury,Traffic Collision Core,Traffic Collision,Injury severity,Collision
2,212013850,T/C – City property involved: Yes,Traffic Collision Core,Traffic Collision,Scene context / property / intersection,Collision
3,212013850,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,Jurisdiction
4,212013850,T/C – At intersection: Yes,Traffic Collision Core,Traffic Collision,Scene context / property / intersection,Collision
5,212013850,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Collision
6,212013850,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Collision
7,212013850,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Collision
8,221417787,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,Jurisdiction
9,221417787,T/C – Vehicle vs fixed object,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing,Collision


In [66]:
# ============================================================
# Step 16C: Build primary entity summary
# ============================================================

primary_entity_summary = (
    primary_entity_base
    .groupby("primary_entity", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_unique_mo_collisions = primary_entity_base["dr_number"].nunique()
total_mo_rows = len(primary_entity_base)

primary_entity_summary["collision_coverage_rate"] = (
    primary_entity_summary["collision_count"] / total_unique_mo_collisions
)

primary_entity_summary["mo_row_share"] = (
    primary_entity_summary["mo_row_count"] / total_mo_rows
)

primary_entity_summary["primary_entity_rank_by_collision_count"] = (
    primary_entity_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

primary_entity_summary["collision_coverage_rate"] = (
    primary_entity_summary["collision_coverage_rate"].round(4)
)

primary_entity_summary["mo_row_share"] = (
    primary_entity_summary["mo_row_share"].round(4)
)

primary_entity_summary = primary_entity_summary.sort_values(
    ["collision_count", "mo_row_count", "primary_entity"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Primary Entity Summary Created")
print("=" * 70)

display(primary_entity_summary)

Primary Entity Summary Created


,primary_entity,collision_count,mo_row_count,collision_coverage_rate,mo_row_share,primary_entity_rank_by_collision_count
0,Collision,505304,3027630,0.9456,0.8598,1
1,Jurisdiction,371557,372037,0.6953,0.1056,2
2,Event,96490,96956,0.1806,0.0275,3
3,Vehicle,9385,9464,0.0176,0.0027,4
4,Evidence,4823,4989,0.0090,0.0014,5
5,Victim,3281,3374,0.0061,0.0010,6
6,Suspect,2890,2950,0.0054,0.0008,7
7,Driver,2334,2334,0.0044,0.0007,8
8,Person,809,814,0.0015,0.0002,9
9,Pedestrian,648,648,0.0012,0.0002,10


In [67]:
# ============================================================
# Step 16D: Export primary entity summary
# ============================================================

file_name = "primary_entity_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    primary_entity_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\primary_entity_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\primary_entity_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\primary_entity_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\primary_entity_summary.csv


## Step 17 — Build Primary Entity by Analytical Domain Summary

### Objective
Create a cross-tab style summary between `primary_entity` and `analytical_domain`.

### Why this table matters
This table helps connect the simplified entity lens with the richer MO domain structure.

It supports questions such as:

- Which analytical domains are mostly collision-centered?
- Which domains are mostly jurisdiction-centered?
- Which entities appear in smaller but meaningful traffic-related domains?
- How does the semantic entity view align with the domain hierarchy?

### Data source
We will use:
`fact_collision_mo_enriched.csv`

because it contains both:
- `primary_entity`
- `analytical_domain`

### Primary metrics
We will compute:

- distinct collision count
- MO row count

for each:
- `primary_entity`
- `analytical_domain`

pair.

### Additional metrics
We will also compute:

- share within primary entity
- share within analytical domain
- pair rank by collision count

### Methodological note
This is a bridge-style summary.
It is designed for interpretation and dashboard drill-down, not as a replacement for the full MO hierarchy.

In [68]:
# ============================================================
# Step 17A: Validate required columns
# ============================================================

required_primary_entity_domain_columns = [
    "dr_number",
    "primary_entity",
    "analytical_domain"
]

missing_primary_entity_domain_columns = [
    col for col in required_primary_entity_domain_columns
    if col not in fact_collision_mo.columns
]

if missing_primary_entity_domain_columns:
    raise ValueError(f"Missing required columns: {missing_primary_entity_domain_columns}")

print("All required primary_entity × analytical_domain columns are available ✔")

All required primary_entity × analytical_domain columns are available ✔


In [69]:
# ============================================================
# Step 17B: Prepare cross-tab base
# ============================================================

primary_entity_domain_base = fact_collision_mo.copy()

for col in ["primary_entity", "analytical_domain"]:
    primary_entity_domain_base[col] = (
        primary_entity_domain_base[col]
        .astype(str)
        .str.strip()
        .replace({
            "": "Unknown / Unclassified",
            "nan": "Unknown / Unclassified",
            "None": "Unknown / Unclassified"
        })
    )

print("=" * 70)
print("Primary entity × analytical domain base prepared")
print("=" * 70)

display(
    primary_entity_domain_base[
        [
            "dr_number",
            "primary_entity",
            "analytical_domain",
            "analytical_category",
            "analytical_subcategory"
        ]
    ].head(15)
)

Primary entity × analytical domain base prepared


,dr_number,primary_entity,analytical_domain,analytical_category,analytical_subcategory
0,212013850,Collision,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing
1,212013850,Collision,Traffic Collision Core,Traffic Collision,Injury severity
2,212013850,Collision,Traffic Collision Core,Traffic Collision,Scene context / property / intersection
3,212013850,Jurisdiction,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division
4,212013850,Collision,Traffic Collision Core,Traffic Collision,Scene context / property / intersection
5,212013850,Collision,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family
6,212013850,Collision,Traffic Collision Type,Collision Pattern,Type of collision
7,212013850,Collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision
8,221417787,Jurisdiction,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division
9,221417787,Collision,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing


In [70]:
# ============================================================
# Step 17C: Build primary_entity × analytical_domain summary
# ============================================================

primary_entity_by_domain_summary = (
    primary_entity_domain_base
    .groupby(["primary_entity", "analytical_domain"], dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

primary_entity_totals = (
    primary_entity_by_domain_summary
    .groupby("primary_entity")["collision_count"]
    .transform("sum")
)

domain_totals = (
    primary_entity_by_domain_summary
    .groupby("analytical_domain")["collision_count"]
    .transform("sum")
)

primary_entity_by_domain_summary["collision_share_within_primary_entity"] = (
    primary_entity_by_domain_summary["collision_count"] / primary_entity_totals
)

primary_entity_by_domain_summary["collision_share_within_domain"] = (
    primary_entity_by_domain_summary["collision_count"] / domain_totals
)

primary_entity_by_domain_summary["pair_rank_by_collision_count"] = (
    primary_entity_by_domain_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

primary_entity_by_domain_summary["collision_share_within_primary_entity"] = (
    primary_entity_by_domain_summary["collision_share_within_primary_entity"].round(4)
)

primary_entity_by_domain_summary["collision_share_within_domain"] = (
    primary_entity_by_domain_summary["collision_share_within_domain"].round(4)
)

primary_entity_by_domain_summary = primary_entity_by_domain_summary.sort_values(
    ["collision_count", "mo_row_count", "primary_entity", "analytical_domain"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

print("=" * 70)
print("Primary Entity × Analytical Domain Summary Created")
print("=" * 70)

display(primary_entity_by_domain_summary.head(30))

Primary Entity × Analytical Domain Summary Created


,primary_entity,analytical_domain,collision_count,mo_row_count,collision_share_within_primary_entity,collision_share_within_domain,pair_rank_by_collision_count
0,Collision,Traffic Collision Core,505112,1801303,0.2921,1.0000,1
1,Collision,Traffic Collision Causation,484688,486978,0.2803,1.0000,2
2,Jurisdiction,Traffic Collision Jurisdiction,371557,372037,1.0000,1.0000,3
3,Collision,Traffic Collision Type,361373,361373,0.2090,1.0000,4
4,Collision,Traffic Collision Pre-Collision Movement,361237,361237,0.2089,1.0000,5
5,Event,Incident Context,83904,83906,0.8655,1.0000,6
6,Collision,Traffic Collision Special Factors,16660,16717,0.0096,1.0000,7
7,Event,Other / Unspecified,12366,12366,0.1276,1.0000,8
8,Vehicle,Vehicle Interaction,9385,9464,1.0000,1.0000,9
9,Evidence,Evidence / Forensics,4823,4989,1.0000,1.0000,10


In [71]:
# ============================================================
# Step 17D: Export primary_entity × analytical_domain summary
# ============================================================

file_name = "primary_entity_by_domain_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    primary_entity_by_domain_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\primary_entity_by_domain_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\primary_entity_by_domain_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\primary_entity_by_domain_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\primary_entity_by_domain_summary.csv


## Step 18 — Build Dashboard-Focused MO Summary (Traffic-Only Filter)

### Objective
Create a dashboard-focused version of the MO layer by filtering the classified MO records
to traffic-relevant analytical domains only.

### Why this step matters
The full MO classification is analytically valuable and should be retained,
but the dashboard overview should emphasize the traffic-collision story.

This step reduces semantic noise by excluding very small non-traffic or peripheral domains
from the main dashboard-oriented summaries.

### Why this is needed
Earlier MO summaries showed that the strongest analytical signal is concentrated in traffic-related domains such as:

- Traffic Collision Core
- Traffic Collision Causation
- Traffic Collision Jurisdiction
- Traffic Collision Type
- Traffic Collision Pre-Collision Movement

These dominate the MO layer and are the most relevant for dashboard interpretation.

### Data source
We will use:
`fact_collision_mo_enriched.csv`

### Filtering logic
We will keep analytical domains that are traffic-focused, including:

- domains starting with `Traffic Collision`
- `Incident Context` when it reflects roadway/traffic incident context

### Outputs
This step will produce:

- `traffic_only_mo_domain_summary.csv`
- `traffic_only_mo_subcategory_summary.csv`

### Methodological note
This is a dashboard-focused derivative layer.
It does not replace the full MO outputs.
Both should be kept:

- full MO outputs for completeness
- traffic-only MO outputs for dashboard clarity

In [72]:
# ============================================================
# Step 18A: Prepare traffic-only MO base
# ============================================================

traffic_only_mo_base = fact_collision_mo.copy()

for col in ["analytical_domain", "analytical_category", "analytical_subcategory"]:
    traffic_only_mo_base[col] = (
        traffic_only_mo_base[col]
        .astype(str)
        .str.strip()
        .replace({
            "": "Unknown / Unclassified",
            "nan": "Unknown / Unclassified",
            "None": "Unknown / Unclassified"
        })
    )

# Keep traffic-focused domains
traffic_only_mo_base = traffic_only_mo_base[
    (
        traffic_only_mo_base["analytical_domain"].str.startswith("Traffic Collision", na=False)
    )
    |
    (
        traffic_only_mo_base["analytical_domain"].eq("Incident Context")
        &
        traffic_only_mo_base["analytical_subcategory"].str.contains("traffic|roadway|road", case=False, na=False)
    )
].copy()

print("=" * 70)
print("Traffic-only MO base created")
print("=" * 70)
print(f"Rows: {len(traffic_only_mo_base):,}")
print(f"Unique collisions: {traffic_only_mo_base['dr_number'].nunique():,}")

display(
    traffic_only_mo_base[
        [
            "dr_number",
            "analytical_domain",
            "analytical_category",
            "analytical_subcategory",
            "primary_entity"
        ]
    ].head(20)
)

Traffic-only MO base created
Rows: 3,486,423
Unique collisions: 525,263


,dr_number,analytical_domain,analytical_category,analytical_subcategory,primary_entity
0,212013850,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing,Collision
1,212013850,Traffic Collision Core,Traffic Collision,Injury severity,Collision
2,212013850,Traffic Collision Core,Traffic Collision,Scene context / property / intersection,Collision
3,212013850,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,Jurisdiction
4,212013850,Traffic Collision Core,Traffic Collision,Scene context / property / intersection,Collision
5,212013850,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Collision
6,212013850,Traffic Collision Type,Collision Pattern,Type of collision,Collision
7,212013850,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Collision
8,221417787,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,Jurisdiction
9,221417787,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing,Collision


In [73]:
# ============================================================
# Step 18B: Build traffic-only MO domain summary
# ============================================================

traffic_only_mo_domain_summary = (
    traffic_only_mo_base
    .groupby("analytical_domain", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

traffic_only_total_collisions = traffic_only_mo_base["dr_number"].nunique()
traffic_only_total_rows = len(traffic_only_mo_base)

traffic_only_mo_domain_summary["collision_coverage_rate"] = (
    traffic_only_mo_domain_summary["collision_count"] / traffic_only_total_collisions
)

traffic_only_mo_domain_summary["mo_row_share"] = (
    traffic_only_mo_domain_summary["mo_row_count"] / traffic_only_total_rows
)

traffic_only_mo_domain_summary["domain_rank_by_collision_count"] = (
    traffic_only_mo_domain_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

traffic_only_mo_domain_summary["collision_coverage_rate"] = (
    traffic_only_mo_domain_summary["collision_coverage_rate"].round(4)
)
traffic_only_mo_domain_summary["mo_row_share"] = (
    traffic_only_mo_domain_summary["mo_row_share"].round(4)
)

traffic_only_mo_domain_summary = traffic_only_mo_domain_summary.sort_values(
    ["collision_count", "mo_row_count", "analytical_domain"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Traffic-only MO Domain Summary Created")
print("=" * 70)

display(traffic_only_mo_domain_summary)

Traffic-only MO Domain Summary Created


,analytical_domain,collision_count,mo_row_count,collision_coverage_rate,mo_row_share,domain_rank_by_collision_count
0,Traffic Collision Core,505112,1801303,0.9616,0.5167,1
1,Traffic Collision Causation,484688,486978,0.9228,0.1397,2
2,Traffic Collision Jurisdiction,371557,372037,0.7074,0.1067,3
3,Traffic Collision Type,361373,361373,0.6880,0.1037,4
4,Traffic Collision Pre-Collision Movement,361237,361237,0.6877,0.1036,5
5,Incident Context,83772,83772,0.1595,0.0240,6
6,Traffic Collision Special Factors,16660,16717,0.0317,0.0048,7
7,Traffic Collision Sobriety,2334,2334,0.0044,0.0007,8
8,Traffic Collision Pedestrian,648,648,0.0012,0.0002,9
9,Traffic Collision Conditions,11,11,0.0000,0.0000,10


In [74]:
# ============================================================
# Step 18C: Build traffic-only MO subcategory summary
# ============================================================

traffic_only_mo_subcategory_summary = (
    traffic_only_mo_base
    .groupby(
        ["analytical_domain", "analytical_category", "analytical_subcategory"],
        dropna=False
    )
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

traffic_only_mo_subcategory_summary["collision_coverage_rate"] = (
    traffic_only_mo_subcategory_summary["collision_count"] / traffic_only_total_collisions
)

traffic_only_mo_subcategory_summary["mo_row_share"] = (
    traffic_only_mo_subcategory_summary["mo_row_count"] / traffic_only_total_rows
)

traffic_only_mo_subcategory_summary["subcategory_rank_by_collision_count"] = (
    traffic_only_mo_subcategory_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

traffic_only_mo_subcategory_summary["collision_coverage_rate"] = (
    traffic_only_mo_subcategory_summary["collision_coverage_rate"].round(4)
)
traffic_only_mo_subcategory_summary["mo_row_share"] = (
    traffic_only_mo_subcategory_summary["mo_row_share"].round(4)
)

category_collision_totals = (
    traffic_only_mo_subcategory_summary
    .groupby(["analytical_domain", "analytical_category"])["collision_count"]
    .transform("sum")
)

category_mo_row_totals = (
    traffic_only_mo_subcategory_summary
    .groupby(["analytical_domain", "analytical_category"])["mo_row_count"]
    .transform("sum")
)

traffic_only_mo_subcategory_summary["subcategory_collision_share_within_category"] = (
    traffic_only_mo_subcategory_summary["collision_count"] / category_collision_totals
)

traffic_only_mo_subcategory_summary["subcategory_mo_row_share_within_category"] = (
    traffic_only_mo_subcategory_summary["mo_row_count"] / category_mo_row_totals
)

traffic_only_mo_subcategory_summary["subcategory_collision_share_within_category"] = (
    traffic_only_mo_subcategory_summary["subcategory_collision_share_within_category"].round(4)
)
traffic_only_mo_subcategory_summary["subcategory_mo_row_share_within_category"] = (
    traffic_only_mo_subcategory_summary["subcategory_mo_row_share_within_category"].round(4)
)

traffic_only_mo_subcategory_summary = traffic_only_mo_subcategory_summary.sort_values(
    [
        "collision_count",
        "mo_row_count",
        "analytical_domain",
        "analytical_category",
        "analytical_subcategory"
    ],
    ascending=[False, False, True, True, True]
).reset_index(drop=True)

print("=" * 70)
print("Traffic-only MO Subcategory Summary Created")
print("=" * 70)

display(traffic_only_mo_subcategory_summary.head(30))

Traffic-only MO Subcategory Summary Created


,analytical_domain,analytical_category,analytical_subcategory,collision_count,mo_row_count,collision_coverage_rate,mo_row_share,subcategory_rank_by_collision_count,subcategory_collision_share_within_category,subcategory_mo_row_share_within_category
0,Traffic Collision Core,Traffic Collision,Collision counterpart / mode pairing,498050,504375,0.9482,0.1447,1,0.3123,0.2800
1,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,484688,486978,0.9228,0.1397,2,1.0000,1.0000
2,Traffic Collision Core,Traffic Collision,Injury severity,419091,419994,0.7979,0.1205,3,0.2628,0.2332
3,Traffic Collision Core,Traffic Collision,Scene context / property / intersection,407841,607186,0.7765,0.1742,4,0.2558,0.3371
4,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,371557,372037,0.7074,0.1067,5,1.0000,1.0000
5,Traffic Collision Type,Collision Pattern,Type of collision,361373,361373,0.6880,0.1037,6,1.0000,1.0000
6,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,361237,361237,0.6877,0.1036,7,1.0000,1.0000
7,Traffic Collision Core,Traffic Collision,Hit-and-run status,258871,258982,0.4928,0.0743,8,0.1623,0.1438
8,Incident Context,Incident Context,Traffic / roadway incident context,83772,83772,0.1595,0.0240,9,1.0000,1.0000
9,Traffic Collision Special Factors,Special Collision Factors,Unlicensed motorist,16033,16033,0.0305,0.0046,10,0.9591,0.9591


In [75]:
# ============================================================
# Step 18D: Export traffic-only MO outputs
# ============================================================

export_files = {
    "traffic_only_mo_domain_summary.csv": traffic_only_mo_domain_summary,
    "traffic_only_mo_subcategory_summary.csv": traffic_only_mo_subcategory_summary,
}

for file_name, df in export_files.items():
    for tool_name, folder_path in tool_export_dirs.items():
        export_path = folder_path / file_name
        df.to_csv(export_path, index=False)
        print(f"Saved {file_name} to {tool_name}: {export_path}")

Saved traffic_only_mo_domain_summary.csv to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\traffic_only_mo_domain_summary.csv
Saved traffic_only_mo_domain_summary.csv to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\traffic_only_mo_domain_summary.csv
Saved traffic_only_mo_domain_summary.csv to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\traffic_only_mo_domain_summary.csv
Saved traffic_only_mo_domain_summary.csv to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\traffic_only_mo_domain_summary.csv
Saved traffic_only_mo_subcategory_summary.csv to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\traffic_only_mo_subcategory_summary.csv
Saved traffic_only_mo_subcategory_summary.csv to sql: C:\Users\MKamal\Deskto

## Step 19 — Build Hit-and-Run Summary from Traffic-Focused MO Layer

### Objective
Create a hit-and-run summary from the traffic-focused MO layer.

### Why this table matters
This table is highly relevant for dashboarding because hit-and-run is a strong public-safety and policy-relevant topic.

It supports:

- hit-and-run monitoring
- public safety storytelling
- comparison of hit-and-run status categories
- dashboard KPI cards and ranking visuals

### Data source
We will use:
`traffic_only_mo_base`

because it already contains only traffic-focused MO records and preserves the structured MO hierarchy.

### Analytical design
We will isolate records where:

- `analytical_subcategory == "Hit-and-run status"`

Then we will summarize the detailed MO description values that represent hit-and-run statuses.

### Primary metrics
We will compute:

- distinct collision count
- MO row count

for each hit-and-run status label.

### Additional metrics
We will also compute:

- share within hit-and-run status records
- coverage relative to all traffic-only collisions
- status rank by collision count

### Methodological note
This summary remains collision-based for counts and MO-row-based for intensity.
Both measures should be interpreted together.

In [76]:
# ============================================================
# Step 19A: Validate hit-and-run availability in traffic-only MO layer
# ============================================================

required_hit_run_columns = [
    "dr_number",
    "mo_description",
    "analytical_subcategory"
]

missing_hit_run_columns = [
    col for col in required_hit_run_columns
    if col not in traffic_only_mo_base.columns
]

if missing_hit_run_columns:
    raise ValueError(f"Missing required hit-and-run columns: {missing_hit_run_columns}")

hit_run_preview = traffic_only_mo_base.loc[
    traffic_only_mo_base["analytical_subcategory"] == "Hit-and-run status",
    ["mo_description"]
].drop_duplicates().sort_values("mo_description").reset_index(drop=True)

if hit_run_preview.empty:
    raise ValueError("No hit-and-run status rows were found in traffic_only_mo_base.")

print("=" * 70)
print("Available hit-and-run MO descriptions")
print("=" * 70)
display(hit_run_preview)

Available hit-and-run MO descriptions


,mo_description
0,T/C – Hit and run (felony)
1,T/C – Hit and run (misdemeanor)


In [77]:
# ============================================================
# Step 19B: Prepare hit-and-run base
# ============================================================

hit_run_base = traffic_only_mo_base.loc[
    traffic_only_mo_base["analytical_subcategory"] == "Hit-and-run status",
    ["dr_number", "mo_description", "analytical_domain", "analytical_category", "analytical_subcategory"]
].copy()

# Clean the display label slightly while preserving the original semantic meaning
hit_run_base["hit_run_status_label"] = (
    hit_run_base["mo_description"]
    .astype(str)
    .str.strip()
    .str.replace(r"^T/C\s*[–-]\s*", "", regex=True)
)

print("=" * 70)
print("Hit-and-run base prepared")
print("=" * 70)

display(hit_run_base.head(20))

Hit-and-run base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory,hit_run_status_label
41,190319680,T/C – Hit and run (misdemeanor),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (misdemeanor)
49,190413769,T/C – Hit and run (misdemeanor),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (misdemeanor)
67,190411883,T/C – Hit and run (felony),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (felony)
72,190514552,T/C – Hit and run (felony),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (felony)
87,190413937,T/C – Hit and run (misdemeanor),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (misdemeanor)
98,190413950,T/C – Hit and run (misdemeanor),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (misdemeanor)
103,190413974,T/C – Hit and run (misdemeanor),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (misdemeanor)
109,190413978,T/C – Hit and run (misdemeanor),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (misdemeanor)
154,190514773,T/C – Hit and run (misdemeanor),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (misdemeanor)
158,190601104,T/C – Hit and run (felony),Traffic Collision Core,Traffic Collision,Hit-and-run status,Hit and run (felony)


In [78]:
# ============================================================
# Step 19C: Build hit-and-run summary
# ============================================================

hit_run_summary = (
    hit_run_base
    .groupby("hit_run_status_label", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_hit_run_collisions = hit_run_summary["collision_count"].sum()
traffic_only_total_collisions = traffic_only_mo_base["dr_number"].nunique()

hit_run_summary["collision_pct_within_hit_run_statuses"] = (
    hit_run_summary["collision_count"] / total_hit_run_collisions
)

hit_run_summary["collision_pct_of_traffic_only_collisions"] = (
    hit_run_summary["collision_count"] / traffic_only_total_collisions
)

hit_run_summary["status_rank_by_collision_count"] = (
    hit_run_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

hit_run_summary["collision_pct_within_hit_run_statuses"] = (
    hit_run_summary["collision_pct_within_hit_run_statuses"].round(4)
)

hit_run_summary["collision_pct_of_traffic_only_collisions"] = (
    hit_run_summary["collision_pct_of_traffic_only_collisions"].round(4)
)

hit_run_summary = hit_run_summary.sort_values(
    ["collision_count", "mo_row_count", "hit_run_status_label"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Hit-and-Run Summary Created")
print("=" * 70)

display(hit_run_summary)

Hit-and-Run Summary Created


,hit_run_status_label,collision_count,mo_row_count,collision_pct_within_hit_run_statuses,collision_pct_of_traffic_only_collisions,status_rank_by_collision_count
0,Hit and run (misdemeanor),204190,204190,0.7884,0.3887,1
1,Hit and run (felony),54792,54792,0.2116,0.1043,2


In [79]:
# ============================================================
# Step 19D: Export hit-and-run summary
# ============================================================

file_name = "hit_run_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    hit_run_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\hit_run_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\hit_run_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\hit_run_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\hit_run_summary.csv


## Step 20 — Build DUI / Sobriety Summary from Traffic-Focused MO Layer

### Objective
Create a structured summary for DUI- and sobriety-related traffic collision records.

### Why this table matters
This table supports:

- impaired-driving monitoring
- safety dashboard KPIs
- comparison between DUI-related and sobriety-related classifications
- focused storytelling around alcohol / impairment signals in traffic collisions

### Data source
We will use:
`traffic_only_mo_base`

because it already contains traffic-focused classified MO records.

### Analytical design
We will isolate rows related to:

- `DUI status`
- `Driver sobriety`

Then we will summarize the detailed MO description values inside this focused subset.

### Primary metrics
We will compute:

- distinct collision count
- MO row count

### Additional metrics
We will also compute:

- share within the DUI / sobriety subset
- coverage relative to all traffic-only collisions
- rank by collision count

### Methodological note
This summary is derived from the structured MO layer rather than raw text parsing from the collision table,
which makes the logic more reproducible and easier to explain.

In [80]:
# ============================================================
# Step 20A: Validate DUI / sobriety availability
# ============================================================

required_dui_columns = [
    "dr_number",
    "mo_description",
    "analytical_subcategory"
]

missing_dui_columns = [
    col for col in required_dui_columns
    if col not in traffic_only_mo_base.columns
]

if missing_dui_columns:
    raise ValueError(f"Missing required DUI / sobriety columns: {missing_dui_columns}")

dui_sobriety_preview = traffic_only_mo_base.loc[
    traffic_only_mo_base["analytical_subcategory"].isin(["DUI status", "Driver sobriety"]),
    ["analytical_subcategory", "mo_description"]
].drop_duplicates().sort_values(
    ["analytical_subcategory", "mo_description"]
).reset_index(drop=True)

if dui_sobriety_preview.empty:
    raise ValueError("No DUI / sobriety rows were found in traffic_only_mo_base.")

print("=" * 70)
print("Available DUI / sobriety MO descriptions")
print("=" * 70)
display(dui_sobriety_preview)

Available DUI / sobriety MO descriptions


,analytical_subcategory,mo_description
0,DUI status,T/C – DUI felony
1,DUI status,T/C – DUI misdemeanor
2,Driver sobriety,T/C – Sobriety


In [81]:
# ============================================================
# Step 20B: Prepare DUI / sobriety base
# ============================================================

dui_sobriety_base = traffic_only_mo_base.loc[
    traffic_only_mo_base["analytical_subcategory"].isin(["DUI status", "Driver sobriety"]),
    ["dr_number", "mo_description", "analytical_domain", "analytical_category", "analytical_subcategory"]
].copy()

dui_sobriety_base["dui_sobriety_group"] = np.where(
    dui_sobriety_base["analytical_subcategory"] == "DUI status",
    "DUI-related",
    "Sobriety-related"
)

dui_sobriety_base["dui_sobriety_status_label"] = (
    dui_sobriety_base["mo_description"]
    .astype(str)
    .str.strip()
    .str.replace(r"^T/C\s*[–-]\s*", "", regex=True)
)

print("=" * 70)
print("DUI / sobriety base prepared")
print("=" * 70)

display(dui_sobriety_base.head(20))

DUI / sobriety base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory,dui_sobriety_group,dui_sobriety_status_label
42,190319680,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
131,190514716,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
509,230118219,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
1267,230216021,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
1283,230216050,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
1522,191013717,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
1542,190915814,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
1719,230411303,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
1776,221008278,T/C – DUI misdemeanor,Traffic Collision Core,Traffic Collision,DUI status,DUI-related,DUI misdemeanor
1794,191115715,T/C – Sobriety,Traffic Collision Sobriety,Sobriety / Impairment,Driver sobriety,Sobriety-related,Sobriety


In [82]:
# ============================================================
# Step 20C: Build DUI / sobriety summary
# ============================================================

dui_sobriety_summary = (
    dui_sobriety_base
    .groupby(["dui_sobriety_group", "dui_sobriety_status_label"], dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_dui_sobriety_collisions = dui_sobriety_summary["collision_count"].sum()
traffic_only_total_collisions = traffic_only_mo_base["dr_number"].nunique()

dui_sobriety_summary["collision_pct_within_dui_sobriety"] = (
    dui_sobriety_summary["collision_count"] / total_dui_sobriety_collisions
)

dui_sobriety_summary["collision_pct_of_traffic_only_collisions"] = (
    dui_sobriety_summary["collision_count"] / traffic_only_total_collisions
)

dui_sobriety_summary["status_rank_by_collision_count"] = (
    dui_sobriety_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

dui_sobriety_summary["collision_pct_within_dui_sobriety"] = (
    dui_sobriety_summary["collision_pct_within_dui_sobriety"].round(4)
)

dui_sobriety_summary["collision_pct_of_traffic_only_collisions"] = (
    dui_sobriety_summary["collision_pct_of_traffic_only_collisions"].round(4)
)

dui_sobriety_summary = dui_sobriety_summary.sort_values(
    ["collision_count", "mo_row_count", "dui_sobriety_group", "dui_sobriety_status_label"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

print("=" * 70)
print("DUI / Sobriety Summary Created")
print("=" * 70)

display(dui_sobriety_summary)

DUI / Sobriety Summary Created


,dui_sobriety_group,dui_sobriety_status_label,collision_count,mo_row_count,collision_pct_within_dui_sobriety,collision_pct_of_traffic_only_collisions,status_rank_by_collision_count
0,DUI-related,DUI misdemeanor,7433,7433,0.5744,0.0142,1
1,DUI-related,DUI felony,3174,3174,0.2453,0.0060,2
2,Sobriety-related,Sobriety,2334,2334,0.1804,0.0044,3


In [83]:
# ============================================================
# Step 20D: Optional group-level rollup
# ============================================================

dui_sobriety_group_summary = (
    dui_sobriety_base
    .groupby("dui_sobriety_group", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

dui_sobriety_group_summary["collision_pct_within_dui_sobriety"] = (
    dui_sobriety_group_summary["collision_count"] /
    dui_sobriety_group_summary["collision_count"].sum()
)

dui_sobriety_group_summary["collision_pct_of_traffic_only_collisions"] = (
    dui_sobriety_group_summary["collision_count"] / traffic_only_total_collisions
)

dui_sobriety_group_summary["group_rank_by_collision_count"] = (
    dui_sobriety_group_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

dui_sobriety_group_summary["collision_pct_within_dui_sobriety"] = (
    dui_sobriety_group_summary["collision_pct_within_dui_sobriety"].round(4)
)

dui_sobriety_group_summary["collision_pct_of_traffic_only_collisions"] = (
    dui_sobriety_group_summary["collision_pct_of_traffic_only_collisions"].round(4)
)

display(dui_sobriety_group_summary)

,dui_sobriety_group,collision_count,mo_row_count,collision_pct_within_dui_sobriety,collision_pct_of_traffic_only_collisions,group_rank_by_collision_count
0,DUI-related,10596,10607,0.8195,0.0202,1
1,Sobriety-related,2334,2334,0.1805,0.0044,2


In [84]:
# ============================================================
# Step 20E: Export DUI / sobriety outputs
# ============================================================

export_files = {
    "dui_sobriety_summary.csv": dui_sobriety_summary,
    "dui_sobriety_group_summary.csv": dui_sobriety_group_summary,
}

for file_name, df in export_files.items():
    for tool_name, folder_path in tool_export_dirs.items():
        export_path = folder_path / file_name
        df.to_csv(export_path, index=False)
        print(f"Saved {file_name} to {tool_name}: {export_path}")

Saved dui_sobriety_summary.csv to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\dui_sobriety_summary.csv
Saved dui_sobriety_summary.csv to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\dui_sobriety_summary.csv
Saved dui_sobriety_summary.csv to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\dui_sobriety_summary.csv
Saved dui_sobriety_summary.csv to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\dui_sobriety_summary.csv
Saved dui_sobriety_group_summary.csv to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\dui_sobriety_group_summary.csv
Saved dui_sobriety_group_summary.csv to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\dui_sobriety_group_summary.csv
Save

## Step 21 — Build Injury Severity Summary from Traffic-Focused MO Layer

### Objective
Create an injury-severity summary from the traffic-focused MO layer.

### Why this table matters
This table supports:

- severity-focused dashboard views
- safety KPI cards
- comparison between fatal, visible-injury, complaint-of-pain, and non-injury collisions
- injury-severity storytelling in the traffic collision dashboard

### Data source
We will use:
`traffic_only_mo_base`

because the structured MO layer already contains classified injury-severity records.

### Analytical design
We will isolate rows where:

- `analytical_subcategory == "Injury severity"`

Then we will summarize the detailed MO description values that represent severity classes.

### Primary metrics
We will compute:

- distinct collision count
- MO row count

for each injury severity label.

### Additional metrics
We will also compute:

- share within injury-severity statuses
- coverage relative to all traffic-only collisions
- severity rank by collision count

### Methodological note
This summary is built from the structured MO layer rather than raw code parsing,
which improves reproducibility and dashboard explainability.

In [85]:
# ============================================================
# Step 21A: Validate injury severity availability
# ============================================================

required_injury_columns = [
    "dr_number",
    "mo_description",
    "analytical_subcategory"
]

missing_injury_columns = [
    col for col in required_injury_columns
    if col not in traffic_only_mo_base.columns
]

if missing_injury_columns:
    raise ValueError(f"Missing required injury severity columns: {missing_injury_columns}")

injury_severity_preview = traffic_only_mo_base.loc[
    traffic_only_mo_base["analytical_subcategory"] == "Injury severity",
    ["mo_description"]
].drop_duplicates().sort_values("mo_description").reset_index(drop=True)

if injury_severity_preview.empty:
    raise ValueError("No injury severity rows were found in traffic_only_mo_base.")

print("=" * 70)
print("Available injury severity MO descriptions")
print("=" * 70)
display(injury_severity_preview)

Available injury severity MO descriptions


,mo_description
0,T/C – (A) Severe injury
1,T/C – (B) Visible injury
2,T/C – (C) Complaint of injury
3,T/C – (K) Fatal injury
4,T/C – (N) Non-injury


In [86]:
# ============================================================
# Step 21B: Prepare injury severity base
# ============================================================

injury_severity_base = traffic_only_mo_base.loc[
    traffic_only_mo_base["analytical_subcategory"] == "Injury severity",
    ["dr_number", "mo_description", "analytical_domain", "analytical_category", "analytical_subcategory"]
].copy()

injury_severity_base["injury_severity_label"] = (
    injury_severity_base["mo_description"]
    .astype(str)
    .str.strip()
    .str.replace(r"^T/C\s*[–-]\s*", "", regex=True)
)

def assign_injury_severity_order(label: str) -> int:
    label = str(label).lower()

    if "(k)" in label or "fatal" in label:
        return 1
    elif "(a)" in label or "severe" in label:
        return 2
    elif "(b)" in label or "visible injury" in label:
        return 3
    elif "(c)" in label or "complaint of pain" in label:
        return 4
    elif "(n)" in label or "non-injury" in label:
        return 5
    else:
        return 99

injury_severity_base["injury_severity_order"] = (
    injury_severity_base["injury_severity_label"]
    .apply(assign_injury_severity_order)
)

print("=" * 70)
print("Injury severity base prepared")
print("=" * 70)

display(injury_severity_base.head(20))

Injury severity base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory,injury_severity_label,injury_severity_order
1,212013850,T/C – (K) Fatal injury,Traffic Collision Core,Traffic Collision,Injury severity,(K) Fatal injury,1
10,221417787,T/C – (N) Non-injury,Traffic Collision Core,Traffic Collision,Injury severity,(N) Non-injury,5
18,221418141,T/C – (B) Visible injury,Traffic Collision Core,Traffic Collision,Injury severity,(B) Visible injury,3
26,222017859,T/C – (C) Complaint of injury,Traffic Collision Core,Traffic Collision,Injury severity,(C) Complaint of injury,4
35,190319651,T/C – (C) Complaint of injury,Traffic Collision Core,Traffic Collision,Injury severity,(C) Complaint of injury,4
40,190319680,T/C – (N) Non-injury,Traffic Collision Core,Traffic Collision,Injury severity,(N) Non-injury,5
60,190319695,T/C – (B) Visible injury,Traffic Collision Core,Traffic Collision,Injury severity,(B) Visible injury,3
66,190411883,T/C – (B) Visible injury,Traffic Collision Core,Traffic Collision,Injury severity,(B) Visible injury,3
71,190514552,T/C – (C) Complaint of injury,Traffic Collision Core,Traffic Collision,Injury severity,(C) Complaint of injury,4
77,190319665,T/C – (N) Non-injury,Traffic Collision Core,Traffic Collision,Injury severity,(N) Non-injury,5


In [87]:
# ============================================================
# Step 21C: Build injury severity summary
# ============================================================

injury_severity_summary = (
    injury_severity_base
    .groupby(["injury_severity_order", "injury_severity_label"], dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_injury_severity_collisions = injury_severity_summary["collision_count"].sum()
traffic_only_total_collisions = traffic_only_mo_base["dr_number"].nunique()

injury_severity_summary["collision_pct_within_injury_severity"] = (
    injury_severity_summary["collision_count"] / total_injury_severity_collisions
)

injury_severity_summary["collision_pct_of_traffic_only_collisions"] = (
    injury_severity_summary["collision_count"] / traffic_only_total_collisions
)

injury_severity_summary["severity_rank_by_collision_count"] = (
    injury_severity_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

injury_severity_summary["collision_pct_within_injury_severity"] = (
    injury_severity_summary["collision_pct_within_injury_severity"].round(4)
)

injury_severity_summary["collision_pct_of_traffic_only_collisions"] = (
    injury_severity_summary["collision_pct_of_traffic_only_collisions"].round(4)
)

injury_severity_summary = injury_severity_summary.sort_values(
    ["injury_severity_order", "collision_count", "injury_severity_label"],
    ascending=[True, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Injury Severity Summary Created")
print("=" * 70)

display(injury_severity_summary)

Injury Severity Summary Created


,injury_severity_order,injury_severity_label,collision_count,mo_row_count,collision_pct_within_injury_severity,collision_pct_of_traffic_only_collisions,severity_rank_by_collision_count
0,1,(K) Fatal injury,3348,3348,0.0080,0.0064,5
1,2,(A) Severe injury,16376,16376,0.0390,0.0312,4
2,3,(B) Visible injury,67523,67523,0.1608,0.1286,3
3,4,(C) Complaint of injury,148649,148649,0.3539,0.2830,2
4,5,(N) Non-injury,184098,184098,0.4383,0.3505,1


In [88]:
# ============================================================
# Step 21D: Export injury severity summary
# ============================================================

file_name = "injury_severity_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    injury_severity_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\injury_severity_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\injury_severity_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\injury_severity_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\injury_severity_summary.csv


## Step 22 — Build Collision Type Summary from Traffic-Focused MO Layer

### Objective
Create a collision-type summary from the traffic-focused MO layer.

### Why this table matters
This table supports:

- collision pattern analysis
- dashboard ranking visuals for collision types
- comparison between major crash configurations
- clearer storytelling around how collisions happen

### Data source
We will use:
`traffic_only_mo_base`

because it already contains structured traffic-focused MO classifications.

### Analytical design
We will isolate rows where:

- `analytical_domain == "Traffic Collision Type"`
or
- `analytical_subcategory == "Type of collision"`

Then we will summarize the detailed MO description values that represent collision-type labels.

### Primary metrics
We will compute:

- distinct collision count
- MO row count

for each collision-type label.

### Additional metrics
We will also compute:

- share within collision-type statuses
- coverage relative to all traffic-only collisions
- type rank by collision count

### Methodological note
This summary is derived from the structured MO layer rather than raw string parsing,
which improves reproducibility and dashboard explainability.

In [89]:
# ============================================================
# Step 22A: Validate collision type availability
# ============================================================

required_collision_type_columns = [
    "dr_number",
    "mo_description",
    "analytical_domain",
    "analytical_subcategory"
]

missing_collision_type_columns = [
    col for col in required_collision_type_columns
    if col not in traffic_only_mo_base.columns
]

if missing_collision_type_columns:
    raise ValueError(f"Missing required collision type columns: {missing_collision_type_columns}")

collision_type_preview = traffic_only_mo_base.loc[
    (
        traffic_only_mo_base["analytical_domain"] == "Traffic Collision Type"
    ) |
    (
        traffic_only_mo_base["analytical_subcategory"] == "Type of collision"
    ),
    ["mo_description"]
].drop_duplicates().sort_values("mo_description").reset_index(drop=True)

if collision_type_preview.empty:
    raise ValueError("No collision type rows were found in traffic_only_mo_base.")

print("=" * 70)
print("Available collision type MO descriptions")
print("=" * 70)
display(collision_type_preview)

Available collision type MO descriptions


,mo_description
0,T/C – Type of collision


In [90]:
# ============================================================
# Step 22B: Prepare collision type base
# ============================================================

collision_type_base = traffic_only_mo_base.loc[
    (
        traffic_only_mo_base["analytical_domain"] == "Traffic Collision Type"
    ) |
    (
        traffic_only_mo_base["analytical_subcategory"] == "Type of collision"
    ),
    ["dr_number", "mo_description", "analytical_domain", "analytical_category", "analytical_subcategory"]
].copy()

collision_type_base["collision_type_label"] = (
    collision_type_base["mo_description"]
    .astype(str)
    .str.strip()
    .str.replace(r"^T/C\s*[–-]\s*", "", regex=True)
)

print("=" * 70)
print("Collision type base prepared")
print("=" * 70)

display(collision_type_base.head(20))

Collision type base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory,collision_type_label
6,212013850,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
14,221417787,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
22,221418141,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
30,222017859,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
46,190413769,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
52,190127578,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
63,190411883,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
84,190413937,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
89,190413949,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision
95,190413950,T/C – Type of collision,Traffic Collision Type,Collision Pattern,Type of collision,Type of collision


In [91]:
# ============================================================
# Step 22C: Build collision type summary
# ============================================================

collision_type_summary = (
    collision_type_base
    .groupby("collision_type_label", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_collision_type_collisions = collision_type_summary["collision_count"].sum()
traffic_only_total_collisions = traffic_only_mo_base["dr_number"].nunique()

collision_type_summary["collision_pct_within_collision_types"] = (
    collision_type_summary["collision_count"] / total_collision_type_collisions
)

collision_type_summary["collision_pct_of_traffic_only_collisions"] = (
    collision_type_summary["collision_count"] / traffic_only_total_collisions
)

collision_type_summary["collision_type_rank_by_collision_count"] = (
    collision_type_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

collision_type_summary["collision_pct_within_collision_types"] = (
    collision_type_summary["collision_pct_within_collision_types"].round(4)
)

collision_type_summary["collision_pct_of_traffic_only_collisions"] = (
    collision_type_summary["collision_pct_of_traffic_only_collisions"].round(4)
)

collision_type_summary = collision_type_summary.sort_values(
    ["collision_count", "mo_row_count", "collision_type_label"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Collision Type Summary Created")
print("=" * 70)

display(collision_type_summary)

Collision Type Summary Created


,collision_type_label,collision_count,mo_row_count,collision_pct_within_collision_types,collision_pct_of_traffic_only_collisions,collision_type_rank_by_collision_count
0,Type of collision,361373,361373,1.0000,0.6880,1


In [92]:
# ============================================================
# Step 22D: Export collision type summary
# ============================================================

file_name = "collision_type_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    collision_type_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\collision_type_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\collision_type_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\collision_type_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\collision_type_summary.csv


## Step 22E — MO Output QA Gate and Output Registry

### Objective
Create a lightweight quality-control layer for MO-derived outputs before using them in dashboard visuals.

### Why this step matters
Not every valid exported table is equally useful for dashboarding.

Some summaries are:
- dashboard-ready
- useful only for audit / structural review
- limited in interpretability because they collapse into one generic label

### What this QA step checks
For each selected MO summary table, we will evaluate:

- number of unique labels
- dominant-label concentration
- whether the output is suitable for dashboard visuals
- a short interpretability note

### Why this is important
This helps us distinguish between:

- valid but low-information outputs
- valid and dashboard-strong outputs

### Example from current project
The collision type summary is structurally valid,
but it collapses into one generic label and therefore is not suitable as a primary dashboard visual.

By contrast, summaries such as:
- hit-and-run
- DUI / sobriety
- injury severity

contain interpretable labels and are much more dashboard-ready.

### Deliverable
This step will create an `mo_output_registry.csv` file that documents the usability status of selected MO outputs.

In [93]:
# ============================================================
# Step 22E-A: Build MO output QA assessment function
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

def assess_mo_output(
    df: pd.DataFrame,
    summary_name: str,
    label_col: str,
    source_table: str = "traffic_only_mo_base",
    dominance_threshold: float = 0.90
) -> dict:
    """
    Assess whether an MO-derived summary is dashboard-ready or mainly audit-oriented.

    Rules:
    - If there is only 1 unique label -> audit_only
    - If one label dominates >= dominance_threshold -> limited_interpretability
    - Otherwise -> dashboard_ready
    """
    if df.empty:
        return {
            "summary_name": summary_name,
            "label_column": label_col,
            "source_table": source_table,
            "row_count": 0,
            "unique_labels": 0,
            "top_label": "N/A",
            "top_label_collision_count": 0,
            "top_label_share": np.nan,
            "output_status": "empty_output",
            "interpretation_note": "No rows available."
        }

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found in {summary_name}")

    if "collision_count" not in df.columns:
        raise ValueError(f"'collision_count' column is required in {summary_name}")

    temp = df.copy().sort_values("collision_count", ascending=False).reset_index(drop=True)

    unique_labels = temp[label_col].nunique(dropna=False)
    total_collision_count = temp["collision_count"].sum()
    top_label = temp.loc[0, label_col]
    top_label_collision_count = temp.loc[0, "collision_count"]
    top_label_share = (
        top_label_collision_count / total_collision_count
        if total_collision_count > 0 else np.nan
    )

    if unique_labels == 1:
        status = "audit_only"
        note = "Valid structure, but collapsed into one generic label; not suitable as a primary dashboard visual."
    elif pd.notna(top_label_share) and top_label_share >= dominance_threshold:
        status = "limited_interpretability"
        note = "Valid output, but heavily dominated by one label; usable with caution."
    else:
        status = "dashboard_ready"
        note = "Multiple interpretable labels with sufficient variation for dashboard use."

    return {
        "summary_name": summary_name,
        "label_column": label_col,
        "source_table": source_table,
        "row_count": len(temp),
        "unique_labels": unique_labels,
        "top_label": top_label,
        "top_label_collision_count": int(top_label_collision_count),
        "top_label_share": round(top_label_share, 4) if pd.notna(top_label_share) else np.nan,
        "output_status": status,
        "interpretation_note": note
    }

In [94]:
# ============================================================
# Step 22E-B: Assess current MO dashboard-oriented outputs
# ============================================================

mo_output_assessments = [
    assess_mo_output(
        df=hit_run_summary,
        summary_name="hit_run_summary",
        label_col="hit_run_status_label"
    ),
    assess_mo_output(
        df=dui_sobriety_summary,
        summary_name="dui_sobriety_summary",
        label_col="dui_sobriety_status_label"
    ),
    assess_mo_output(
        df=injury_severity_summary,
        summary_name="injury_severity_summary",
        label_col="injury_severity_label"
    ),
    assess_mo_output(
        df=collision_type_summary,
        summary_name="collision_type_summary",
        label_col="collision_type_label"
    ),
]

mo_output_registry = pd.DataFrame(mo_output_assessments)

print("=" * 70)
print("MO Output QA Registry")
print("=" * 70)
display(mo_output_registry)

MO Output QA Registry


,summary_name,label_column,source_table,row_count,unique_labels,top_label,top_label_collision_count,top_label_share,output_status,interpretation_note
0,hit_run_summary,hit_run_status_label,traffic_only_mo_base,2,2,Hit and run (misdemeanor),204190,0.7884,dashboard_ready,Multiple interpretable labels with sufficient ...
1,dui_sobriety_summary,dui_sobriety_status_label,traffic_only_mo_base,3,3,DUI misdemeanor,7433,0.5744,dashboard_ready,Multiple interpretable labels with sufficient ...
2,injury_severity_summary,injury_severity_label,traffic_only_mo_base,5,5,(N) Non-injury,184098,0.4383,dashboard_ready,Multiple interpretable labels with sufficient ...
3,collision_type_summary,collision_type_label,traffic_only_mo_base,1,1,Type of collision,361373,1.0000,audit_only,"Valid structure, but collapsed into one generi..."


In [95]:
# ============================================================
# Step 22E-C: Define and save MO output registry
# ============================================================

FINAL_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables" / "final"
FINAL_TABLES_DIR.mkdir(parents=True, exist_ok=True)

mo_output_registry_path = FINAL_TABLES_DIR / "mo_output_registry.csv"

mo_output_registry.to_csv(mo_output_registry_path, index=False)

print("=" * 70)
print("MO output registry saved")
print("=" * 70)
print(f"Saved to: {mo_output_registry_path}")

MO output registry saved
Saved to: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables\final\mo_output_registry.csv


## Step 23 — Build Reporting Division Summary from Traffic-Focused MO Layer

### Objective
Create a reporting-division summary from the traffic-focused MO layer.

### Why this table matters
This table supports:

- geographic / jurisdiction dashboard views
- division-level ranking visuals
- reporting-area comparison
- operational storytelling around which traffic divisions appear most frequently

### Data source
We will use:
`traffic_only_mo_base`

because it contains the structured jurisdiction layer inside the classified MO data.

### Analytical design
We will isolate rows where:

- `analytical_domain == "Traffic Collision Jurisdiction"`
or
- `analytical_subcategory == "LAPD division / traffic division"`

Then we will summarize the detailed MO description values that represent reporting divisions.

### Primary metrics
We will compute:

- distinct collision count
- MO row count

for each reporting division label.

### Additional metrics
We will also compute:

- share within reporting-division records
- coverage relative to all traffic-only collisions
- rank by collision count

### Methodological note
This summary is derived from the structured MO layer,
which makes the reporting-division logic reproducible and dashboard-ready.

In [96]:
# ============================================================
# Step 23A: Validate reporting division availability
# ============================================================

required_reporting_division_columns = [
    "dr_number",
    "mo_description",
    "analytical_domain",
    "analytical_subcategory"
]

missing_reporting_division_columns = [
    col for col in required_reporting_division_columns
    if col not in traffic_only_mo_base.columns
]

if missing_reporting_division_columns:
    raise ValueError(f"Missing required reporting division columns: {missing_reporting_division_columns}")

reporting_division_preview = traffic_only_mo_base.loc[
    (
        traffic_only_mo_base["analytical_domain"] == "Traffic Collision Jurisdiction"
    ) |
    (
        traffic_only_mo_base["analytical_subcategory"] == "LAPD division / traffic division"
    ),
    ["mo_description"]
].drop_duplicates().sort_values("mo_description").reset_index(drop=True)

if reporting_division_preview.empty:
    raise ValueError("No reporting division rows were found in traffic_only_mo_base.")

print("=" * 70)
print("Available reporting division MO descriptions")
print("=" * 70)
display(reporting_division_preview)

Available reporting division MO descriptions


,mo_description
0,T/C – 77th Street Division
1,T/C – Central Division
2,T/C – Central Traffic Division (CTD)
3,T/C – Devonshire Division
4,T/C – Foothill Division
5,T/C – Harbor Division
6,T/C – Hollenbeck Division
7,T/C – Hollywood Division
8,T/C – Mission Division
9,T/C – Newton Division


In [97]:
# ============================================================
# Step 23B: Prepare reporting division base
# ============================================================

reporting_division_base = traffic_only_mo_base.loc[
    (
        traffic_only_mo_base["analytical_domain"] == "Traffic Collision Jurisdiction"
    ) |
    (
        traffic_only_mo_base["analytical_subcategory"] == "LAPD division / traffic division"
    ),
    ["dr_number", "mo_description", "analytical_domain", "analytical_category", "analytical_subcategory"]
].copy()

reporting_division_base["reporting_division_label"] = (
    reporting_division_base["mo_description"]
    .astype(str)
    .str.strip()
    .str.replace(r"^T/C\s*[–-]\s*", "", regex=True)
)

print("=" * 70)
print("Reporting division base prepared")
print("=" * 70)

display(reporting_division_base.head(20))

Reporting division base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory,reporting_division_label
3,212013850,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,West Traffic Division (WTD)
8,221417787,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,West Traffic Division (WTD)
16,221418141,T/C – West Traffic Division (WTD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,West Traffic Division (WTD)
32,222017859,T/C – Olympic Division,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,Olympic Division
37,190319651,T/C – Southwest Division,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,Southwest Division
44,190319680,T/C – Southwest Division,Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,Southwest Division
57,190319695,T/C – South Traffic Division (STD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,South Traffic Division (STD)
74,190319665,T/C – South Traffic Division (STD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,South Traffic Division (STD)
79,190319702,T/C – South Traffic Division (STD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,South Traffic Division (STD)
110,190514653,T/C – South Traffic Division (STD),Traffic Collision Jurisdiction,Reporting Division,LAPD division / traffic division,South Traffic Division (STD)


In [98]:
# ============================================================
# Step 23C: Build reporting division summary
# ============================================================

reporting_division_summary = (
    reporting_division_base
    .groupby("reporting_division_label", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_reporting_division_collisions = reporting_division_summary["collision_count"].sum()
traffic_only_total_collisions = traffic_only_mo_base["dr_number"].nunique()

reporting_division_summary["collision_pct_within_reporting_divisions"] = (
    reporting_division_summary["collision_count"] / total_reporting_division_collisions
)

reporting_division_summary["collision_pct_of_traffic_only_collisions"] = (
    reporting_division_summary["collision_count"] / traffic_only_total_collisions
)

reporting_division_summary["reporting_division_rank_by_collision_count"] = (
    reporting_division_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

reporting_division_summary["collision_pct_within_reporting_divisions"] = (
    reporting_division_summary["collision_pct_within_reporting_divisions"].round(4)
)

reporting_division_summary["collision_pct_of_traffic_only_collisions"] = (
    reporting_division_summary["collision_pct_of_traffic_only_collisions"].round(4)
)

reporting_division_summary = reporting_division_summary.sort_values(
    ["collision_count", "mo_row_count", "reporting_division_label"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Reporting Division Summary Created")
print("=" * 70)

display(reporting_division_summary)

Reporting Division Summary Created


,reporting_division_label,collision_count,mo_row_count,collision_pct_within_reporting_divisions,collision_pct_of_traffic_only_collisions,reporting_division_rank_by_collision_count
0,Valley Traffic Division (VTD),94376,94376,0.2537,0.1797,1
1,West Traffic Division (WTD),69973,69973,0.1881,0.1332,2
2,South Traffic Division (STD),44635,44635,0.1200,0.0850,3
3,Central Traffic Division (CTD),15124,15124,0.0407,0.0288,4
4,Olympic Division,11407,11407,0.0307,0.0217,5
5,77th Street Division,11113,11113,0.0299,0.0212,6
6,Pacific Division,10824,10824,0.0291,0.0206,7
7,West Los Angeles Division,10816,10816,0.0291,0.0206,8
8,Hollywood Division,10807,10807,0.0290,0.0206,9
9,North Hollywood Division,9965,9965,0.0268,0.0190,10


In [99]:
# ============================================================
# Step 23D: Export reporting division summary
# ============================================================

file_name = "reporting_division_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    reporting_division_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\reporting_division_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\reporting_division_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\reporting_division_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\reporting_division_summary.csv


## Step 24 — Build Pre-Collision Movement Summary from Traffic-Focused MO Layer

### Objective
Create a pre-collision movement summary from the traffic-focused MO layer.

### Why this table matters
This table would ideally support:

- movement-pattern dashboard visuals
- interpretation of what vehicles were doing before collision
- comparison between pre-collision behavior categories

### Important methodological note
After the collision-type summary collapsed into a single generic label,
we should explicitly test whether the pre-collision movement layer contains
interpretable labels or only a structural umbrella label.

Therefore, this step includes:

- label preview
- summary generation
- QA classification of usability

### Data source
We will use:
`traffic_only_mo_base`

### Filtering logic
We will isolate rows where:

- `analytical_domain == "Traffic Collision Pre-Collision Movement"`
or
- `analytical_subcategory == "Movement preceding collision"`

### Expected output
This step will produce:

- `pre_collision_movement_summary.csv`
- a QA status indicating whether the output is:
  - dashboard_ready
  - limited_interpretability
  - audit_only

In [100]:
# ============================================================
# Step 24A: Validate pre-collision movement availability
# ============================================================

required_pre_collision_columns = [
    "dr_number",
    "mo_description",
    "analytical_domain",
    "analytical_subcategory"
]

missing_pre_collision_columns = [
    col for col in required_pre_collision_columns
    if col not in traffic_only_mo_base.columns
]

if missing_pre_collision_columns:
    raise ValueError(f"Missing required pre-collision movement columns: {missing_pre_collision_columns}")

pre_collision_movement_preview = traffic_only_mo_base.loc[
    (
        traffic_only_mo_base["analytical_domain"] == "Traffic Collision Pre-Collision Movement"
    ) |
    (
        traffic_only_mo_base["analytical_subcategory"] == "Movement preceding collision"
    ),
    ["mo_description"]
].drop_duplicates().sort_values("mo_description").reset_index(drop=True)

if pre_collision_movement_preview.empty:
    raise ValueError("No pre-collision movement rows were found in traffic_only_mo_base.")

print("=" * 70)
print("Available pre-collision movement MO descriptions")
print("=" * 70)
display(pre_collision_movement_preview)

Available pre-collision movement MO descriptions


,mo_description
0,T/C – Movement preceding collision


In [101]:
# ============================================================
# Step 24B: Prepare pre-collision movement base
# ============================================================

pre_collision_movement_base = traffic_only_mo_base.loc[
    (
        traffic_only_mo_base["analytical_domain"] == "Traffic Collision Pre-Collision Movement"
    ) |
    (
        traffic_only_mo_base["analytical_subcategory"] == "Movement preceding collision"
    ),
    ["dr_number", "mo_description", "analytical_domain", "analytical_category", "analytical_subcategory"]
].copy()

pre_collision_movement_base["pre_collision_movement_label"] = (
    pre_collision_movement_base["mo_description"]
    .astype(str)
    .str.strip()
    .str.replace(r"^T/C\s*[–-]\s*", "", regex=True)
)

print("=" * 70)
print("Pre-collision movement base prepared")
print("=" * 70)

display(pre_collision_movement_base.head(20))

Pre-collision movement base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory,pre_collision_movement_label
7,212013850,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
15,221417787,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
23,221418141,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
31,222017859,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
47,190413769,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
53,190127578,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
64,190411883,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
85,190413937,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
90,190413949,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision
96,190413950,T/C – Movement preceding collision,Traffic Collision Pre-Collision Movement,Pre-Collision Movement,Movement preceding collision,Movement preceding collision


In [102]:
# ============================================================
# Step 24C: Build pre-collision movement summary
# ============================================================

pre_collision_movement_summary = (
    pre_collision_movement_base
    .groupby("pre_collision_movement_label", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_pre_collision_movement_collisions = pre_collision_movement_summary["collision_count"].sum()
traffic_only_total_collisions = traffic_only_mo_base["dr_number"].nunique()

pre_collision_movement_summary["collision_pct_within_pre_collision_movement"] = (
    pre_collision_movement_summary["collision_count"] / total_pre_collision_movement_collisions
)

pre_collision_movement_summary["collision_pct_of_traffic_only_collisions"] = (
    pre_collision_movement_summary["collision_count"] / traffic_only_total_collisions
)

pre_collision_movement_summary["movement_rank_by_collision_count"] = (
    pre_collision_movement_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

pre_collision_movement_summary["collision_pct_within_pre_collision_movement"] = (
    pre_collision_movement_summary["collision_pct_within_pre_collision_movement"].round(4)
)

pre_collision_movement_summary["collision_pct_of_traffic_only_collisions"] = (
    pre_collision_movement_summary["collision_pct_of_traffic_only_collisions"].round(4)
)

pre_collision_movement_summary = pre_collision_movement_summary.sort_values(
    ["collision_count", "mo_row_count", "pre_collision_movement_label"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Pre-Collision Movement Summary Created")
print("=" * 70)

display(pre_collision_movement_summary)

Pre-Collision Movement Summary Created


,pre_collision_movement_label,collision_count,mo_row_count,collision_pct_within_pre_collision_movement,collision_pct_of_traffic_only_collisions,movement_rank_by_collision_count
0,Movement preceding collision,361237,361237,1.0000,0.6877,1


In [103]:
# ============================================================
# Step 24D: QA assessment for pre-collision movement output
# ============================================================

pre_collision_movement_qc = pd.DataFrame([
    assess_mo_output(
        df=pre_collision_movement_summary,
        summary_name="pre_collision_movement_summary",
        label_col="pre_collision_movement_label"
    )
])

print("=" * 70)
print("Pre-Collision Movement QA")
print("=" * 70)
display(pre_collision_movement_qc)

# Optional: append to existing registry
mo_output_registry = pd.concat(
    [mo_output_registry, pre_collision_movement_qc],
    ignore_index=True
).drop_duplicates(subset=["summary_name"], keep="last")

mo_output_registry.to_csv(mo_output_registry_path, index=False)
print(f"Updated MO output registry: {mo_output_registry_path}")

Pre-Collision Movement QA


,summary_name,label_column,source_table,row_count,unique_labels,top_label,top_label_collision_count,top_label_share,output_status,interpretation_note
0,pre_collision_movement_summary,pre_collision_movement_label,traffic_only_mo_base,1,1,Movement preceding collision,361237,1.0000,audit_only,"Valid structure, but collapsed into one generi..."


Updated MO output registry: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables\final\mo_output_registry.csv


In [104]:
# ============================================================
# Step 24E: Export pre-collision movement summary
# ============================================================

file_name = "pre_collision_movement_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    pre_collision_movement_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\pre_collision_movement_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\pre_collision_movement_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\pre_collision_movement_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\pre_collision_movement_summary.csv


## Step 25 — Build Causation Summary from Traffic-Focused MO Layer

### Objective
Create a causation summary from the traffic-focused MO layer.

### Why this table matters
This table is intended to support:

- collision-causation dashboard visuals
- explanation of why collisions happen
- comparison between causation-related labels
- safety storytelling around primary collision factors

### Important methodological note
Because some MO layers collapsed into a single umbrella label,
this step includes an explicit QA-aware approach.

We will:
- preview the available causation labels
- build the summary
- assess whether it is dashboard-ready or audit-only

### Data source
We will use:
`traffic_only_mo_base`

### Filtering logic
We will isolate rows where:

- `analytical_domain == "Traffic Collision Causation"`
or
- `analytical_subcategory == "Primary collision factor family"`

### Expected output
This step will produce:

- `causation_summary.csv`
- a QA status indicating whether the output is:
  - dashboard_ready
  - limited_interpretability
  - audit_only

In [105]:
# ============================================================
# Step 25A: Validate causation availability
# ============================================================

required_causation_columns = [
    "dr_number",
    "mo_description",
    "analytical_domain",
    "analytical_subcategory"
]

missing_causation_columns = [
    col for col in required_causation_columns
    if col not in traffic_only_mo_base.columns
]

if missing_causation_columns:
    raise ValueError(f"Missing required causation columns: {missing_causation_columns}")

causation_preview = traffic_only_mo_base.loc[
    (
        traffic_only_mo_base["analytical_domain"] == "Traffic Collision Causation"
    ) |
    (
        traffic_only_mo_base["analytical_subcategory"] == "Primary collision factor family"
    ),
    ["mo_description"]
].drop_duplicates().sort_values("mo_description").reset_index(drop=True)

if causation_preview.empty:
    raise ValueError("No causation rows were found in traffic_only_mo_base.")

print("=" * 70)
print("Available causation MO descriptions")
print("=" * 70)
display(causation_preview)

Available causation MO descriptions


,mo_description
0,T/C – PCF (B) Other improper driving
1,T/C – PCF (C) Other than driver
2,T/C – PCF (D) Unknown
3,T/C – Primary collision factor (A) in narrative


In [106]:
# ============================================================
# Step 25B: Prepare causation base
# ============================================================

causation_base = traffic_only_mo_base.loc[
    (
        traffic_only_mo_base["analytical_domain"] == "Traffic Collision Causation"
    ) |
    (
        traffic_only_mo_base["analytical_subcategory"] == "Primary collision factor family"
    ),
    ["dr_number", "mo_description", "analytical_domain", "analytical_category", "analytical_subcategory"]
].copy()

causation_base["causation_label"] = (
    causation_base["mo_description"]
    .astype(str)
    .str.strip()
    .str.replace(r"^T/C\s*[–-]\s*", "", regex=True)
)

print("=" * 70)
print("Causation base prepared")
print("=" * 70)

display(causation_base.head(20))

Causation base prepared


,dr_number,mo_description,analytical_domain,analytical_category,analytical_subcategory,causation_label
5,212013850,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
13,221417787,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
21,221418141,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
29,222017859,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
36,190319651,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
43,190319680,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
45,190413769,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
51,190127578,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
61,190319695,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative
62,190411883,T/C – Primary collision factor (A) in narrative,Traffic Collision Causation,Traffic Collision Cause,Primary collision factor family,Primary collision factor (A) in narrative


In [107]:
# ============================================================
# Step 25C: Build causation summary
# ============================================================

causation_summary = (
    causation_base
    .groupby("causation_label", dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        mo_row_count=("dr_number", "size")
    )
    .reset_index()
)

total_causation_collisions = causation_summary["collision_count"].sum()
traffic_only_total_collisions = traffic_only_mo_base["dr_number"].nunique()

causation_summary["collision_pct_within_causation"] = (
    causation_summary["collision_count"] / total_causation_collisions
)

causation_summary["collision_pct_of_traffic_only_collisions"] = (
    causation_summary["collision_count"] / traffic_only_total_collisions
)

causation_summary["causation_rank_by_collision_count"] = (
    causation_summary["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

causation_summary["collision_pct_within_causation"] = (
    causation_summary["collision_pct_within_causation"].round(4)
)

causation_summary["collision_pct_of_traffic_only_collisions"] = (
    causation_summary["collision_pct_of_traffic_only_collisions"].round(4)
)

causation_summary = causation_summary.sort_values(
    ["collision_count", "mo_row_count", "causation_label"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Causation Summary Created")
print("=" * 70)

display(causation_summary)

Causation Summary Created


,causation_label,collision_count,mo_row_count,collision_pct_within_causation,collision_pct_of_traffic_only_collisions,causation_rank_by_collision_count
0,Primary collision factor (A) in narrative,454441,454441,0.9332,0.8652,1
1,PCF (B) Other improper driving,22351,22351,0.0459,0.0426,2
2,PCF (D) Unknown,6580,6580,0.0135,0.0125,3
3,PCF (C) Other than driver,3606,3606,0.0074,0.0069,4


In [108]:
# ============================================================
# Step 25D: QA assessment for causation output
# ============================================================

causation_qc = pd.DataFrame([
    assess_mo_output(
        df=causation_summary,
        summary_name="causation_summary",
        label_col="causation_label"
    )
])

print("=" * 70)
print("Causation QA")
print("=" * 70)
display(causation_qc)

# Update registry
mo_output_registry = pd.concat(
    [mo_output_registry, causation_qc],
    ignore_index=True
).drop_duplicates(subset=["summary_name"], keep="last")

mo_output_registry.to_csv(mo_output_registry_path, index=False)
print(f"Updated MO output registry: {mo_output_registry_path}")

Causation QA


,summary_name,label_column,source_table,row_count,unique_labels,top_label,top_label_collision_count,top_label_share,output_status,interpretation_note
0,causation_summary,causation_label,traffic_only_mo_base,4,4,Primary collision factor (A) in narrative,454441,0.9332,limited_interpretability,"Valid output, but heavily dominated by one lab..."


Updated MO output registry: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables\final\mo_output_registry.csv


In [109]:
# ============================================================
# Step 25E: Export causation summary
# ============================================================

file_name = "causation_summary.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    causation_summary.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\causation_summary.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\causation_summary.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\causation_summary.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\causation_summary.csv


## Section 4 — Map-Ready Outputs

## Step 26 — Build Map-Ready Collision Points File

### Objective
Create a map-ready file from the cleaned collision-level dataset using valid latitude and longitude coordinates.

### Why this file matters
This file is intended for point-based map visuals such as:

- Tableau symbol maps
- Power BI map visuals
- Excel 3D Maps
- scatter-based geographic visuals

### Why `collisions_clean` is the correct source
We will use:
`collisions_clean.csv`

because it is already:
- cleaned
- collision-level
- non-duplicated by `dr_number`
- enriched with valid coordinate flags

### Filtering rule
Only rows with valid coordinates will be retained.

### Output grain
One row per collision with valid coordinates.

### Main benefit
This creates a lightweight, dashboard-friendly geographic point file
without using the MO-expanded table, which would duplicate collisions.

In [110]:
# ============================================================
# Step 26A: Validate required columns for map-ready file
# ============================================================

required_map_columns = [
    "dr_number",
    "date_occurred",
    "occ_year",
    "occ_month",
    "occ_month_name",
    "occ_weekday_name",
    "occ_hour",
    "area_id",
    "area_name",
    "reporting_district",
    "premise_code",
    "premise_description",
    "victim_age",
    "victim_sex",
    "victim_descent",
    "latitude",
    "longitude",
    "has_valid_coordinates"
]

missing_map_columns = [
    col for col in required_map_columns
    if col not in collisions_clean.columns
]

if missing_map_columns:
    raise ValueError(f"Missing required map columns: {missing_map_columns}")

print("All required map columns are available ✔")

All required map columns are available ✔


In [111]:
# ============================================================
# Step 26B: Build map-ready collision points file
# ============================================================

map_collision_points = (
    collisions_clean
    .loc[collisions_clean["has_valid_coordinates"] == True, required_map_columns]
    .copy()
)

map_collision_points["point_count"] = 1

map_collision_points["map_popup_label"] = (
    "DR#: " + map_collision_points["dr_number"].astype(str)
    + " | Area: " + map_collision_points["area_name"].astype(str)
    + " | Premise: " + map_collision_points["premise_description"].astype(str)
    + " | Date: " + map_collision_points["date_occurred"].astype(str)
)

map_collision_points = map_collision_points.sort_values(
    ["occ_year", "occ_month", "dr_number"]
).reset_index(drop=True)

print("=" * 70)
print("Map-ready collision points file created")
print("=" * 70)
print(f"Rows: {len(map_collision_points):,}")
print(f"Unique collisions: {map_collision_points['dr_number'].nunique():,}")

display(map_collision_points.head(10))

Map-ready collision points file created
Rows: 620,684
Unique collisions: 620,684


,dr_number,date_occurred,occ_year,occ_month,occ_month_name,occ_weekday_name,occ_hour,area_id,area_name,reporting_district,premise_code,premise_description,victim_age,victim_sex,victim_descent,latitude,longitude,has_valid_coordinates,point_count,map_popup_label
0,100104017,2010-01-01,2010,1,January,Friday,12,1,Central,171,101.0000,STREET,NaN,F,H,34.0453,-118.2651,True,1,DR#: 100104017 | Area: Central | Premise: STRE...
1,100104019,2010-01-01,2010,1,January,Friday,14,1,Central,143,101.0000,STREET,47.0000,M,H,34.0503,-118.2504,True,1,DR#: 100104019 | Area: Central | Premise: STRE...
2,100104025,2010-01-01,2010,1,January,Friday,17,1,Central,185,101.0000,STREET,36.0000,M,H,34.0369,-118.2522,True,1,DR#: 100104025 | Area: Central | Premise: STRE...
3,100104026,2010-01-01,2010,1,January,Friday,15,1,Central,185,101.0000,STREET,NaN,M,W,34.0371,-118.2551,True,1,DR#: 100104026 | Area: Central | Premise: STRE...
4,100104049,2010-01-01,2010,1,January,Friday,20,1,Central,105,101.0000,STREET,59.0000,M,B,34.0684,-118.2367,True,1,DR#: 100104049 | Area: Central | Premise: STRE...
5,100104051,2010-01-01,2010,1,January,Friday,22,1,Central,159,101.0000,STREET,NaN,M,H,34.0421,-118.2327,True,1,DR#: 100104051 | Area: Central | Premise: STRE...
6,100104092,2010-01-02,2010,1,January,Saturday,18,1,Central,161,101.0000,STREET,31.0000,M,H,34.0487,-118.2627,True,1,DR#: 100104092 | Area: Central | Premise: STRE...
7,100104094,2010-01-02,2010,1,January,Saturday,19,1,Central,159,101.0000,STREET,34.0000,F,A,34.0404,-118.2330,True,1,DR#: 100104094 | Area: Central | Premise: STRE...
8,100104103,2010-01-03,2010,1,January,Sunday,11,1,Central,195,101.0000,STREET,55.0000,F,H,34.0341,-118.2591,True,1,DR#: 100104103 | Area: Central | Premise: STRE...
9,100104126,2010-01-03,2010,1,January,Sunday,14,1,Central,123,101.0000,STREET,20.0000,M,A,34.0531,-118.2478,True,1,DR#: 100104126 | Area: Central | Premise: STRE...


In [112]:
# ============================================================
# Step 26C: Export map-ready file to dashboard tool folders
# ============================================================

file_name = "map_collision_points.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    map_collision_points.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")

Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\map_collision_points.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\map_collision_points.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\map_collision_points.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\map_collision_points.csv


## Step 27 — Build Map Hotspots by Coordinate

### Objective
Create an aggregated hotspot map file by grouping collisions at the coordinate level.

### Why this file matters
The point-level map file is useful for detailed exploration,
but a coordinate-aggregated hotspot file is often better for dashboard maps because it is:

- lighter
- faster
- easier to read
- better for bubble maps

### Output grain
One row per unique latitude-longitude pair.

### Main metric
- `collision_count` = number of collisions at the same coordinate pair

### Additional fields
We will also calculate:

- earliest year
- latest year
- latest collision date
- dominant area name
- dominant reporting district
- dominant premise description
- hotspot popup label

### Recommended use
This file is best for:

- Tableau bubble maps
- Power BI bubble maps
- hotspot ranking maps
- geographic hotspot storytelling

In [113]:
# ============================================================
# Step 27A: Resolve map source
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

def safe_mode(series):
    series = series.dropna().astype(str).str.strip()
    if series.empty:
        return "Unknown"
    mode_vals = series.mode()
    if len(mode_vals) == 0:
        return "Unknown"
    return mode_vals.iloc[0]

# Priority:
# 1) use existing map_collision_points if already created
# 2) otherwise load from exported map file
if "map_collision_points" in globals() and isinstance(map_collision_points, pd.DataFrame):
    map_hotspot_source = map_collision_points.copy()
    print("Using existing DataFrame: map_collision_points")
else:
    if "PROJECT_ROOT" not in globals():
        PROJECT_ROOT = Path.cwd()

    map_file_path = PROJECT_ROOT / "outputs" / "excel_ready" / "map_collision_points.csv"

    if not map_file_path.exists():
        raise FileNotFoundError(
            f"Could not find map source file: {map_file_path}"
        )

    map_hotspot_source = pd.read_csv(map_file_path, low_memory=False)
    print(f"Loaded map source from: {map_file_path}")

required_hotspot_columns = [
    "dr_number",
    "date_occurred",
    "occ_year",
    "area_name",
    "reporting_district",
    "premise_description",
    "latitude",
    "longitude"
]

missing_hotspot_columns = [
    col for col in required_hotspot_columns
    if col not in map_hotspot_source.columns
]

if missing_hotspot_columns:
    raise ValueError(f"Missing required hotspot columns: {missing_hotspot_columns}")

print("=" * 70)
print("Map hotspot source resolved successfully")
print("=" * 70)
print(f"Rows: {len(map_hotspot_source):,}")
print(f"Unique collisions: {map_hotspot_source['dr_number'].nunique():,}")

Using existing DataFrame: map_collision_points
Map hotspot source resolved successfully
Rows: 620,684
Unique collisions: 620,684


In [114]:
# ============================================================
# Step 27B: Build hotspot file by coordinate
# ============================================================

map_hotspots_by_coordinate = (
    map_hotspot_source
    .groupby(["latitude", "longitude"], dropna=False)
    .agg(
        collision_count=("dr_number", "nunique"),
        earliest_year=("occ_year", "min"),
        latest_year=("occ_year", "max"),
        latest_collision_date=("date_occurred", "max"),
        dominant_area_name=("area_name", safe_mode),
        dominant_reporting_district=("reporting_district", safe_mode),
        dominant_premise_description=("premise_description", safe_mode),
        unique_area_count=("area_name", "nunique"),
        unique_premise_count=("premise_description", "nunique")
    )
    .reset_index()
)

map_hotspots_by_coordinate["hotspot_rank"] = (
    map_hotspots_by_coordinate["collision_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

map_hotspots_by_coordinate["hotspot_popup_label"] = (
    "Hotspot Count: " + map_hotspots_by_coordinate["collision_count"].astype(str)
    + " | Area: " + map_hotspots_by_coordinate["dominant_area_name"].astype(str)
    + " | Premise: " + map_hotspots_by_coordinate["dominant_premise_description"].astype(str)
    + " | Latest Year: " + map_hotspots_by_coordinate["latest_year"].astype(str)
)

map_hotspots_by_coordinate = map_hotspots_by_coordinate.sort_values(
    ["collision_count", "latest_year", "dominant_area_name"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("=" * 70)
print("Map hotspots by coordinate created")
print("=" * 70)
print(f"Rows: {len(map_hotspots_by_coordinate):,}")

display(map_hotspots_by_coordinate.head(15))

Map hotspots by coordinate created
Rows: 54,784


,latitude,longitude,collision_count,earliest_year,latest_year,latest_collision_date,dominant_area_name,dominant_reporting_district,dominant_premise_description,unique_area_count,unique_premise_count,hotspot_rank,hotspot_popup_label
0,33.9892,-118.3089,626,2010,2025,2025-03-02,77th Street,1213,STREET,2,6,1,Hotspot Count: 626 | Area: 77th Street | Premi...
1,33.9601,-118.2827,555,2010,2025,2025-02-24,Southeast,1267,STREET,2,3,2,Hotspot Count: 555 | Area: Southeast | Premise...
2,34.2012,-118.4662,527,2010,2020,2020-06-21,Van Nuys,911,STREET,3,5,3,Hotspot Count: 527 | Area: Van Nuys | Premise:...
3,34.2355,-118.5536,525,2010,2025,2025-02-19,Devonshire,1764,STREET,1,4,4,Hotspot Count: 525 | Area: Devonshire | Premis...
4,34.2216,-118.4488,517,2010,2021,2021-12-16,Mission,1985,STREET,2,6,5,Hotspot Count: 517 | Area: Mission | Premise: ...
5,34.1867,-118.4662,463,2010,2025,2025-02-18,Van Nuys,932,STREET,3,3,6,Hotspot Count: 463 | Area: Van Nuys | Premise:...
6,34.2012,-118.4313,452,2010,2020,2020-06-07,Van Nuys,919,STREET,3,4,7,Hotspot Count: 452 | Area: Van Nuys | Premise:...
7,34.1016,-118.3387,451,2010,2025,2025-01-23,Hollywood,645,STREET,1,6,8,Hotspot Count: 451 | Area: Hollywood | Premise...
8,34.1721,-118.4662,434,2010,2020,2020-05-24,Van Nuys,941,STREET,3,4,9,Hotspot Count: 434 | Area: Van Nuys | Premise:...
9,33.9420,-118.4095,430,2010,2025,2025-02-25,Pacific,1494,STREET,1,5,10,Hotspot Count: 430 | Area: Pacific | Premise: ...


In [115]:
# ============================================================
# Step 27C: Export hotspot map file
# ============================================================

file_name = "map_hotspots_by_coordinate.csv"

for tool_name, folder_path in tool_export_dirs.items():
    export_path = folder_path / file_name
    map_hotspots_by_coordinate.to_csv(export_path, index=False)
    print(f"Saved to {tool_name}: {export_path}")


Saved to excel: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\map_hotspots_by_coordinate.csv
Saved to sql: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\sql_ready\map_hotspots_by_coordinate.csv
Saved to tableau: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\map_hotspots_by_coordinate.csv
Saved to powerbi: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\map_hotspots_by_coordinate.csv


## Section 5 — Output Governance, Manifests, and Cleanup

## Step 28 — Build Dashboard Output Registry and Tool Manifests

### Objective
Create the governed delivery layer for the official outputs produced in Notebook 05.

### Why this step matters
By this stage, the notebook has produced many tables, but not all of them should be treated equally.

This registry documents:
- output name
- file name
- row and column counts
- source table
- analytical layer
- population scope
- data grain
- dashboard usability status
- whether the output belongs in the main dashboard
- recommended visual and dashboard page
- preferred tools

### Deliverables
This step creates:
- `dashboard_output_registry.csv`
- `excel_output_manifest.csv`
- `tableau_output_manifest.csv`
- `powerbi_output_manifest.csv`
- `sql_output_manifest.csv`

### Governance note
The registry is the official reference for which Notebook 05 outputs should be used, where they belong, and how they should be interpreted.


In [116]:
# ============================================================
# Step 28A: Build dashboard output registry
# ============================================================

output_catalog = {
    "yearly_summary": yearly_summary,
    "monthly_summary": monthly_summary,
    "weekday_hour_summary": weekday_hour_summary,
    "area_summary": area_summary,
    "premise_summary": premise_summary,
    "premise_summary_non_street": premise_summary_non_street,
    "victim_age_group_summary": victim_age_group_summary,
    "victim_sex_summary": victim_sex_summary,
    "victim_descent_summary": victim_descent_summary,
    "vulnerable_users_summary": vulnerable_users_summary,
    "traffic_only_mo_domain_summary": traffic_only_mo_domain_summary,
    "hit_run_summary": hit_run_summary,
    "dui_sobriety_summary": dui_sobriety_summary,
    "dui_sobriety_group_summary": dui_sobriety_group_summary,
    "injury_severity_summary": injury_severity_summary,
    "reporting_division_summary": reporting_division_summary,
    "collision_type_summary": collision_type_summary,
    "pre_collision_movement_summary": pre_collision_movement_summary,
    "causation_summary": causation_summary,
    "primary_entity_summary": primary_entity_summary,
    "primary_entity_by_domain_summary": primary_entity_by_domain_summary,
    "map_collision_points": map_collision_points,
    "map_hotspots_by_coordinate": map_hotspots_by_coordinate,
}

dashboard_registry_rules = {
    "yearly_summary": {
        "source_table": "collisions_clean",
        "analytical_layer": "time",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "line chart",
        "recommended_dashboard_page": "Overview",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "monthly_summary": {
        "source_table": "collisions_clean",
        "analytical_layer": "time",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "line chart / heatmap",
        "recommended_dashboard_page": "Overview",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "weekday_hour_summary": {
        "source_table": "collisions_clean",
        "analytical_layer": "time",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "heatmap",
        "recommended_dashboard_page": "Time Risk",
        "preferred_tools": "tableau,powerbi"
    },
    "area_summary": {
        "source_table": "collisions_clean",
        "analytical_layer": "geography",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "ranked bar chart / map",
        "recommended_dashboard_page": "Geography",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "premise_summary": {
        "source_table": "collisions_clean",
        "analytical_layer": "context",
        "output_status": "limited_interpretability",
        "include_in_main_dashboard": False,
        "recommended_visual": "bar chart",
        "recommended_dashboard_page": "Context",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "premise_summary_non_street": {
        "source_table": "collisions_clean",
        "analytical_layer": "context",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "ranked bar chart",
        "recommended_dashboard_page": "Context",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "victim_age_group_summary": {
        "source_table": "collisions_clean",
        "analytical_layer": "demographics",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "ordered bar chart",
        "recommended_dashboard_page": "Victim Profile",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "victim_sex_summary": {
        "source_table": "collisions_clean",
        "analytical_layer": "demographics",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "bar chart",
        "recommended_dashboard_page": "Victim Profile",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "victim_descent_summary": {
        "source_table": "collisions_clean",
        "analytical_layer": "demographics",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "ranked bar chart",
        "recommended_dashboard_page": "Victim Profile",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "vulnerable_users_summary": {
        "source_table": "fact_collision_mo_enriched",
        "analytical_layer": "safety",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "bar chart / KPI cards",
        "recommended_dashboard_page": "Safety",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "traffic_only_mo_domain_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "mo_overview",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "ranked bar chart",
        "recommended_dashboard_page": "MO Overview",
        "preferred_tools": "tableau,powerbi"
    },
    "hit_run_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "safety",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "bar chart / KPI cards",
        "recommended_dashboard_page": "Safety",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "dui_sobriety_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "safety",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "bar chart",
        "recommended_dashboard_page": "Safety",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "dui_sobriety_group_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "safety",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": False,
        "recommended_visual": "KPI support / small bar",
        "recommended_dashboard_page": "Safety",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "injury_severity_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "safety",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "ordered bar chart",
        "recommended_dashboard_page": "Safety",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "reporting_division_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "jurisdiction",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "ranked bar chart",
        "recommended_dashboard_page": "Geography / Jurisdiction",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "collision_type_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "mo_structure",
        "output_status": "audit_only",
        "include_in_main_dashboard": False,
        "recommended_visual": "none",
        "recommended_dashboard_page": "Audit Only",
        "preferred_tools": "excel"
    },
    "pre_collision_movement_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "mo_structure",
        "output_status": "audit_only",
        "include_in_main_dashboard": False,
        "recommended_visual": "none",
        "recommended_dashboard_page": "Audit Only",
        "preferred_tools": "excel"
    },
    "causation_summary": {
        "source_table": "traffic_only_mo_base",
        "analytical_layer": "safety",
        "output_status": "limited_interpretability",
        "include_in_main_dashboard": False,
        "recommended_visual": "secondary bar chart",
        "recommended_dashboard_page": "Safety (secondary)",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "primary_entity_summary": {
        "source_table": "fact_collision_mo_enriched",
        "analytical_layer": "mo_overview",
        "output_status": "limited_interpretability",
        "include_in_main_dashboard": False,
        "recommended_visual": "small overview chart / slicer support",
        "recommended_dashboard_page": "MO Overview",
        "preferred_tools": "tableau,powerbi"
    },
    "primary_entity_by_domain_summary": {
        "source_table": "fact_collision_mo_enriched",
        "analytical_layer": "mo_bridge",
        "output_status": "limited_interpretability",
        "include_in_main_dashboard": False,
        "recommended_visual": "matrix / appendix",
        "recommended_dashboard_page": "MO Appendix",
        "preferred_tools": "tableau,powerbi"
    },
}

dashboard_registry_rules.update({
    "map_collision_points": {
        "source_table": "collisions_clean",
        "analytical_layer": "geospatial",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "point map",
        "recommended_dashboard_page": "Geography",
        "preferred_tools": "excel,tableau,powerbi"
    },
    "map_hotspots_by_coordinate": {
        "source_table": "collisions_clean",
        "analytical_layer": "geospatial",
        "output_status": "dashboard_ready",
        "include_in_main_dashboard": True,
        "recommended_visual": "bubble map / hotspot map",
        "recommended_dashboard_page": "Geography",
        "preferred_tools": "excel,tableau,powerbi"
    },
})

population_scope_map = {
    "yearly_summary": "all collisions",
    "monthly_summary": "all collisions",
    "weekday_hour_summary": "all collisions",
    "area_summary": "all collisions",
    "premise_summary": "all collisions",
    "premise_summary_non_street": "all collisions excluding STREET premise",
    "victim_age_group_summary": "all collisions with usable victim age",
    "victim_sex_summary": "all collisions",
    "victim_descent_summary": "all collisions",
    "vulnerable_users_summary": "MO-enriched collisions with vulnerable-user signal",
    "traffic_only_mo_domain_summary": "traffic-focused MO subset",
    "hit_run_summary": "traffic-focused MO subset",
    "dui_sobriety_summary": "traffic-focused MO subset",
    "dui_sobriety_group_summary": "traffic-focused MO subset",
    "injury_severity_summary": "traffic-focused MO subset",
    "reporting_division_summary": "traffic-focused MO subset",
    "collision_type_summary": "traffic-focused MO subset",
    "pre_collision_movement_summary": "traffic-focused MO subset",
    "causation_summary": "traffic-focused MO subset",
    "primary_entity_summary": "full MO-enriched layer",
    "primary_entity_by_domain_summary": "full MO-enriched layer",
    "map_collision_points": "collisions with valid coordinates only",
    "map_hotspots_by_coordinate": "collisions with valid coordinates only",
}

data_grain_map = {
    "yearly_summary": "one row per year",
    "monthly_summary": "one row per year-month",
    "weekday_hour_summary": "one row per weekday-hour",
    "area_summary": "one row per area",
    "premise_summary": "one row per premise",
    "premise_summary_non_street": "one row per non-street premise",
    "victim_age_group_summary": "one row per age band",
    "victim_sex_summary": "one row per victim sex label",
    "victim_descent_summary": "one row per victim descent label",
    "vulnerable_users_summary": "one row per vulnerable-user category",
    "traffic_only_mo_domain_summary": "one row per analytical domain",
    "hit_run_summary": "one row per hit-and-run label",
    "dui_sobriety_summary": "one row per sobriety label",
    "dui_sobriety_group_summary": "one row per grouped sobriety label",
    "injury_severity_summary": "one row per injury severity label",
    "reporting_division_summary": "one row per reporting division",
    "collision_type_summary": "one row per collision type label",
    "pre_collision_movement_summary": "one row per pre-collision movement label",
    "causation_summary": "one row per causation label",
    "primary_entity_summary": "one row per primary entity",
    "primary_entity_by_domain_summary": "one row per primary entity x analytical domain",
    "map_collision_points": "one row per collision point",
    "map_hotspots_by_coordinate": "one row per coordinate pair",
}

registry_rows = []

for output_name, df in output_catalog.items():
    rule = dashboard_registry_rules.get(output_name, {})
    registry_rows.append({
        "output_name": output_name,
        "file_name": f"{output_name}.csv",
        "row_count": len(df),
        "column_count": df.shape[1],
        "source_table": rule.get("source_table", "unknown"),
        "analytical_layer": rule.get("analytical_layer", "unknown"),
        "output_status": rule.get("output_status", "unknown"),
        "include_in_main_dashboard": rule.get("include_in_main_dashboard", False),
        "recommended_visual": rule.get("recommended_visual", "unknown"),
        "recommended_dashboard_page": rule.get("recommended_dashboard_page", "unknown"),
        "preferred_tools": rule.get("preferred_tools", "unknown"),
        "population_scope": population_scope_map.get(output_name, "unspecified"),
        "data_grain": data_grain_map.get(output_name, "unspecified")
    })

dashboard_output_registry = pd.DataFrame(registry_rows)

dashboard_output_registry = dashboard_output_registry.sort_values(
    ["include_in_main_dashboard", "output_status", "output_name"],
    ascending=[False, True, True]
).reset_index(drop=True)

print("=" * 70)
print("Dashboard Output Registry")
print("=" * 70)
display(dashboard_output_registry)

Dashboard Output Registry


,output_name,file_name,row_count,column_count,source_table,analytical_layer,output_status,include_in_main_dashboard,recommended_visual,recommended_dashboard_page,preferred_tools,population_scope,data_grain
0,area_summary,area_summary.csv,21,9,collisions_clean,geography,dashboard_ready,True,ranked bar chart / map,Geography,"excel,tableau,powerbi",all collisions,one row per area
1,dui_sobriety_summary,dui_sobriety_summary.csv,3,7,traffic_only_mo_base,safety,dashboard_ready,True,bar chart,Safety,"excel,tableau,powerbi",traffic-focused MO subset,one row per sobriety label
2,hit_run_summary,hit_run_summary.csv,2,6,traffic_only_mo_base,safety,dashboard_ready,True,bar chart / KPI cards,Safety,"excel,tableau,powerbi",traffic-focused MO subset,one row per hit-and-run label
3,injury_severity_summary,injury_severity_summary.csv,5,7,traffic_only_mo_base,safety,dashboard_ready,True,ordered bar chart,Safety,"excel,tableau,powerbi",traffic-focused MO subset,one row per injury severity label
4,map_collision_points,map_collision_points.csv,620684,20,collisions_clean,geospatial,dashboard_ready,True,point map,Geography,"excel,tableau,powerbi",collisions with valid coordinates only,one row per collision point
5,map_hotspots_by_coordinate,map_hotspots_by_coordinate.csv,54784,13,collisions_clean,geospatial,dashboard_ready,True,bubble map / hotspot map,Geography,"excel,tableau,powerbi",collisions with valid coordinates only,one row per coordinate pair
6,monthly_summary,monthly_summary.csv,183,11,collisions_clean,time,dashboard_ready,True,line chart / heatmap,Overview,"excel,tableau,powerbi",all collisions,one row per year-month
7,premise_summary_non_street,premise_summary_non_street.csv,123,9,collisions_clean,context,dashboard_ready,True,ranked bar chart,Context,"excel,tableau,powerbi",all collisions excluding STREET premise,one row per non-street premise
8,reporting_division_summary,reporting_division_summary.csv,25,6,traffic_only_mo_base,jurisdiction,dashboard_ready,True,ranked bar chart,Geography / Jurisdiction,"excel,tableau,powerbi",traffic-focused MO subset,one row per reporting division
9,traffic_only_mo_domain_summary,traffic_only_mo_domain_summary.csv,12,6,traffic_only_mo_base,mo_overview,dashboard_ready,True,ranked bar chart,MO Overview,"tableau,powerbi",traffic-focused MO subset,one row per analytical domain


In [117]:
# ============================================================
# Step 28B: Save registry + build tool manifests
# ============================================================

FINAL_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables" / "final"
FINAL_TABLES_DIR.mkdir(parents=True, exist_ok=True)

dashboard_registry_path = FINAL_TABLES_DIR / "dashboard_output_registry.csv"
dashboard_output_registry.to_csv(dashboard_registry_path, index=False)

def build_tool_manifest(registry_df: pd.DataFrame, tool_keyword: str) -> pd.DataFrame:
    manifest = registry_df[
        registry_df["preferred_tools"].str.contains(tool_keyword, case=False, na=False)
    ].copy()
    manifest["tool_name"] = tool_keyword
    return manifest

excel_output_manifest = build_tool_manifest(dashboard_output_registry, "excel")
tableau_output_manifest = build_tool_manifest(dashboard_output_registry, "tableau")
powerbi_output_manifest = build_tool_manifest(dashboard_output_registry, "powerbi")
sql_output_manifest = build_tool_manifest(dashboard_output_registry, "sql")

manifest_specs = {
    "excel": ("excel_output_manifest.csv", excel_output_manifest, EXCEL_READY_DIR),
    "tableau": ("tableau_output_manifest.csv", tableau_output_manifest, TABLEAU_READY_DIR),
    "powerbi": ("powerbi_output_manifest.csv", powerbi_output_manifest, POWERBI_READY_DIR),
    "sql": ("sql_output_manifest.csv", sql_output_manifest, SQL_READY_DIR),
}

for tool_name, (file_name, df, tool_dir) in manifest_specs.items():
    final_manifest_path = FINAL_TABLES_DIR / file_name
    tool_manifest_path = tool_dir / file_name

    df.to_csv(final_manifest_path, index=False)
    df.to_csv(tool_manifest_path, index=False)

    print(f"Saved {tool_name} manifest to final folder: {final_manifest_path}")
    print(f"Saved {tool_name} manifest to tool folder : {tool_manifest_path}")

print(f"Saved registry: {dashboard_registry_path}")


Saved excel manifest to final folder: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables\final\excel_output_manifest.csv
Saved excel manifest to tool folder : C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\excel_ready\excel_output_manifest.csv
Saved tableau manifest to final folder: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables\final\tableau_output_manifest.csv
Saved tableau manifest to tool folder : C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tableau_ready\tableau_output_manifest.csv
Saved powerbi manifest to final folder: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables\final\powerbi_output_manifest.csv
Saved powerbi manifest to tool folder : C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\powerbi_ready\powerbi_output_manifest.csv
Saved sql manifes

## Step 29 — Remove Unused Export Files

### Objective
Remove exported files that are not planned for final dashboard usage.

### Deletion rule
Cleanup is driven by the final `dashboard_output_registry` and removes only files where:

- `include_in_main_dashboard == False`

### Important scope note
This cleanup only targets exported output CSV files inside tool folders.
It does **not** delete:
- registries
- manifests
- audit tables under `outputs/tables/`
- project documentation


In [118]:
# ============================================================
# Step 29A: Preview unused files planned for deletion
# ============================================================

unused_outputs_plan = dashboard_output_registry.loc[
    dashboard_output_registry["include_in_main_dashboard"] == False,
    [
        "output_name",
        "file_name",
        "output_status",
        "recommended_visual",
        "recommended_dashboard_page"
    ]
].copy()

def build_delete_reason(row):
    if row["output_status"] == "audit_only":
        return "Structural/audit output only; not suitable for final dashboard visuals."
    elif row["output_status"] == "limited_interpretability":
        return "Low-interpretability output; retained analytically but excluded from final dashboard."
    elif row["output_status"] == "dashboard_ready":
        return "Support-only output; not selected for the main dashboard."
    else:
        return "Not selected for final dashboard usage."

unused_outputs_plan["delete_reason"] = unused_outputs_plan.apply(build_delete_reason, axis=1)

tool_folders = {
    "excel": EXCEL_READY_DIR,
    "sql": SQL_READY_DIR,
    "tableau": TABLEAU_READY_DIR,
    "powerbi": POWERBI_READY_DIR
}

preview_rows = []

for _, row in unused_outputs_plan.iterrows():
    for tool_name, folder_path in tool_folders.items():
        file_path = folder_path / row["file_name"]
        preview_rows.append({
            "output_name": row["output_name"],
            "file_name": row["file_name"],
            "tool_name": tool_name,
            "file_path": str(file_path),
            "exists": file_path.exists(),
            "output_status": row["output_status"],
            "delete_reason": row["delete_reason"]
        })

unused_file_delete_preview = pd.DataFrame(preview_rows)

print("=" * 70)
print("Unused export files planned for deletion")
print("=" * 70)

display(unused_outputs_plan)
print("\nDetailed file-level preview:")
display(unused_file_delete_preview)

Unused export files planned for deletion


,output_name,file_name,output_status,recommended_visual,recommended_dashboard_page,delete_reason
16,collision_type_summary,collision_type_summary.csv,audit_only,none,Audit Only,Structural/audit output only; not suitable for...
17,pre_collision_movement_summary,pre_collision_movement_summary.csv,audit_only,none,Audit Only,Structural/audit output only; not suitable for...
18,dui_sobriety_group_summary,dui_sobriety_group_summary.csv,dashboard_ready,KPI support / small bar,Safety,Support-only output; not selected for the main...
19,causation_summary,causation_summary.csv,limited_interpretability,secondary bar chart,Safety (secondary),Low-interpretability output; retained analytic...
20,premise_summary,premise_summary.csv,limited_interpretability,bar chart,Context,Low-interpretability output; retained analytic...
21,primary_entity_by_domain_summary,primary_entity_by_domain_summary.csv,limited_interpretability,matrix / appendix,MO Appendix,Low-interpretability output; retained analytic...
22,primary_entity_summary,primary_entity_summary.csv,limited_interpretability,small overview chart / slicer support,MO Overview,Low-interpretability output; retained analytic...



Detailed file-level preview:


,output_name,file_name,tool_name,file_path,exists,output_status,delete_reason
0,collision_type_summary,collision_type_summary.csv,excel,C:\Users\MKamal\Desktop\Project\without output...,True,audit_only,Structural/audit output only; not suitable for...
1,collision_type_summary,collision_type_summary.csv,sql,C:\Users\MKamal\Desktop\Project\without output...,True,audit_only,Structural/audit output only; not suitable for...
2,collision_type_summary,collision_type_summary.csv,tableau,C:\Users\MKamal\Desktop\Project\without output...,True,audit_only,Structural/audit output only; not suitable for...
3,collision_type_summary,collision_type_summary.csv,powerbi,C:\Users\MKamal\Desktop\Project\without output...,True,audit_only,Structural/audit output only; not suitable for...
4,pre_collision_movement_summary,pre_collision_movement_summary.csv,excel,C:\Users\MKamal\Desktop\Project\without output...,True,audit_only,Structural/audit output only; not suitable for...
5,pre_collision_movement_summary,pre_collision_movement_summary.csv,sql,C:\Users\MKamal\Desktop\Project\without output...,True,audit_only,Structural/audit output only; not suitable for...
6,pre_collision_movement_summary,pre_collision_movement_summary.csv,tableau,C:\Users\MKamal\Desktop\Project\without output...,True,audit_only,Structural/audit output only; not suitable for...
7,pre_collision_movement_summary,pre_collision_movement_summary.csv,powerbi,C:\Users\MKamal\Desktop\Project\without output...,True,audit_only,Structural/audit output only; not suitable for...
8,dui_sobriety_group_summary,dui_sobriety_group_summary.csv,excel,C:\Users\MKamal\Desktop\Project\without output...,True,dashboard_ready,Support-only output; not selected for the main...
9,dui_sobriety_group_summary,dui_sobriety_group_summary.csv,sql,C:\Users\MKamal\Desktop\Project\without output...,True,dashboard_ready,Support-only output; not selected for the main...


In [119]:
# ============================================================
# Step 29B: Delete unused exported files from tool folders
# ============================================================

delete_log_rows = []

for _, row in unused_file_delete_preview.iterrows():
    file_path = Path(row["file_path"])

    if file_path.exists():
        file_path.unlink()
        delete_result = "deleted"
    else:
        delete_result = "not_found"

    delete_log_rows.append({
        "output_name": row["output_name"],
        "file_name": row["file_name"],
        "tool_name": row["tool_name"],
        "file_path": row["file_path"],
        "output_status": row["output_status"],
        "delete_reason": row["delete_reason"],
        "delete_result": delete_result
    })

unused_output_delete_log = pd.DataFrame(delete_log_rows)

DELETE_LOG_PATH = FINAL_TABLES_DIR / "unused_output_delete_log.csv"
unused_output_delete_log.to_csv(DELETE_LOG_PATH, index=False)

print("=" * 70)
print("Unused export cleanup completed")
print("=" * 70)

display(
    unused_output_delete_log.groupby(
        ["output_status", "delete_result"], dropna=False
    ).size().reset_index(name="file_count")
)

print(f"\nSaved delete log: {DELETE_LOG_PATH}")

Unused export cleanup completed


,output_status,delete_result,file_count
0,audit_only,deleted,8
1,dashboard_ready,deleted,4
2,limited_interpretability,deleted,16



Saved delete log: C:\Users\MKamal\Desktop\Project\without output\la_traffic_collision_project\outputs\tables\final\unused_output_delete_log.csv


## Notebook 05 Final Closure

Notebook 05 is now organized as a true export and governance layer.

### Final outcome
This notebook now:
- builds collision-level business summaries
- builds MO-focused and traffic-focused summaries
- produces map-ready outputs
- documents official deliverables through a governed registry
- generates cleaner tool manifests
- removes non-retained exports from tool folders

### Final retained output philosophy
The remaining exported files represent the curated delivery layer for:
- Excel
- SQL
- Tableau
- Power BI

### Transition
The project is now ready to move into the next downstream layer with cleaner handoff logic and stronger output governance.
